# Market Module: Extrinsic Market-State Encoder

**Core question:** Does representing market state through engineered temporal features (rolling price averages, momentum, transaction volume, realised volatility) improve valuation stability compared to a static metadata baseline and a pure sequence model in thin, regime-dependent collectible markets?

**Design requirement:** The framework shall include a hybrid temporal regressor that maps engineered market variables into a 16- to 64-dimensional market-state vector.

---

### Notebook Structure

| Section | Content |
|---------|---------|
| 0 | Setup and Configuration |
| 1 | Data Exploration and Temporal Analysis |
| 1.5 | Sequence Length Feasibility Check |
| 2 | Temporal Split Construction |
| 3 | Feature Engineering |
| 4 | Missingness and Data Survival |
| 5 | Target Variable Handling |
| 6 | Primary Model Training (XGBoost) |
| 7 | Baseline Comparisons (market-encoder Core) |
| 8 | Feature Importance and Interpretability |
| 9 | Ablation Study |
| 10 | Overfitting and Stability Checks |
| 11 | Market-State Embedding Construction (Dual) |
| 12 | Embedding Quality Assessment |
| 13 | Fusion Readiness Check |
| 14 | Failure Mode Documentation |
| 15 | Final market-encoder Assessment |
---
##


## Section 0: Setup and configuration


In [ ]:
## Project root — works locally or in Google Colab
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/pokemon-card-valuation')
except ImportError:
    ## Local / non-Colab: assume this notebook runs from the repo's notebooks/ folder
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()


In [ ]:
## Imports
import pandas as pd
import numpy as np
from pathlib import Path
import json
import time
import random

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, normalize
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_distances

import xgboost as xgb
import pickle

import matplotlib.pyplot as plt
import seaborn as sns

print(f'XGBoost: {xgb.__version__}')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')

In [ ]:
## Reproducibility
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)

set_seed()
print(f'Seed set: {SEED}')

In [ ]:
## Path configuration and master config

CONFIG = {
    ## Paths
    'data_dir': PROJECT_ROOT / 'data/processed',
    'model_dir': PROJECT_ROOT / 'models/market',
    'results_dir': PROJECT_ROOT / 'results/market',
    'figures_dir': PROJECT_ROOT / 'results/market/figures',
    'embeddings_dir': PROJECT_ROOT / 'data/embeddings',
    'features_dir': PROJECT_ROOT / 'data/processed/features',

    ## Data
    'cards_file': 'cards_clean_v2.parquet',
    'prices_file': 'prices_clean_v2.parquet',
    'seed': SEED,

    ## Feature engineering
    'rolling_windows': [7, 14, 30],
    'lag_days': [1, 3, 7, 14],
    'min_obs_per_window': 2,

    ## Temporal split
    'train_ratio': 0.70,
    'val_ratio': 0.15,
    'test_ratio': 0.15,

    ## XGBoost primary model
    'xgb_params': {
        'n_estimators': 300,
        'max_depth': 6,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 5,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': SEED,
        'n_jobs': -1,
    },

    ## Embedding
    'embedding_dim': 64,

    ## Target
    'use_log_price': True,

    ## Price cap (already applied in cleaning, document here)
    'price_floor': 1.0,
    'price_cap': 50000.0,
}

## Create output directories
for d in ['model_dir', 'results_dir', 'figures_dir', 'embeddings_dir', 'features_dir']:
    CONFIG[d].mkdir(parents=True, exist_ok=True)

## Verify data paths
cards_path = CONFIG['data_dir'] / CONFIG['cards_file']
prices_path = CONFIG['data_dir'] / CONFIG['prices_file']
assert cards_path.exists(), f'Cards not found: {cards_path}'
assert prices_path.exists(), f'Prices not found: {prices_path}'

print('MARKET MODULE CONFIGURATION')
print(f'Data: {CONFIG["data_dir"]}')
print(f'Models: {CONFIG["model_dir"]}')
print(f'Results: {CONFIG["results_dir"]}')
print(f'\nRolling windows: {CONFIG["rolling_windows"]} days')
print(f'Lag features: {CONFIG["lag_days"]} days')
print(f'Min obs per window: {CONFIG["min_obs_per_window"]}')
print(f'\nXGBoost: n_estimators={CONFIG["xgb_params"]["n_estimators"]}, '
      f'max_depth={CONFIG["xgb_params"]["max_depth"]}, '
      f'lr={CONFIG["xgb_params"]["learning_rate"]}')
print(f'\nTarget: {"log(price+1)" if CONFIG["use_log_price"] else "raw price"}')
print(f'Embedding dim: {CONFIG["embedding_dim"]}')
print(f'Split: {CONFIG["train_ratio"]}/{CONFIG["val_ratio"]}/{CONFIG["test_ratio"]} (temporal)')
print(f'Seed: {CONFIG["seed"]}')

In [ ]:
## Save config
config_save = {k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()}
with open(CONFIG['results_dir'] / 'market_config.json', 'w') as f:
    json.dump(config_save, f, indent=2, default=str)
print('Config saved')

## Section 1: Data exploration and temporal analysis


**Data contract:** This module uses prices, dates, and card metadata only. No images or visual embeddings enter this module.

In [ ]:
## Load data. Merge cards and prices into single market dataframe
cards = pd.read_parquet(CONFIG['data_dir'] / CONFIG['cards_file'])
prices = pd.read_parquet(CONFIG['data_dir'] / CONFIG['prices_file'])

## Merge on listing_id
df = prices.merge(cards[['listing_id', 'card_name', 'rarity']], on='listing_id',
                  how='left', suffixes=('', '_cards'))

## Use card_name from cards if prices has it too
if 'card_name_cards' in df.columns:
    df['card_name'] = df['card_name_cards'].fillna(df['card_name'])
    df.drop(columns=['card_name_cards'], inplace=True)

## Handle rarity duplicates similarly
if 'rarity_cards' in df.columns:
    df['rarity'] = df['rarity_cards'].fillna(df['rarity'])
    df.drop(columns=['rarity_cards'], inplace=True)

## Ensure date_sold is datetime
df['date_sold'] = pd.to_datetime(df['date_sold'])

## Sort by date. Critical for temporal features
df = df.sort_values('date_sold').reset_index(drop=True)

## Verify no images or vision data
assert 'local_image_path' not in df.columns, 'Image data leaked into market module!'
assert 'image_url' not in df.columns, 'Image URL leaked into market module!'

print(f'Market dataframe: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Date range: {df["date_sold"].min().date()} to {df["date_sold"].max().date()}')
print(f'\nConfirmed. No image/vision data in market module')

In [ ]:
## Transaction distribution over time
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

## Monthly transaction count
monthly = df.groupby(df['date_sold'].dt.to_period('M')).size()
monthly.index = monthly.index.to_timestamp()
axes[0].bar(monthly.index, monthly.values, width=25, alpha=0.7)
axes[0].set_title('Monthly Transaction Volume')
axes[0].set_ylabel('Transactions')
axes[0].grid(True, alpha=0.3)

## Daily transaction count
daily = df.groupby(df['date_sold'].dt.date).size()
axes[1].plot(daily.index, daily.values, alpha=0.5, linewidth=0.5)
axes[1].set_title('Daily Transaction Volume')
axes[1].set_ylabel('Transactions')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Transaction Distribution Over Time', fontsize=14)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'transaction_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## Identify spike periods and sparse regions
print('TEMPORAL DENSITY ANALYSIS')
print(f'Total transactions: {len(df)}')
print(f'Unique dates: {df["date_sold"].dt.date.nunique()}')
print(f'Temporal span: {(df["date_sold"].max() - df["date_sold"].min()).days} days')
print(f'Mean transactions/day: {len(df) / df["date_sold"].dt.date.nunique():.1f}')

## By year
print(f'\nBy year:')
yearly = df.groupby(df['date_sold'].dt.year).agg(
    count=('price', 'size'),
    mean_price=('price', 'mean'),
    median_price=('price', 'median'),
    unique_dates=('date_sold', lambda x: x.dt.date.nunique())
)
for year, row in yearly.iterrows():
    print(f'  {year}: {row["count"]:5.0f} txns, {row["unique_dates"]:4.0f} dates, '
          f'median ${row["median_price"]:,.0f}')

## Check for temporal clustering (the problem from v1 dataset)
top_dates = df['date_sold'].dt.date.value_counts().head(10)
top_date_pct = top_dates.sum() / len(df) * 100
print(f'\nTop 10 dates contain: {top_dates.sum()} txns ({top_date_pct:.1f}%)')
print('Top 10 dates:')
for date, count in top_dates.items():
    print(f'  {date}: {count} transactions')

print(f'\nv1 reference: 76% of data in final 3 dates (severe clustering)')
print(f'v2 status: {"RESOLVED" if top_date_pct < 30 else "STILL CLUSTERED"}')

In [ ]:
## Per-card_name+grade transaction counts
cg_counts = df.groupby(['card_name', 'grade']).agg(
    count=('price', 'size'),
    unique_dates=('date_sold', lambda x: x.dt.date.nunique()),
    date_span=('date_sold', lambda x: (x.max() - x.min()).days),
    mean_price=('price', 'mean')
).reset_index()

print('PER CARD+GRADE GROUP STATISTICS')
print(cg_counts.to_string(index=False))
print(f'\nGroups: {len(cg_counts)}')
print(f'Min transactions: {cg_counts["count"].min()}')
print(f'Median transactions: {cg_counts["count"].median():.0f}')
print(f'Min unique dates: {cg_counts["unique_dates"].min()}')
print(f'All groups >= 5 transactions: {(cg_counts["count"] >= 5).all()}')

In [ ]:
## Price distribution analysis
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

## Raw price histogram
axes[0].hist(df['price'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Price Distribution (Raw)')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')

## Log price histogram
axes[1].hist(np.log1p(df['price']), bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_title('Price Distribution (Log)')
axes[1].set_xlabel('log(price + 1)')

## Price by grade
for g in [8, 9, 10]:
    grade_prices = df[df['grade'] == g]['price']
    axes[2].hist(np.log1p(grade_prices), bins=30, alpha=0.5, label=f'PSA {g}')
axes[2].set_title('Log Price by Grade')
axes[2].set_xlabel('log(price + 1)')
axes[2].legend()

plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'price_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nPrice statistics:')
print(f'Mean:   ${df["price"].mean():,.2f}')
print(f'Median: ${df["price"].median():,.2f}')
print(f'Std:    ${df["price"].std():,.2f}')
print(f'Skew:     {df["price"].skew():.2f}')
print(f'Log skew: {np.log1p(df["price"]).skew():.2f}')
print(f'\nLog transform {"recommended" if df["price"].skew() > 2 else "optional"} (skew={df["price"].skew():.1f})')
print('\nALL CONFIRMED')


## Section 1.5: Sequence length feasibility check


**Purpose:** Determine whether per-card+grade transaction sequences are long enough to justify an LSTM baseline. If median sequence length < 5, document infeasibility with data evidence and cite Hewamalage et al. (2023). This gates whether we build an LSTM or document its infeasibility as a finding.

In [ ]:
## Compute sequence length distribution per card_name+grade
## A "sequence" here = chronologically ordered transactions for a specific card+grade group

seq_lengths = df.groupby(['card_name', 'grade']).size().reset_index(name='seq_length')

print('SEQUENCE LENGTH STATISTICS')
print(f'Groups: {len(seq_lengths)}')
print(f'Min:    {seq_lengths["seq_length"].min()}')
print(f'Q1:     {seq_lengths["seq_length"].quantile(0.25):.0f}')
print(f'Median: {seq_lengths["seq_length"].median():.0f}')
print(f'Q3:     {seq_lengths["seq_length"].quantile(0.75):.0f}')
print(f'Max:    {seq_lengths["seq_length"].max()}')
print(f'Mean:   {seq_lengths["seq_length"].mean():.1f}')
print(f'\nGroups with >= 5 txns:  {(seq_lengths["seq_length"] >= 5).sum()}/{len(seq_lengths)}')
print(f'Groups with >= 10 txns:   {(seq_lengths["seq_length"] >= 10).sum()}/{len(seq_lengths)}')
print(f'Groups with >= 30 txns:   {(seq_lengths["seq_length"] >= 30).sum()}/{len(seq_lengths)}')
print(f'Groups with >= 50 txns:   {(seq_lengths["seq_length"] >= 50).sum()}/{len(seq_lengths)}')
print(f'\nPer-group detail:')
for _, row in seq_lengths.sort_values('seq_length').iterrows():
    print(f'{row["card_name"]:12s} PSA {row["grade"]}: {row["seq_length"]:4d} transactions')

In [ ]:
## Plot histogram of sequence lengths
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(seq_lengths)), seq_lengths.sort_values('seq_length')['seq_length'].values,
       color='steelblue', edgecolor='black')
ax.set_xticks(range(len(seq_lengths)))
ax.set_xticklabels(
    [f"{r['card_name'][:4]}_{r['grade']}"
     for _, r in seq_lengths.sort_values('seq_length').iterrows()],
    rotation=45, ha='right', fontsize=8
)
ax.axhline(y=5, color='red', linestyle='--', label='Min viable (5)')
ax.axhline(y=30, color='orange', linestyle='--', label='Comfortable (30)')
ax.set_ylabel('Sequence Length (transactions)')
ax.set_title('Per card+Grade sequence lengths')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'sequence_lengths.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## Also check temporal regularity. Irregular spacing makes LSTM harder
spacing_stats = []
for (card, grade), group in df.groupby(['card_name', 'grade']):
    dates = group['date_sold'].sort_values()
    if len(dates) >= 2:
        gaps = dates.diff().dt.days.dropna()
        spacing_stats.append({
            'card_name': card, 'grade': grade,
            'n_txns': len(dates),
            'mean_gap_days': gaps.mean(),
            'median_gap_days': gaps.median(),
            'max_gap_days': gaps.max(),
            'std_gap_days': gaps.std()
        })

spacing_df = pd.DataFrame(spacing_stats)
print('TRANSACTION SPACING (days between consecutive sales)')
print(f'Mean gap:   {spacing_df["mean_gap_days"].mean():.1f} days')
print(f'Median gap: {spacing_df["median_gap_days"].mean():.1f} days')
print(f'Max gap:    {spacing_df["max_gap_days"].mean():.0f} days (mean of per-group maxes)')
print(f'Std gap:    {spacing_df["std_gap_days"].mean():.1f} days')
print(f'\nPer-group spacing:')
for _, row in spacing_df.sort_values('mean_gap_days', ascending=False).iterrows():
    print(f'{row["card_name"]:12s} PSA {row["grade"]}: '
          f'mean={row["mean_gap_days"]:5.1f}d  median={row["median_gap_days"]:4.1f}d  '
          f'max={row["max_gap_days"]:4.0f}d  (n={row["n_txns"]})')

In [ ]:
## Decision: build LSTM or document infeasibility
median_seq = seq_lengths['seq_length'].median()
min_seq = seq_lengths['seq_length'].min()
all_above_5 = (seq_lengths['seq_length'] >= 5).all()
all_above_30 = (seq_lengths['seq_length'] >= 30).all()
mean_gap = spacing_df['mean_gap_days'].mean()

print('LSTM FEASIBILITY DECISION')
print(f'Median sequence length:     {median_seq:.0f}')
print(f'Min sequence length:        {min_seq}')
print(f'All groups >= 5:            {all_above_5}')
print(f'All groups >= 30:           {all_above_30}')
print(f'Mean inter-transaction gap: {mean_gap:.1f} days')

if median_seq >= 30 and all_above_5:
    lstm_feasible = True
    print(f'\nDECISION: LSTM BASELINE IS FEASIBLE')
    print(f'All groups have sufficient sequence length for a simple LSTM.')
    print(f'Will build LSTM baseline in Section 7 using per-group sequences.')
    print(f'Note: Irregular spacing (mean {mean_gap:.0f} days) means LSTM must')
    print(f'handle variable time gaps. Will use time-gap features as inputs.')
elif median_seq >= 5:
    lstm_feasible = True
    print(f'\nDECISION: LSTM BASELINE IS MARGINALLY FEASIBLE')
    print(f'Median >= 5 but some groups are thin. Will attempt LSTM with')
    print(f'reduced lookback window. Document limitations.')
else:
    lstm_feasible = False
    print(f'\nDECISION: LSTM BASELINE IS INFEASIBLE')
    print(f'Median sequence length < 5. Citing Hewamalage et al. (2023):')
    print(f'sequence models are fragile in thin markets with insufficient history.')
    print(f'Infeasibility documented as market-encoder finding.')

## Save decision
lstm_decision = {
    'feasible': lstm_feasible,
    'median_seq_length': float(median_seq),
    'min_seq_length': int(min_seq),
    'mean_gap_days': float(mean_gap),
    'all_above_5': bool(all_above_5),
    'all_above_30': bool(all_above_30),
}
with open(CONFIG['results_dir'] / 'lstm_feasibility.json', 'w') as f:
    json.dump(lstm_decision, f, indent=2)

print(f'\nDecision saved to results/market/lstm_feasibility.json')

## Section 2: Temporal split construction

**Design decision:** Temporal split is madatory for the market module. Unlike the vision module (where images are time-invariant), prices are time-dependent. Training on future prices to predict past ones is leakage.

**Strategy:** Sort all transactions by date_sold, then split by date boundaries:
- Train: earliest 70% of dates
- Val: next 15% of dates
- Test: final 15% of dates

This ensures no future information leaks into training.

In [ ]:
## Define temporal split boundaries
## Sort by date (already done in data loading)
all_dates = sorted(df['date_sold'].dt.date.unique())
n_dates = len(all_dates)

train_end_idx = int(n_dates * CONFIG['train_ratio'])
val_end_idx = int(n_dates * (CONFIG['train_ratio'] + CONFIG['val_ratio']))

train_end_date = pd.Timestamp(all_dates[train_end_idx - 1])
val_end_date = pd.Timestamp(all_dates[val_end_idx - 1])

## Assign splits
df['split'] = 'test'  # default
df.loc[df['date_sold'] <= train_end_date, 'split'] = 'train'
df.loc[(df['date_sold'] > train_end_date) & (df['date_sold'] <= val_end_date), 'split'] = 'val'

print('TEMPORAL SPLIT')
print(f'Total unique dates: {n_dates}')
print(f'\nTrain: <= {train_end_date.date()}')
print(f'Val: {train_end_date.date()} < date <= {val_end_date.date()}')
print(f'Test: > {val_end_date.date()}')

for split in ['train', 'val', 'test']:
    split_df = df[df['split'] == split]
    print(f'\n{split.upper():5s}: {len(split_df):5d} rows ({len(split_df)/len(df)*100:.1f}%)  '
          f'dates: {split_df["date_sold"].min().date()} to {split_df["date_sold"].max().date()}  '
          f'unique dates: {split_df["date_sold"].dt.date.nunique()}')

In [ ]:
## Verify no future data leaks into past
train_max = df[df['split'] == 'train']['date_sold'].max()
val_min = df[df['split'] == 'val']['date_sold'].min()
val_max = df[df['split'] == 'val']['date_sold'].max()
test_min = df[df['split'] == 'test']['date_sold'].min()

assert train_max < val_min, f'Train-Val overlap! Train max: {train_max}, Val min: {val_min}'
assert val_max < test_min, f'Val-Test overlap! Val max: {val_max}, Test min: {test_min}'

print(f'Train max date: {train_max.date()}')
print(f'Val min date:   {val_min.date()}')
print(f'Val max date:   {val_max.date()}')
print(f'Test min date:  {test_min.date()}')
print(f'\nNo temporal leakage')

In [ ]:
## Justify split choice with data evidence
print('SPLIT QUALITY CHECK')

## Grade distribution per split
print('\nGrade distribution:')
for split in ['train', 'val', 'test']:
    split_df = df[df['split'] == split]
    dist = split_df['grade'].value_counts().sort_index()
    pct = dist / len(split_df) * 100
    print(f'{split:5s}: PSA8={pct.get(8,0):5.1f}%  PSA9={pct.get(9,0):5.1f}%  PSA10={pct.get(10,0):5.1f}%')

## Card name coverage per split
print(f'\nCard name coverage:')
for split in ['train', 'val', 'test']:
    split_df = df[df['split'] == split]
    names = set(split_df['card_name'].unique())
    print(f'{split:5s}: {len(names)} Pokémon: {sorted(names)}')

## Price range per split
print(f'\nPrice statistics:')
for split in ['train', 'val', 'test']:
    split_df = df[df['split'] == split]
    print(f'{split:5s}: median=${split_df["price"].median():,.0f}  '
          f'mean=${split_df["price"].mean():,.0f}  '
          f'range=[${split_df["price"].min():,.0f}, ${split_df["price"].max():,.0f}]')

In [ ]:
## Log date ranges and sample sizes per split
split_info = {}
for split in ['train', 'val', 'test']:
    split_df = df[df['split'] == split]
    split_info[split] = {
        'n_rows': len(split_df),
        'n_dates': int(split_df['date_sold'].dt.date.nunique()),
        'date_min': str(split_df['date_sold'].min().date()),
        'date_max': str(split_df['date_sold'].max().date()),
        'median_price': float(split_df['price'].median()),
    }

with open(CONFIG['results_dir'] / 'temporal_split_info.json', 'w') as f:
    json.dump(split_info, f, indent=2)

print('Split info saved')

In [ ]:
## Verify no listing_id appears across splits
## (Each listing_id is a unique transaction, so this should be trivially true,
#  but verify)
train_ids = set(df[df['split'] == 'train']['listing_id'])
val_ids = set(df[df['split'] == 'val']['listing_id'])
test_ids = set(df[df['split'] == 'test']['listing_id'])

assert len(train_ids & val_ids) == 0, 'Train-Val listing_id overlap!'
assert len(train_ids & test_ids) == 0, 'Train-Test listing_id overlap!'
assert len(val_ids & test_ids) == 0, 'Val-Test listing_id overlap!'

print(f'No listing_id overlap across splits')

In [ ]:
## Track listing_id overlap with vision embeddings
## Load vision embedding listing_ids for fusion readiness
vision_emb_path = CONFIG['embeddings_dir'] / 'visual_embeddings_v2.parquet'

if vision_emb_path.exists():
    vision_ids = set(pd.read_parquet(vision_emb_path, columns=['listing_id'])['listing_id'])
    market_ids = set(df['listing_id'])
    overlap = market_ids & vision_ids

    print(f'VISION-MARKET LISTING_ID OVERLAP')
    print(f'Market listing_ids: {len(market_ids)}')
    print(f'Vision listing_ids: {len(vision_ids)}')
    print(f'Overlap:            {len(overlap)}')
    print(f'Market-only:        {len(market_ids - vision_ids)}')
    print(f'Vision-only:        {len(vision_ids - market_ids)}')
    print(f'\nOverlap rate:     {len(overlap)/len(market_ids)*100:.1f}% of market data')

    if len(overlap) < 300:
        print(f'\nWARNING: Overlap < 300. Fusion module viability at risk!')
    else:
        print(f'\nSufficient overlap for fusion module')
else:
    print('Vision embeddings not found. Will check in Section 12')

## Section 3: Feature engineering

**Core of the market module.** Features grouped by class as specified in the problem statement. Every rolling/lag feature uses strictly past data only. The current transaction must not be included in its own rolling window.

**Feature classes:**
- 3a: Static features (grade, card_name, rarity)
- 3b: Rolling price features (7d, 14d, 30d)
- 3c: Momentum features
- 3d: Volume features
- 3e: Volatility features
- 3f: Calendar features

**Critical constraint:** All temporal features are computed per card_name+grade group using time-based windows with strictly past data.

In [ ]:
## Preserve original dataframe before feature engineering
df_original = df.copy()
print(f'Starting feature engineering on {len(df)} rows')
print(f'Date range: {df["date_sold"].min().date()} to {df["date_sold"].max().date()}')

In [ ]:
## Static Features
## Grade (ordinal. Use as numeric)
df['grade_numeric'] = df['grade'].astype(int)

## Card name (label encoding)
card_encoder = LabelEncoder()
df['card_name_encoded'] = card_encoder.fit_transform(df['card_name'])
card_name_map = dict(zip(card_encoder.classes_, card_encoder.transform(card_encoder.classes_)))
print(f'Card name encoding: {card_name_map}')

## Rarity encoding
rarity_encoder = LabelEncoder()
df['rarity_encoded'] = rarity_encoder.fit_transform(df['rarity'].fillna('Unknown'))
rarity_map = dict(zip(rarity_encoder.classes_, rarity_encoder.transform(rarity_encoder.classes_)))
print(f'Rarity encoding: {rarity_map}')

## Verify no target leakage in encoding
## Label encoding uses category identity, not price. No leakage
print(f'\nStatic features added: grade_numeric, card_name_encoded, rarity_encoded')
print(f'Confirmed. No leakage from target variable in encoding')

In [ ]:
## Rolling Price Features
## Critical: Use strictly past data. For each transaction, the rolling window
## includes only transactions before the current one.

## Implementation: For each card_name+grade group, sort by date, then compute
## time-based rolling statistics using only preceding transactions.

def compute_rolling_features(group, windows=[7, 14, 30], min_obs=2):
    """Compute rolling price features for a single card_name+grade group.

    Uses strictly past data: for transaction at time t, only uses
    transactions with date_sold < t (not <=, excluding current).
    """
    group = group.sort_values('date_sold').copy()

    for w in windows:
        roll_mean = []
        roll_median = []
        roll_std = []
        roll_count = []

        for idx, row in group.iterrows():
            current_date = row['date_sold']
            window_start = current_date - pd.Timedelta(days=w)

            ## STRICTLY PAST: date_sold < current_date (not <=)
            past_mask = (group['date_sold'] < current_date) & (group['date_sold'] >= window_start)
            past_prices = group.loc[past_mask, 'price']

            if len(past_prices) >= min_obs:
                roll_mean.append(past_prices.mean())
                roll_median.append(past_prices.median())
                roll_std.append(past_prices.std() if len(past_prices) > 1 else 0.0)
                roll_count.append(len(past_prices))
            else:
                roll_mean.append(np.nan)
                roll_median.append(np.nan)
                roll_std.append(np.nan)
                roll_count.append(len(past_prices))

        group[f'roll_mean_{w}d'] = roll_mean
        group[f'roll_median_{w}d'] = roll_median
        group[f'roll_std_{w}d'] = roll_std
        group[f'roll_count_{w}d'] = roll_count

    return group


start_time = time.time()
df = df.groupby(['card_name', 'grade'], group_keys=False).apply(
    lambda g: compute_rolling_features(g, windows=CONFIG['rolling_windows'],
                                        min_obs=CONFIG['min_obs_per_window'])
)
df = df.sort_values('date_sold').reset_index(drop=True)

elapsed = time.time() - start_time
print(f'Rolling features computed in {elapsed:.0f}s')

##mVerify no inclusion of current transaction
## Spot check: for a random row, verify the rolling mean excludes its own price
sample_idx = df[df['roll_mean_7d'].notna()].index[50]
sample_row = df.loc[sample_idx]
sample_group = df[(df['card_name'] == sample_row['card_name']) &
                   (df['grade'] == sample_row['grade'])]
past_7d = sample_group[
    (sample_group['date_sold'] < sample_row['date_sold']) &
    (sample_group['date_sold'] >= sample_row['date_sold'] - pd.Timedelta(days=7))
]['price']

if len(past_7d) >= 2:
    expected = past_7d.mean()
    actual = sample_row['roll_mean_7d']
    assert abs(expected - actual) < 0.01, f'Rolling mean mismatch! Expected {expected}, got {actual}'
    print(f'\nSpot check passed: roll_mean_7d={actual:.2f}, manual={expected:.2f}')

rolling_cols = [c for c in df.columns if c.startswith('roll_')]
print(f'\nRolling features added: {len(rolling_cols)}')
print(f'Features: {rolling_cols}')
print(f'\nConfirmed. Strictly past rolling windows')

In [ ]:
## Momentum Features
## Price change over 7d and 14d windows, computed from rolling means

def compute_momentum_features(group, windows=[7, 14]):
    """Compute momentum: how much the rolling average has changed."""
    group = group.sort_values('date_sold').copy()

    for w in windows:
        momentum_abs = []
        momentum_pct = []

        for idx, row in group.iterrows():
            current_date = row['date_sold']

            ## Current window: past w days
            curr_start = current_date - pd.Timedelta(days=w)
            curr_mask = (group['date_sold'] < current_date) & (group['date_sold'] >= curr_start)
            curr_prices = group.loc[curr_mask, 'price']

            ## Previous window: w to 2w days ago
            prev_start = current_date - pd.Timedelta(days=2*w)
            prev_mask = (group['date_sold'] < curr_start) & (group['date_sold'] >= prev_start)
            prev_prices = group.loc[prev_mask, 'price']

            if len(curr_prices) >= 2 and len(prev_prices) >= 2:
                curr_mean = curr_prices.mean()
                prev_mean = prev_prices.mean()
                momentum_abs.append(curr_mean - prev_mean)
                momentum_pct.append((curr_mean - prev_mean) / prev_mean * 100 if prev_mean > 0 else 0)
            else:
                momentum_abs.append(np.nan)
                momentum_pct.append(np.nan)

        group[f'momentum_abs_{w}d'] = momentum_abs
        group[f'momentum_pct_{w}d'] = momentum_pct

    return group

print('Compute momentum features')
start_time = time.time()
df = df.groupby(['card_name', 'grade'], group_keys=False).apply(
    lambda g: compute_momentum_features(g, windows=[7, 14])
)
df = df.sort_values('date_sold').reset_index(drop=True)
elapsed = time.time() - start_time

momentum_cols = [c for c in df.columns if c.startswith('momentum_')]
print(f'Momentum features computed in {elapsed:.0f}s')
print(f'Features: {momentum_cols}')

In [ ]:
## Volume Features
## Transaction count in windows. Both per-card and market-wide

def compute_volume_features(group, windows=[7, 14, 30]):
    """Count transactions in past windows (per card+grade group)."""
    group = group.sort_values('date_sold').copy()

    for w in windows:
        volumes = []
        for idx, row in group.iterrows():
            current_date = row['date_sold']
            window_start = current_date - pd.Timedelta(days=w)
            past_mask = (group['date_sold'] < current_date) & (group['date_sold'] >= window_start)
            volumes.append(past_mask.sum())
        group[f'card_volume_{w}d'] = volumes

    return group

print('Compute per-card volume features')
start_time = time.time()
df = df.groupby(['card_name', 'grade'], group_keys=False).apply(
    lambda g: compute_volume_features(g, windows=CONFIG['rolling_windows'])
)
df = df.sort_values('date_sold').reset_index(drop=True)

## Market-wide volume (across all cards)
for w in CONFIG['rolling_windows']:
    market_vol = []
    for idx, row in df.iterrows():
        current_date = row['date_sold']
        window_start = current_date - pd.Timedelta(days=w)
        past_mask = (df['date_sold'] < current_date) & (df['date_sold'] >= window_start)
        market_vol.append(past_mask.sum())
    df[f'market_volume_{w}d'] = market_vol

elapsed = time.time() - start_time

volume_cols = [c for c in df.columns if 'volume' in c]
print(f'Volume features computed in {elapsed:.0f}s')
print(f'Features: {volume_cols}')

In [ ]:
## Volatility Features
## Already computed roll_std. Add coefficient of variation.

for w in CONFIG['rolling_windows']:
    mean_col = f'roll_mean_{w}d'
    std_col = f'roll_std_{w}d'
    cv_col = f'roll_cv_{w}d'

    ## Coefficient of variation = std / mean (handles scale differences)
    df[cv_col] = np.where(
        df[mean_col] > 0,
        df[std_col] / df[mean_col],
        np.nan
    )

volatility_cols = [c for c in df.columns if 'cv_' in c or 'std_' in c]
print(f'Volatility features: {volatility_cols}')

In [ ]:
## Calendar Features
## Day of week, month
df['day_of_week'] = df['date_sold'].dt.dayofweek
df['month'] = df['date_sold'].dt.month

## Days since first transaction in dataset
df['days_since_start'] = (df['date_sold'] - df['date_sold'].min()).dt.days

## Verify no leakage from future timestamps
## Calendar features are derived from date_sold of the current transaction
## This is the transaction's own timestamp, not future information

calendar_cols = ['day_of_week', 'month', 'days_since_start']
print(f'Calendar features: {calendar_cols}')

In [ ]:
## Feature summary
static_cols = ['grade_numeric', 'card_name_encoded', 'rarity_encoded']
rolling_cols = [c for c in df.columns if c.startswith('roll_mean') or c.startswith('roll_median')]
momentum_cols = [c for c in df.columns if c.startswith('momentum_')]
volume_cols = [c for c in df.columns if 'volume' in c]
volatility_cols = [c for c in df.columns if 'cv_' in c or c.startswith('roll_std')]
calendar_cols = ['day_of_week', 'month', 'days_since_start']

all_feature_cols = static_cols + rolling_cols + momentum_cols + volume_cols + volatility_cols + calendar_cols
## Remove duplicates while preserving order
all_feature_cols = list(dict.fromkeys(all_feature_cols))

## Also include roll_count columns as features (transaction density signal)
count_cols = [c for c in df.columns if c.startswith('roll_count')]
all_feature_cols += count_cols
all_feature_cols = list(dict.fromkeys(all_feature_cols))

print('FEATURE ENGINEERING SUMMARY')
print(f'Static features:  {len(static_cols):3d}  {static_cols}')
print(f'Rolling price:    {len(rolling_cols):3d}  {rolling_cols}')
print(f'Momentum:         {len(momentum_cols):3d}  {momentum_cols}')
print(f'Volume:           {len(volume_cols):3d}  {volume_cols}')
print(f'Volatility:       {len(volatility_cols):3d}  {volatility_cols}')
print(f'Calendar:         {len(calendar_cols):3d}  {calendar_cols}')
print(f'Count:            {len(count_cols):3d}  {count_cols}')
print(f'\nTOTAL FEATURES: {len(all_feature_cols)}')

## Section 4: Missingness and data survival


Feature engineering drops records. Early transactions in each group have insufficient history for rolling windows. This section quantifies the damage.

In [ ]:
## Missingness rate per feature
print('FEATURE MISSINGNESS')
for col in all_feature_cols:
    n_missing = df[col].isna().sum()
    pct = n_missing / len(df) * 100
    if pct > 0:
        print(f'{col:25s}: {n_missing:5d} missing ({pct:5.1f}%)')
    else:
        print(f'{col:25s}: complete')

In [ ]:
## Rows remaining after dropping NaN in key rolling features
## Strategy: XGBoost handles NaN natively, so we don't need to drop rows.
## But we document how many rows have complete features vs partial.

key_rolling = ['roll_mean_7d', 'roll_mean_14d', 'roll_mean_30d']
has_7d = df['roll_mean_7d'].notna().sum()
has_14d = df['roll_mean_14d'].notna().sum()
has_30d = df['roll_mean_30d'].notna().sum()
has_all = df[key_rolling].notna().all(axis=1).sum()

print(f'DATA SURVIVAL')
print(f'Total rows:      {len(df)}')
print(f'Has 7d rolling:  {has_7d} ({has_7d/len(df)*100:.1f}%)')
print(f'Has 14d rolling: {has_14d} ({has_14d/len(df)*100:.1f}%)')
print(f'Has 30d rolling: {has_30d} ({has_30d/len(df)*100:.1f}%)')
print(f'Has all rolling: {has_all} ({has_all/len(df)*100:.1f}%)')
print(f'\nDecision: Keep all rows. XGBoost handles NaN natively.')
print(f'Early transactions with missing rolling features will use')
print(f'static features + calendar features + available partial windows.')

In [ ]:
## Check which card_name+grade groups lose the most data
print('MISSINGNESS BY GROUP')
for (card, grade), group in df.groupby(['card_name', 'grade']):
    total = len(group)
    complete = group[key_rolling].notna().all(axis=1).sum()
    pct = complete / total * 100
    flag = ' danger' if pct < 50 else ''
    print(f'{card:12s} PSA {grade}: {complete:4d}/{total:4d} complete ({pct:5.1f}%){flag}')

In [ ]:
## Verify dataset remains viable
## Per split: how many rows have at least 7d rolling features?
print('SURVIVAL BY SPLIT')
for split in ['train', 'val', 'test']:
    split_mask = df['split'] == split
    total = split_mask.sum()
    has_features = (split_mask & df['roll_mean_7d'].notna()).sum()
    print(f'{split:5s}: {has_features}/{total} have 7d rolling ({has_features/total*100:.1f}%)')

## The test set should have high coverage since it's the most recent period
test_coverage = (df[df['split'] == 'test']['roll_mean_7d'].notna()).mean() * 100
assert test_coverage > 50, f'Test set has only {test_coverage:.1f}% 7d rolling coverage!'

print(f'\nConfirmed. Dataset viable')

In [ ]:
## Document NaN handling strategy
print('NaN HANDLING STRATEGY')
print('XGBoost handles missing values natively by learning optimal')
print('split directions for NaN at each tree node. No imputation needed.')
print('')
print('For the LSTM baseline (Section 7), missing values will be')
print('forward-filled within each card+grade group, then zero-filled')
print('for remaining NaN (sequence start).')

In [ ]:
## Save engineered features
df.to_parquet(CONFIG['features_dir'] / 'market_features.parquet', index=False)
print(f'Saved market features: {df.shape}')
print(f'Columns: {len(df.columns)}')
print(f'Feature columns: {len(all_feature_cols)}')

## Save feature column list for reference
feature_info = {
    'all_features': all_feature_cols,
    'static': static_cols,
    'rolling': rolling_cols,
    'momentum': momentum_cols,
    'volume': volume_cols,
    'volatility': volatility_cols,
    'calendar': calendar_cols,
    'count': count_cols,
    'total_features': len(all_feature_cols),
}
with open(CONFIG['results_dir'] / 'feature_columns.json', 'w') as f:
    json.dump(feature_info, f, indent=2)

print(f'Feature list saved to results/market/feature_columns.json')

In [ ]:
## Count check
df[df['roll_mean_7d'].isna()]['roll_count_7d'].value_counts(dropna=False).head()

## Section 5: Target variable handling

Price distribution is heavily right-skewed (Section 1 found skew = 7.73 on raw
price, 0.46 on log). Log transformation is confirmed. This section locks down
the training target and the evaluation convention.

**Training target:** log_price = log(price + 1)

**Reporting convention:**
- Error metrics (MAE, RMSE, MAPE) reported in **original dollar scale**
  via exp(prediction) - 1
- R² reported on **log scale** (standard convention for log-transformed
  regression, avoids dominance by high-value cards)

The +1 in log(price + 1) is defensive. No zero prices exist in the dataset
but the transform remains valid if any appear.

In [ ]:
## Construct log target and validate
print('TARGET VARIABLE CONSTRUCTION')

## Sanity: no zero or negative prices
assert (df['price'] > 0).all(), f'Found non-positive prices: {(df["price"] <= 0).sum()}'
print(f'Price range: ${df["price"].min():.2f} to ${df["price"].max():,.2f}')

## Construct log target
df['log_price'] = np.log1p(df['price'])  ## log(1 + price)

## Distribution comparison
print(f'\nRaw price    : skew={df["price"].skew():.2f}, kurt={df["price"].kurt():.2f}')
print(f'Log price    : skew={df["log_price"].skew():.2f}, kurt={df["log_price"].kurt():.2f}')
print(f'\nRaw price    : min={df["price"].min():.2f}, max={df["price"].max():,.2f}, '
      f'median={df["price"].median():.2f}')
print(f'Log price    : min={df["log_price"].min():.3f}, max={df["log_price"].max():.3f}, '
      f'median={df["log_price"].median():.3f}')

## Verify round-trip is lossless
reconstructed = np.expm1(df['log_price'])
max_error = (reconstructed - df['price']).abs().max()
assert max_error < 1e-6, f'Round-trip error: {max_error}'
print(f'\nRound-trip check passed (max error: {max_error:.2e})')

In [ ]:
## Compare target distribution across temporal splits
## This exposes the nonstationarity already flagged in the handoff

print('TARGET DISTRIBUTION BY SPLIT')
print(f'{"Split":<6} {"n":>5} {"mean_$":>10} {"median_$":>10} '
      f'{"mean_log":>10} {"std_log":>8}')
for split in ['train', 'val', 'test']:
    s = df[df['split'] == split]
    print(f'{split:<6} {len(s):>5d} ${s["price"].mean():>9,.0f} '
          f'${s["price"].median():>9,.0f} {s["log_price"].mean():>10.3f} '
          f'{s["log_price"].std():>8.3f}')

## Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for split, color in zip(['train', 'val', 'test'], ['tab:blue', 'tab:orange', 'tab:green']):
    s = df[df['split'] == split]
    axes[0].hist(s['price'], bins=80, alpha=0.5, label=split, color=color,
                 density=True, range=(0, 3000))
    axes[1].hist(s['log_price'], bins=60, alpha=0.5, label=split, color=color,
                 density=True)

axes[0].set_xlabel('Price ($)'); axes[0].set_ylabel('Density')
axes[0].set_title('Raw price distribution (truncated at $3K for readability)')
axes[0].legend()

axes[1].set_xlabel('log(price + 1)'); axes[1].set_ylabel('Density')
axes[1].set_title('Log price distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig(CONFIG['results_dir'] / 'figures' / 'target_distribution_by_split.png',
            dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
## Define evaluation convention as reusable helpers
## All downstream sections use these. No ad-hoc metric code elsewhere.

def evaluate_predictions(y_true_log, y_pred_log, label=''):
    """
    Evaluate log-space predictions against log-space truth.
    Returns dict with metrics in dollar space (MAE, RMSE, MAPE) and log space (R²).

    Parameters
    ----------
    y_true_log : array-like, log(price+1) actuals
    y_pred_log : array-like, log(price+1) predictions
    label : optional, for printing

    Returns
    -------
    dict with keys: mae_usd, rmse_usd, mape_pct, r2_log, n
    """
    y_true_log = np.asarray(y_true_log)
    y_pred_log = np.asarray(y_pred_log)

    ## Dollar space for error metrics
    y_true_usd = np.expm1(y_true_log)
    y_pred_usd = np.expm1(y_pred_log)

    ## Clip negative predictions (expm1 of negative log-pred = negative dollars)
    ## This is a legitimate adjustment — no card can be worth < $0
    y_pred_usd = np.clip(y_pred_usd, 0, None)

    mae_usd  = mean_absolute_error(y_true_usd, y_pred_usd)
    rmse_usd = np.sqrt(mean_squared_error(y_true_usd, y_pred_usd))

    ## MAPE: guard against zero truths (none expected)
    nonzero = y_true_usd > 0
    mape_pct = np.mean(np.abs((y_true_usd[nonzero] - y_pred_usd[nonzero])
                              / y_true_usd[nonzero])) * 100

    ## R² on log scale (standard for log-transformed regression)
    r2_log = r2_score(y_true_log, y_pred_log)

    results = {
        'mae_usd':  float(mae_usd),
        'rmse_usd': float(rmse_usd),
        'mape_pct': float(mape_pct),
        'r2_log':   float(r2_log),
        'n':        int(len(y_true_log)),
    }

    if label:
        print(f'{label:<25} n={results["n"]:>4d}  '
              f'MAE=${results["mae_usd"]:>7,.2f}  '
              f'RMSE=${results["rmse_usd"]:>8,.2f}  '
              f'MAPE={results["mape_pct"]:>6.1f}%  '
              f'R²(log)={results["r2_log"]:>6.3f}')

    return results


## Verify the function with a trivial sanity test: predicting the mean
mean_log_train = df[df['split'] == 'train']['log_price'].mean()
sanity = evaluate_predictions(
    y_true_log=df[df['split'] == 'test']['log_price'].values,
    y_pred_log=np.full(len(df[df['split'] == 'test']), mean_log_train),
    label='Sanity (predict train mean)'
)

## Save evaluation convention to config for reproducibility
with open(CONFIG['results_dir'] / 'target_handling.json', 'w') as f:
    json.dump({
        'target_column': 'log_price',
        'transform': 'log1p (i.e. log(price + 1))',
        'inverse_transform': 'expm1 (i.e. exp(pred) - 1)',
        'metrics_dollar_scale': ['mae_usd', 'rmse_usd', 'mape_pct'],
        'metrics_log_scale': ['r2_log'],
        'clip_negative_predictions': True,
        'sanity_baseline_predict_train_mean': sanity,
        'price_min': float(df['price'].min()),
        'price_max': float(df['price'].max()),
        'price_skew_raw': float(df['price'].skew()),
        'price_skew_log': float(df['log_price'].skew()),
    }, f, indent=2)

print(f'\nTarget handling locked.')
print(f'Evaluation helper: evaluate_predictions(y_true_log, y_pred_log, label)')

## Section 6. Primary model training: XGBoost hybrid regressor

Train the primary market-state model on all 31 engineered features.
Monitor val split for early stopping. Evaluate on test exactly once at the end.

**Discipline:**
- Training: Train split only
- Model selection (num trees): Val split via early stopping
- Final evaluation: Test split, once
- All metrics via `evaluate_predictions()` (Section 5)

This section addresses market-encoder partially. Full market-encoder comparison requires
the static baseline and LSTM baseline in Section 7.

In [ ]:
## Construct feature matrices and targets
## Feature column order is locked here and reused everywhere downstream
## Reload feature list to guarantee order matches saved artefact

with open(CONFIG['results_dir'] / 'feature_columns.json', 'r') as f:
    feature_info = json.load(f)
FEATURE_COLS = feature_info['all_features']
print(f'Feature count: {len(FEATURE_COLS)}')
assert len(FEATURE_COLS) == 31, f'Expected 31 features, got {len(FEATURE_COLS)}'

## Split-specific matrices
X = {}
y = {}
listing_ids = {}
for split in ['train', 'val', 'test']:
    mask = df['split'] == split
    X[split] = df.loc[mask, FEATURE_COLS].values
    y[split] = df.loc[mask, 'log_price'].values
    listing_ids[split] = df.loc[mask, 'listing_id'].values
    print(f'{split:<6}: X shape {X[split].shape}, y shape {y[split].shape}')

## Sanity: no overlap between splits
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    overlap = set(listing_ids[a]) & set(listing_ids[b])
    assert len(overlap) == 0, f'{a}/{b} overlap: {len(overlap)}'
print('No listing_id overlap across splits')

## NaN audit per split (should match Section 4 missingness, split-scoped)
print(f'\nNaN counts per split (total cells):')
for split in ['train', 'val', 'test']:
    n_nan = np.isnan(X[split]).sum()
    total = X[split].size
    print(f'{split:<6}: {n_nan:>6,}/{total:>7,} NaN cells ({n_nan/total*100:.1f}%)')
print('(XGBoost handles NaN natively. No imputation)')

In [ ]:
## Train primary XGBoost model
## Monitor val for early stopping. Track train+val learning curves.

print('TRAINING PRIMARY XGBOOST MODEL')
print(f'Params: {CONFIG["xgb_params"]}')

## Construct DMatrices (native XGBoost API, NaN-aware)
dtrain = xgb.DMatrix(X['train'], label=y['train'], feature_names=FEATURE_COLS)
dval   = xgb.DMatrix(X['val'],   label=y['val'],   feature_names=FEATURE_COLS)
dtest  = xgb.DMatrix(X['test'],  label=y['test'],  feature_names=FEATURE_COLS)

## Training parameters
xgb_params = {
    'objective':        'reg:squarederror',
    'max_depth':        CONFIG['xgb_params']['max_depth'],
    'learning_rate':    CONFIG['xgb_params']['learning_rate'],
    'subsample':        CONFIG['xgb_params']['subsample'],
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'reg_alpha':        0.0,
    'reg_lambda':       1.0,
    'seed':             CONFIG['seed'],
    'verbosity':        0,
}

## Train with early stopping on val
evals_result = {}
import time
t0 = time.time()
model_primary = xgb.train(
    params=xgb_params,
    dtrain=dtrain,
    num_boost_round=CONFIG['xgb_params']['n_estimators'],
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=30,
    evals_result=evals_result,
    verbose_eval=50,
)
elapsed = time.time() - t0

print(f'\nTraining complete in {elapsed:.1f}s')
print(f'Best iteration: {model_primary.best_iteration}')
print(f'Best val RMSE (log): {model_primary.best_score:.4f}')

In [ ]:
## Learning curves. Train vs val RMSE in log space

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(evals_result['train']['rmse'], label='train', color='tab:blue')
ax.plot(evals_result['val']['rmse'],   label='val',   color='tab:orange')
ax.axvline(model_primary.best_iteration, color='red', linestyle='--',
           alpha=0.6, label=f'best iter ({model_primary.best_iteration})')
ax.set_xlabel('Boosting round')
ax.set_ylabel('RMSE (log space)')
ax.set_title('Primary XGBoost learning curves')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CONFIG['results_dir'] / 'figures' / 'primary_learning_curve.png',
            dpi=120, bbox_inches='tight')
plt.show()

## Flag: if train RMSE is much lower than val RMSE at best_iter, overfitting
final_train = evals_result['train']['rmse'][model_primary.best_iteration]
final_val   = evals_result['val']['rmse'][model_primary.best_iteration]
gap = final_val - final_train
print(f'\nAt best iteration ({model_primary.best_iteration}):')
print(f'train RMSE (log): {final_train:.4f}')
print(f'val   RMSE (log): {final_val:.4f}')
print(f'gap:              {gap:.4f}')
if gap > 0.3:
    print('FLAG: Large train-val gap. Possible overfitting or severe train-val drift.')
elif gap > 0.15:
    print('NOTE: Moderate gap. Review in Section 9 stability checks.')
else:
    print('Gap within acceptable range.')

In [ ]:
## Generate predictions and evaluate
## Test is touched exactly once, here.

## Predict (use best_iteration to honour early stopping)
pred = {}
for split, dmat in [('train', dtrain), ('val', dval), ('test', dtest)]:
    pred[split] = model_primary.predict(
        dmat, iteration_range=(0, model_primary.best_iteration + 1)
    )

## Evaluate all three splits
print('PRIMARY XGBOOST: EVALUATION')
results_primary = {}
for split in ['train', 'val', 'test']:
    results_primary[split] = evaluate_predictions(
        y_true_log=y[split],
        y_pred_log=pred[split],
        label=f'Primary / {split}'
    )

## Check for pathological negative log predictions (flagged in Section 5)
for split in ['train', 'val', 'test']:
    n_neg = (pred[split] < 0).sum()
    if n_neg > 0:
        print(f'FLAG: {split} has {n_neg} negative log predictions '
              f'(min: {pred[split].min():.2f})')
    else:
        print(f'{split}: no negative log predictions (min: {pred[split].min():.2f})')

In [ ]:
## Residual analysis. Use VAL split only, keep test untouched for diagnostics
## Residual = y_true_log - y_pred_log (log-space residuals)

resid_val = y['val'] - pred['val']
y_val_usd = np.expm1(y['val'])
pred_val_usd = np.clip(np.expm1(pred['val']), 0, None)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

## 1. Residuals vs predicted (log space). Heteroscedasticity check
axes[0, 0].scatter(pred['val'], resid_val, alpha=0.3, s=12)
axes[0, 0].axhline(0, color='red', linestyle='--', alpha=0.7)
axes[0, 0].set_xlabel('Predicted log price')
axes[0, 0].set_ylabel('Residual (log space)')
axes[0, 0].set_title('Residuals vs predicted (val)')
axes[0, 0].grid(alpha=0.3)

## 2. Residual distribution
axes[0, 1].hist(resid_val, bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].axvline(0, color='red', linestyle='--', alpha=0.7)
axes[0, 1].axvline(resid_val.mean(), color='orange', linestyle='--',
                    alpha=0.7, label=f'mean={resid_val.mean():.3f}')
axes[0, 1].set_xlabel('Residual (log space)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title(f'Residual distribution (val)')
axes[0, 1].legend()

## 3. Predicted vs actual (dollar space)
axes[1, 0].scatter(y_val_usd, pred_val_usd, alpha=0.3, s=12)
lim = max(y_val_usd.max(), pred_val_usd.max())
axes[1, 0].plot([0, lim], [0, lim], color='red', linestyle='--', alpha=0.7)
axes[1, 0].set_xlabel('Actual price ($)')
axes[1, 0].set_ylabel('Predicted price ($)')
axes[1, 0].set_title('Predicted vs actual (val, dollar space)')
axes[1, 0].set_xscale('log')
axes[1, 0].set_yscale('log')
axes[1, 0].grid(alpha=0.3, which='both')

## 4. Residual by grade (bias check)
val_df = df[df['split'] == 'val'].copy()
val_df['resid'] = resid_val
for g in sorted(val_df['grade'].unique()):
    axes[1, 1].boxplot(
        val_df[val_df['grade'] == g]['resid'].values,
        positions=[int(g)], widths=0.6,
    )
axes[1, 1].axhline(0, color='red', linestyle='--', alpha=0.7)
axes[1, 1].set_xlabel('PSA grade')
axes[1, 1].set_ylabel('Residual (log space)')
axes[1, 1].set_title('Residuals by grade (val)')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CONFIG['results_dir'] / 'figures' / 'primary_residuals_val.png',
            dpi=120, bbox_inches='tight')
plt.show()

## Quantify bias
print(f'\nVAL RESIDUAL DIAGNOSTICS')
print(f'Mean residual: {resid_val.mean():.4f} (|mean| should be near 0 if unbiased)')
print(f'Median residual: {np.median(resid_val):.4f}')
print(f'Std residual: {resid_val.std():.4f}')
print(f'\nMean residual by grade:')
for g in sorted(val_df['grade'].unique()):
    g_resid = val_df[val_df['grade'] == g]['resid']
    print(f'PSA {g}: mean={g_resid.mean():+.3f}, median={g_resid.median():+.3f}, n={len(g_resid)}')

In [ ]:
## Verify model is not spike-specific
## Split test into temporal thirds and evaluate each separately

test_df = df[df['split'] == 'test'].copy()
test_df['pred_log'] = pred['test']
test_df = test_df.sort_values('date_sold').reset_index(drop=True)

n_test = len(test_df)
segment_size = n_test // 3

print(f'TEMPORAL SEGMENT ANALYSIS (test split, n={n_test})')
print(f'Checking whether performance is stable across test period')

segment_results = {}
for i, name in enumerate(['early', 'middle', 'late']):
    if i < 2:
        seg = test_df.iloc[i*segment_size:(i+1)*segment_size]
    else:
        seg = test_df.iloc[i*segment_size:]

    date_range = f'{seg["date_sold"].min().date()} to {seg["date_sold"].max().date()}'
    print(f'\n{name.upper()} segment: {date_range} (n={len(seg)})')
    segment_results[name] = evaluate_predictions(
        y_true_log=seg['log_price'].values,
        y_pred_log=seg['pred_log'].values,
        label=f'test/{name}'
    )

## Additionally: evaluate on rolling-available vs rolling-missing test rows
## This directly addresses risk R-M3 flagged during Section 4 review
test_has_roll = test_df['roll_mean_7d'].notna()
print(f'\nCOVERAGE-STRATIFIED TEST PERFORMANCE (addresses R-M3)')
print(f'(Rolling-available vs cold-start rows within test)')

results_by_coverage = {}
for has_roll, label in [(True, 'has_7d_rolling'), (False, 'cold_start')]:
    mask = test_has_roll == has_roll
    if mask.sum() > 0:
        results_by_coverage[label] = evaluate_predictions(
            y_true_log=test_df.loc[mask, 'log_price'].values,
            y_pred_log=test_df.loc[mask, 'pred_log'].values,
            label=f'test/{label}'
        )
        print(f'n={mask.sum()}')

In [ ]:
## Save primary model, predictions, and all results
models_dir = CONFIG['models_dir'] if 'models_dir' in CONFIG else Path(PROJECT_ROOT / 'models/market')
models_dir = Path(models_dir)
models_dir.mkdir(parents=True, exist_ok=True)

## Model (JSON format. Portable, future-proof)
model_primary.save_model(str(models_dir / 'primary_xgboost.json'))

## Predictions (with listing_id for fusion module alignment)
pred_records = []
for split in ['train', 'val', 'test']:
    for lid, y_true, y_pred in zip(listing_ids[split], y[split], pred[split]):
        pred_records.append({
            'listing_id': lid,
            'split': split,
            'log_price_actual':    float(y_true),
            'log_price_predicted': float(y_pred),
        })
pred_df = pd.DataFrame(pred_records)
pred_df.to_parquet(CONFIG['results_dir'] / 'primary_predictions.parquet', index=False)
print(f'Saved predictions: {pred_df.shape}')

## Consolidated results JSON
primary_results = {
    'model': 'xgboost_primary',
    'features': FEATURE_COLS,
    'n_features': len(FEATURE_COLS),
    'params': xgb_params,
    'best_iteration': int(model_primary.best_iteration),
    'best_val_rmse_log': float(model_primary.best_score),
    'metrics': results_primary,
    'segment_metrics': segment_results,
    'coverage_stratified_metrics': results_by_coverage,
    'val_residual_diagnostics': {
        'mean':   float(resid_val.mean()),
        'median': float(np.median(resid_val)),
        'std':    float(resid_val.std()),
    },
}
with open(CONFIG['results_dir'] / 'primary_xgboost_results.json', 'w') as f:
    json.dump(primary_results, f, indent=2, default=str)

print(f'\nALL SAVED')
print(f'Model:         {models_dir / "primary_xgboost.json"}')
print(f'Predictions:   {CONFIG["results_dir"] / "primary_predictions.parquet"}')
print(f'Results JSON:  {CONFIG["results_dir"] / "primary_xgboost_results.json"}')

In [ ]:
## Diagnostic: does the primary model weight rolling features at all?
## This is to determine whether the model actually learned anything useful from the rollong features
## or whether it would be better to drop them entirely
feature_importance = model_primary.feature_importances_ if hasattr(model_primary, 'feature_importances_') else None
if feature_importance is None:
    ## Native booster API
    score = model_primary.get_score(importance_type='gain')
    total = sum(score.values())
    importance_dict = {k: v/total for k, v in score.items()}
else:
    importance_dict = dict(zip(FEATURE_COLS, feature_importance))

## Group by feature class
import json
with open(CONFIG['results_dir'] / 'feature_columns.json') as f:
    finfo = json.load(f)

for cls in ['static', 'rolling', 'momentum', 'volume', 'volatility', 'calendar', 'count']:
    cols = finfo.get(cls, [])
    imp = sum(importance_dict.get(c, 0) for c in cols)
    print(f'{cls:<12}: {imp*100:5.1f}%  (n={len(cols)})')

## Section 7: Baseline comparisons. market-encoder core experiment

Build the baselines the primary model must beat to support market-encoder.

**Models compared:**
1. Sanity: predict train mean (floor)
2. Static-only XGBoost: grade, card_name, rarity (no temporal information)
3. Static + calendar XGBoost: adds day_of_week, month, days_since_start
4. Primary XGBoost: all 31 engineered features (from Section 6)
5. LSTM: per-group sequences with time-gap features
6. Card+grade median: non-ML baseline (training-period median per group)

**Critical subgroup analysis** (given Section 6 findings):
- Coverage-stratified (has_7d_rolling vs cold_start)
- Temporal thirds (early/middle/late on test)

This addresses the drift finding from Section 6 directly.

In [ ]:
## Static baselines: two variants for clean comparison

STATIC_ONLY_COLS = ['grade_numeric', 'card_name_encoded', 'rarity_encoded']
STATIC_PLUS_CAL_COLS = STATIC_ONLY_COLS + ['day_of_week', 'month', 'days_since_start']

## Same xgb_params as primary for fair comparison
## Using same CONFIG values so any differences come from features, not hyperparams

def train_xgb_baseline(feature_cols, label):
    """Train XGBoost on a subset of features with same protocol as primary."""
    X_tr = df.loc[df['split']=='train', feature_cols].values
    X_va = df.loc[df['split']=='val',   feature_cols].values
    X_te = df.loc[df['split']=='test',  feature_cols].values
    y_tr = df.loc[df['split']=='train', 'log_price'].values
    y_va = df.loc[df['split']=='val',   'log_price'].values
    y_te = df.loc[df['split']=='test',  'log_price'].values

    dtr = xgb.DMatrix(X_tr, label=y_tr, feature_names=feature_cols)
    dva = xgb.DMatrix(X_va, label=y_va, feature_names=feature_cols)
    dte = xgb.DMatrix(X_te, label=y_te, feature_names=feature_cols)

    params = {
        'objective': 'reg:squarederror',
        'max_depth': CONFIG['xgb_params']['max_depth'],
        'learning_rate': CONFIG['xgb_params']['learning_rate'],
        'subsample': CONFIG['xgb_params']['subsample'],
        'colsample_bytree': 0.8,
        'min_child_weight': 3,
        'reg_lambda': 1.0,
        'seed': CONFIG['seed'],
        'verbosity': 0,
    }

    model = xgb.train(
        params=params, dtrain=dtr,
        num_boost_round=CONFIG['xgb_params']['n_estimators'],
        evals=[(dtr, 'train'), (dva, 'val')],
        early_stopping_rounds=30,
        verbose_eval=0,
    )

    pred_tr = model.predict(dtr, iteration_range=(0, model.best_iteration+1))
    pred_va = model.predict(dva, iteration_range=(0, model.best_iteration+1))
    pred_te = model.predict(dte, iteration_range=(0, model.best_iteration+1))

    print(f'\n{label} : best iter {model.best_iteration}')
    m_tr = evaluate_predictions(y_tr, pred_tr, label=f'  {label}/train')
    m_va = evaluate_predictions(y_va, pred_va, label=f'  {label}/val')
    m_te = evaluate_predictions(y_te, pred_te, label=f'  {label}/test')

    return {
        'model': model,
        'best_iteration': model.best_iteration,
        'features': feature_cols,
        'pred_train': pred_tr,
        'pred_val':   pred_va,
        'pred_test':  pred_te,
        'metrics': {'train': m_tr, 'val': m_va, 'test': m_te},
    }


static_only = train_xgb_baseline(STATIC_ONLY_COLS,    'Static-only')
static_plus_cal = train_xgb_baseline(STATIC_PLUS_CAL_COLS, 'Static+calendar')


In [ ]:
## extension: Card+grade median from training period
## Non-ML baseline. Tests whether any ML is beating "memorise the per-group average"

train_medians = (df[df['split']=='train']
                 .groupby(['card_name', 'grade'])['log_price']
                 .median())
global_median = df[df['split']=='train']['log_price'].median()

def predict_group_median(row):
    key = (row['card_name'], row['grade'])
    return train_medians.get(key, global_median)

df['pred_group_median'] = df.apply(predict_group_median, axis=1)

print('Card+grade median baseline')
median_results = {}
for split in ['train', 'val', 'test']:
    mask = df['split'] == split
    median_results[split] = evaluate_predictions(
        y_true_log=df.loc[mask, 'log_price'].values,
        y_pred_log=df.loc[mask, 'pred_group_median'].values,
        label=f'  Group median/{split}'
    )

group_median_pred = {
    'train': df.loc[df['split']=='train', 'pred_group_median'].values,
    'val':   df.loc[df['split']=='val',   'pred_group_median'].values,
    'test':  df.loc[df['split']=='test',  'pred_group_median'].values,
}

In [ ]:
## LSTM baseline
## Per-group sequences with time-gap features. Forward-fill + zero-pad sequence starts.

K_SEQ = 10  ## sequence length (truncated history)
LSTM_FEATURES = ['log_price', 'days_since_prev', 'grade_numeric',
                 'card_name_encoded', 'rarity_encoded']
N_LSTM_FEATURES = len(LSTM_FEATURES)

## Compute days_since_prev within each card+grade group
df_lstm = df.sort_values(['card_name', 'grade', 'date_sold']).reset_index(drop=True).copy()
df_lstm['days_since_prev'] = (df_lstm
    .groupby(['card_name', 'grade'])['date_sold']
    .diff()
    .dt.days
    .fillna(0))  ## first transaction in a group gets 0

print(f'days_since_prev stats: median={df_lstm["days_since_prev"].median():.1f}, '
      f'mean={df_lstm["days_since_prev"].mean():.1f}, '
      f'max={df_lstm["days_since_prev"].max():.0f}')

## Build sequences: for each row, take the K most recent prior transactions within its group
## If fewer than K exist, pad with zeros and a mask

def build_sequences(df_sorted, feature_cols, k=K_SEQ):
    """
    Returns
    -------
    X  : (N, k, F)  : sequence features (zero-padded at start if needed)
    mask : (N, k)   : 1 where real data, 0 where padding
    y  : (N,)       : target log_price for the current row
    idx: (N,)       : original df index (for alignment)
    splits: (N,)    : split labels
    listing_ids: (N,)
    """
    X, mask, y, idx, splits, lids = [], [], [], [], [], []

    for (cn, gr), g in df_sorted.groupby(['card_name', 'grade']):
        g = g.reset_index(drop=False)  ## keep original index
        for i in range(len(g)):
            hist = g.iloc[max(0, i-k):i]  ## strictly past, at most k
            n_hist = len(hist)

            seq = np.zeros((k, len(feature_cols)), dtype=np.float32)
            m   = np.zeros(k, dtype=np.float32)

            if n_hist > 0:
                seq[k-n_hist:k, :] = hist[feature_cols].values.astype(np.float32)
                m[k-n_hist:k] = 1.0

            X.append(seq)
            mask.append(m)
            y.append(g.iloc[i]['log_price'])
            idx.append(g.iloc[i]['index'])
            splits.append(g.iloc[i]['split'])
            lids.append(g.iloc[i]['listing_id'])

    return (np.stack(X), np.stack(mask), np.array(y),
            np.array(idx), np.array(splits), np.array(lids))

print('Build sequences')
t0 = time.time()
X_seq, mask_seq, y_seq, idx_seq, split_seq, lid_seq = build_sequences(
    df_lstm, LSTM_FEATURES, k=K_SEQ
)
print(f'Done in {time.time()-t0:.1f}s')
print(f'Shapes: X={X_seq.shape}, mask={mask_seq.shape}, y={y_seq.shape}')

## Split
tr_m = split_seq == 'train'
va_m = split_seq == 'val'
te_m = split_seq == 'test'
print(f'\nSplit counts (sequences): train={tr_m.sum()}, val={va_m.sum()}, test={te_m.sum()}')
print(f'Avg real-history fraction: train={mask_seq[tr_m].mean():.2f}, '
      f'val={mask_seq[va_m].mean():.2f}, test={mask_seq[te_m].mean():.2f}')

## Normalise continuous features using TRAIN statistics only (no leakage)
CONT_IDX = [0, 1]  ## log_price, days_since_prev
stats = {}
for i in CONT_IDX:
    train_vals = X_seq[tr_m][mask_seq[tr_m] > 0][:, i] if X_seq[tr_m].ndim == 3 else None
    ## Correctly flatten across sequence dimension
    vals = X_seq[tr_m, :, i][mask_seq[tr_m] > 0]
    stats[i] = (float(vals.mean()), float(vals.std() + 1e-8))

def apply_norm(X, mask):
    X = X.copy()
    for i in CONT_IDX:
        mu, sd = stats[i]
        X[:, :, i] = (X[:, :, i] - mu) / sd
        X[:, :, i] = X[:, :, i] * mask  ## zero out padding after normalisation
    return X

X_seq_n = apply_norm(X_seq, mask_seq)
print(f'Normalisation stats (train): {stats}')

In [ ]:
## LSTM model: small, regularised

class LSTMRegressor(nn.Module):
    def __init__(self, n_features, hidden=64, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x, mask):
        ## x: (B, K, F), mask: (B, K)
        lengths = mask.sum(dim=1).clamp(min=1).long()
        ## Use the LSTM output at the last real position for each sequence
        out, _ = self.lstm(x)               ## (B, K, H)
        ## Gather last real timestep per sequence
        idx = (lengths - 1).unsqueeze(1).unsqueeze(2).expand(-1, 1, out.size(-1))
        last = out.gather(1, idx).squeeze(1)  ## (B, H)
        return self.head(self.drop(last)).squeeze(-1)

## Tensors
def to_tensor(a, dtype=torch.float32):
    return torch.from_numpy(a).to(dtype)

Xt_tr, Mt_tr, yt_tr = to_tensor(X_seq_n[tr_m]), to_tensor(mask_seq[tr_m]), to_tensor(y_seq[tr_m])
Xt_va, Mt_va, yt_va = to_tensor(X_seq_n[va_m]), to_tensor(mask_seq[va_m]), to_tensor(y_seq[va_m])
Xt_te, Mt_te, yt_te = to_tensor(X_seq_n[te_m]), to_tensor(mask_seq[te_m]), to_tensor(y_seq[te_m])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

model_lstm = LSTMRegressor(N_LSTM_FEATURES, hidden=64, dropout=0.2).to(device)
opt = torch.optim.Adam(model_lstm.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = nn.MSELoss()

n_params = sum(p.numel() for p in model_lstm.parameters() if p.requires_grad)
print(f'LSTM trainable params: {n_params:,}')

## Simple batched training loop
from torch.utils.data import TensorDataset, DataLoader
train_ds = TensorDataset(Xt_tr, Mt_tr, yt_tr)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

best_val = float('inf')
patience, patience_ctr = 10, 0
history = {'train_loss': [], 'val_loss': []}
best_state = None

print('\nTraining LSTM...')
for epoch in range(1, 81):
    ## Train
    model_lstm.train()
    tr_losses = []
    for xb, mb, yb in train_loader:
        xb, mb, yb = xb.to(device), mb.to(device), yb.to(device)
        pred = model_lstm(xb, mb)
        loss = loss_fn(pred, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        tr_losses.append(loss.item())
    tr_mean = float(np.mean(tr_losses))

    ## Val
    model_lstm.eval()
    with torch.no_grad():
        val_pred = model_lstm(Xt_va.to(device), Mt_va.to(device)).cpu().numpy()
    val_loss = float(np.mean((val_pred - y_seq[va_m])**2))

    history['train_loss'].append(tr_mean)
    history['val_loss'].append(val_loss)

    if val_loss < best_val - 1e-4:
        best_val = val_loss
        best_state = {k: v.clone().cpu() for k, v in model_lstm.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f'epoch {epoch:3d}: train_loss={tr_mean:.4f}  val_loss={val_loss:.4f}  '
              f'(best={best_val:.4f}, patience={patience_ctr}/{patience})')

    if patience_ctr >= patience:
        print(f'Early stopping at epoch {epoch}')
        break

## Restore best
model_lstm.load_state_dict({k: v.to(device) for k, v in best_state.items()})
model_lstm.eval()

In [ ]:
## Evaluate LSTM on all three splits

with torch.no_grad():
    lstm_pred_tr = model_lstm(Xt_tr.to(device), Mt_tr.to(device)).cpu().numpy()
    lstm_pred_va = model_lstm(Xt_va.to(device), Mt_va.to(device)).cpu().numpy()
    lstm_pred_te = model_lstm(Xt_te.to(device), Mt_te.to(device)).cpu().numpy()

lstm_results = {
    'train': evaluate_predictions(y_seq[tr_m], lstm_pred_tr, label='LSTM/train'),
    'val':   evaluate_predictions(y_seq[va_m], lstm_pred_va, label='LSTM/val'),
    'test':  evaluate_predictions(y_seq[te_m], lstm_pred_te, label='LSTM/test'),
}

## Save LSTM predictions aligned by listing_id
lstm_pred_df = pd.DataFrame({
    'listing_id': np.concatenate([lid_seq[tr_m], lid_seq[va_m], lid_seq[te_m]]),
    'split':      np.concatenate([split_seq[tr_m], split_seq[va_m], split_seq[te_m]]),
    'log_price_actual':    np.concatenate([y_seq[tr_m], y_seq[va_m], y_seq[te_m]]),
    'log_price_predicted': np.concatenate([lstm_pred_tr, lstm_pred_va, lstm_pred_te]),
})
lstm_pred_df.to_parquet(CONFIG['results_dir'] / 'lstm_predictions.parquet', index=False)
print(f'\nSaved LSTM predictions: {lstm_pred_df.shape}')

## Training curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history['train_loss'], label='train')
ax.plot(history['val_loss'], label='val')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE (log space)')
ax.set_title('LSTM learning curves')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CONFIG['results_dir']/'figures'/'lstm_learning_curve.png',
            dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
## Comparison table on the same test set
## All models to be evaluated on the same 1050 test rows.

## Load primary already saved predictions from Section 6
primary_preds = pd.read_parquet(CONFIG['results_dir'] / 'primary_predictions.parquet')

## Align all models by listing_id
## Build a master test-level dataframe
test_master = df[df['split'] == 'test'][['listing_id', 'card_name', 'grade',
                                          'date_sold', 'price', 'log_price',
                                          'roll_mean_7d']].copy()
test_master = test_master.rename(columns={'log_price': 'actual_log'})

## Add each model's predictions
test_master = test_master.merge(
    primary_preds[primary_preds['split']=='test'][['listing_id', 'log_price_predicted']]
        .rename(columns={'log_price_predicted': 'primary_log'}),
    on='listing_id', how='left'
)
test_master['static_only_log'] = static_only['pred_test']
test_master['static_cal_log']  = static_plus_cal['pred_test']
test_master['group_med_log']   = group_median_pred['test']

## LSTM aligned by listing_id
test_master = test_master.merge(
    lstm_pred_df[lstm_pred_df['split']=='test'][['listing_id', 'log_price_predicted']]
        .rename(columns={'log_price_predicted': 'lstm_log'}),
    on='listing_id', how='left'
)

## Sanity: all predictions present
assert test_master[['primary_log','static_only_log','static_cal_log','group_med_log','lstm_log']].notna().all().all(), 'Missing predictions after merge'
print(f'Test master aligned: {len(test_master)} rows, all predictions present')

## Sanity baseline
sanity_pred = np.full(len(test_master), df[df['split']=='train']['log_price'].mean())

def eval_col(col_name, label):
    return evaluate_predictions(
        test_master['actual_log'].values,
        test_master[col_name].values if col_name else sanity_pred,
        label=label,
    )

## Main comparison
print('\n' + '-' * 100)
print('market-encoder COMPARISON TABLE: TEST SET (n=1050)')
comparison = {
    'Sanity (train mean)': eval_col(None, '1. Sanity (train mean)  '),
    'Group median':        eval_col('group_med_log',   '2. Card+grade median    '),
    'Static-only':         eval_col('static_only_log', '3. Static-only          '),
    'Static+calendar':     eval_col('static_cal_log',  '4. Static + calendar    '),
    'LSTM':                eval_col('lstm_log',        '5. LSTM                 '),
    'Primary':             eval_col('primary_log',     '6. Primary (all 31 feat)'),
}


In [ ]:
## Critical subgroup analysis given Section 6 findings

print('\nCOVERAGE-STRATIFIED TEST PERFORMANCE')
print('(Does primary beat baselines on cold-start rows? On rolling-available rows?)')

has_roll_mask = test_master['roll_mean_7d'].notna()
coverage_results = {}
for mask_val, label in [(True, 'has_7d_rolling'), (False, 'cold_start')]:
    m = has_roll_mask == mask_val
    print(f'\n  {label} (n={m.sum()})')
    coverage_results[label] = {
        'Sanity':          evaluate_predictions(
            test_master.loc[m,'actual_log'].values,
            np.full(m.sum(), df[df['split']=='train']['log_price'].mean()),
            label=f'  {label}/Sanity        '),
        'Group median':    evaluate_predictions(
            test_master.loc[m,'actual_log'].values,
            test_master.loc[m,'group_med_log'].values,
            label=f'  {label}/Group median  '),
        'Static-only':     evaluate_predictions(
            test_master.loc[m,'actual_log'].values,
            test_master.loc[m,'static_only_log'].values,
            label=f'  {label}/Static-only   '),
        'Static+calendar': evaluate_predictions(
            test_master.loc[m,'actual_log'].values,
            test_master.loc[m,'static_cal_log'].values,
            label=f'  {label}/Static+cal    '),
        'LSTM':              evaluate_predictions(
            test_master.loc[m,'actual_log'].values,
            test_master.loc[m,'lstm_log'].values,
            label=f'  {label}/LSTM         '),
        'Primary':           evaluate_predictions(
            test_master.loc[m,'actual_log'].values,
            test_master.loc[m,'primary_log'].values,
            label=f'  {label}/Primary      '),
    }

print('\n\nTEMPORAL SEGMENT ANALYSIS (test thirds)')

test_sorted = test_master.sort_values('date_sold').reset_index(drop=True)
n_seg = len(test_sorted) // 3
segment_results_all = {}
for i, name in enumerate(['early', 'middle', 'late']):
    seg = test_sorted.iloc[i*n_seg : (i+1)*n_seg if i<2 else len(test_sorted)]
    date_range = f'{seg["date_sold"].min().date()} to {seg["date_sold"].max().date()}'
    print(f'\n  {name.upper()} ({date_range}, n={len(seg)})')
    segment_results_all[name] = {
        'Static-only': evaluate_predictions(
            seg['actual_log'].values, seg['static_only_log'].values,
            label=f'  {name}/Static-only'),
        'Static+cal':  evaluate_predictions(
            seg['actual_log'].values, seg['static_cal_log'].values,
            label=f'  {name}/Static+cal '),
        'LSTM':        evaluate_predictions(
            seg['actual_log'].values, seg['lstm_log'].values,
            label=f'  {name}/LSTM       '),
        'Primary':     evaluate_predictions(
            seg['actual_log'].values, seg['primary_log'].values,
            label=f'  {name}/Primary    '),
    }

In [ ]:
## Save comparison and subgroup results

section7_results = {
    'main_comparison': {k: v for k, v in comparison.items()},
    'coverage_stratified': coverage_results,
    'temporal_segments': segment_results_all,
    'static_only': {
        'features': STATIC_ONLY_COLS,
        'best_iteration': int(static_only['best_iteration']),
        'metrics': static_only['metrics'],
    },
    'static_plus_calendar': {
        'features': STATIC_PLUS_CAL_COLS,
        'best_iteration': int(static_plus_cal['best_iteration']),
        'metrics': static_plus_cal['metrics'],
    },
    'lstm': {
        'architecture': 'LSTM(1-layer, hidden=64, dropout=0.2) → Linear',
        'features': LSTM_FEATURES,
        'seq_length': K_SEQ,
        'n_params': int(n_params),
        'best_val_mse': float(best_val),
        'metrics': lstm_results,
    },
    'group_median_baseline': {
        'metrics': median_results,
    },
}

with open(CONFIG['results_dir'] / 'section7_baselines.json', 'w') as f:
    json.dump(section7_results, f, indent=2, default=str)

## Save LSTM model weights
torch.save(best_state, models_dir / 'lstm_baseline.pt')

## Save static baseline models
static_only['model'].save_model(str(models_dir / 'static_only_xgboost.json'))
static_plus_cal['model'].save_model(str(models_dir / 'static_plus_calendar_xgboost.json'))

print('ALL SECTION 7 RESULTS SAVED')
print(f'Comparison JSON: {CONFIG["results_dir"] / "section7_baselines.json"}')
print(f'LSTM model:      {models_dir / "lstm_baseline.pt"}')
print(f'Static models:   {models_dir / "static_only_xgboost.json"}')
print(f'                 {models_dir / "static_plus_calendar_xgboost.json"}')

The LSTM MAPE is 112.6%. This is way higher than its MAE/RMSE/R² would suggest. That is because MAPE blows up on cheap cards where a small absolute error is a huge relative error, and the LSTM appears to overshoot cheap cards disproportionately.

In [ ]:
## Sanity check: LSTM MAPE by price decile
test_master['lstm_pred_usd'] = np.clip(np.expm1(test_master['lstm_log']), 0, None)
test_master['lstm_abs_pct_err'] = np.abs(
    (test_master['price'] - test_master['lstm_pred_usd']) / test_master['price']
) * 100
test_master['price_decile'] = pd.qcut(test_master['price'], 10, labels=False, duplicates='drop')
print(test_master.groupby('price_decile').agg(
    n=('price', 'size'),
    median_price=('price', 'median'),
    mean_mape=('lstm_abs_pct_err', 'mean'),
).round(1))

## Section 8: Feature importance and interpretability for both models

**XGBoost primary:**
- Gain-based importance (native XGBoost)
- Permutation importance on val (robustness check against gain bias)
- Feature-class aggregation

**LSTM baseline:**
- Permutation importance on the 5 sequence features
- Sequence-position ablation (which timesteps matter)

**Specific flag:** `days_since_start` is a monotonic time index. Under drift
it may appear informative during training (it correlates with prices trending
upward over time) but the learned mapping fails at test time. Check this
explicitly.

In [ ]:
## XGBoost: gain-based importance on primary model
## Reload gain-based computation for completeness
gain_scores = model_primary.get_booster().get_score(importance_type='gain') \
    if hasattr(model_primary, 'get_booster') else model_primary.get_score(importance_type='gain')

## Normalise
total_gain = sum(gain_scores.values())
gain_norm = {k: v/total_gain for k, v in gain_scores.items()}

## Pad missing features (XGBoost omits unused features)
gain_full = {f: gain_norm.get(f, 0.0) for f in FEATURE_COLS}

## Top-20 plot
gain_sorted = sorted(gain_full.items(), key=lambda x: -x[1])
top20 = gain_sorted[:20]

fig, ax = plt.subplots(figsize=(9, 7))
feats = [t[0] for t in top20][::-1]
vals  = [t[1]*100 for t in top20][::-1]
bars = ax.barh(feats, vals, color='steelblue')
ax.set_xlabel('Gain-based importance (%)')
ax.set_title('XGBoost primary: top 20 features (gain)')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(CONFIG['results_dir']/'figures'/'xgb_feature_importance_gain.png',
            dpi=120, bbox_inches='tight')
plt.show()

print('TOP 20 FEATURES BY GAIN')
for rank, (f, v) in enumerate(gain_sorted[:20], 1):
    print(f'{rank:2d}. {f:<28}: {v*100:5.2f}%')

In [ ]:
## Permutation importance on val. Unbiased, asks "how much does val R² drop"

def permutation_importance_xgb(model, X_val, y_val, feature_names, n_repeats=5, seed=42):
    """Permutation importance: drop in val R²(log) when feature is shuffled."""
    rng = np.random.default_rng(seed)
    dval = xgb.DMatrix(X_val, label=y_val, feature_names=feature_names)
    best_iter = model.best_iteration if hasattr(model, 'best_iteration') else None
    iter_range = (0, best_iter+1) if best_iter is not None else None

    base_pred = model.predict(dval, iteration_range=iter_range) if iter_range else model.predict(dval)
    base_r2 = r2_score(y_val, base_pred)

    results = {}
    for i, fname in enumerate(feature_names):
        drops = []
        for _ in range(n_repeats):
            X_perm = X_val.copy()
            X_perm[:, i] = rng.permutation(X_perm[:, i])
            dperm = xgb.DMatrix(X_perm, label=y_val, feature_names=feature_names)
            perm_pred = model.predict(dperm, iteration_range=iter_range) if iter_range else model.predict(dperm)
            drops.append(base_r2 - r2_score(y_val, perm_pred))
        results[fname] = {'mean_drop': float(np.mean(drops)), 'std_drop': float(np.std(drops))}
    return base_r2, results

print('Compute permutation importance on val')
t0 = time.time()
base_r2, perm_scores = permutation_importance_xgb(
    model_primary, X['val'], y['val'], FEATURE_COLS, n_repeats=5, seed=CONFIG['seed']
)
print(f'Done in {time.time()-t0:.1f}s. Base val R²(log) = {base_r2:.4f}')

## Sort by mean drop
perm_sorted = sorted(perm_scores.items(), key=lambda x: -x[1]['mean_drop'])
print('\nTOP 20 FEATURES BY PERMUTATION IMPORTANCE (val R² drop)')
for rank, (f, d) in enumerate(perm_sorted[:20], 1):
    print(f'{rank:2d}. {f:<28}: Δ R² = {d["mean_drop"]:+.4f} ± {d["std_drop"]:.4f}')

## Compare gain vs permutation rankings and flag disagreements
print('\nRANK COMPARISON (top 10): gain vs permutation')
print(f'{"feature":<28} {"gain_rank":>10} {"perm_rank":>10} {"Δ":>6}')
gain_ranks = {f: i+1 for i, (f, _) in enumerate(gain_sorted)}
perm_ranks = {f: i+1 for i, (f, _) in enumerate(perm_sorted)}
for f, _ in gain_sorted[:10]:
    gr, pr = gain_ranks[f], perm_ranks[f]
    flag = ' <-- disagreement' if abs(gr - pr) >= 5 else ''
    print(f'{f:<28} {gr:>10d} {pr:>10d} {pr-gr:>+6d}{flag}')

In [ ]:
## Feature class aggregation with both gain and permutation
## Specific spurious-feature check for days_since_start

## Aggregate by class
with open(CONFIG['results_dir'] / 'feature_columns.json') as f:
    finfo = json.load(f)

classes = ['static', 'rolling', 'momentum', 'volume', 'volatility', 'calendar', 'count']

print(f"{'class':<12} {'n':>3} {'gain%':>8} {'perm_ΔR²':>10} {'gain/feat':>10} {'perm/feat':>10}")
class_summary = {}
for cls in classes:
    cols = finfo.get(cls, [])
    gain_total = sum(gain_full.get(c, 0) for c in cols) * 100
    perm_total = sum(perm_scores.get(c, {'mean_drop':0})['mean_drop'] for c in cols)
    gain_per = gain_total / len(cols) if cols else 0
    perm_per = perm_total / len(cols) if cols else 0
    class_summary[cls] = {
        'n_features': len(cols),
        'gain_total_pct': float(gain_total),
        'perm_total_drop': float(perm_total),
        'gain_per_feature': float(gain_per),
        'perm_per_feature': float(perm_per),
    }
    print(f'{cls:<12} {len(cols):>3} {gain_total:>7.1f}% {perm_total:>+10.4f} '
          f'{gain_per:>10.2f} {perm_per:>+10.4f}')

## Flag days_since_start specifically
dss_gain = gain_full.get('days_since_start', 0) * 100
dss_perm = perm_scores.get('days_since_start', {'mean_drop': 0})['mean_drop']
print(f'\nFLAG CHECK: days_since_start (drift risk):')
print(f'Gain: {dss_gain:.2f}%  (rank {gain_ranks.get("days_since_start","?")})')
print(f'Perm: {dss_perm:+.4f}  (rank {perm_ranks.get("days_since_start","?")})')
if dss_gain > 5 or perm_ranks.get("days_since_start", 99) <= 10:
    print('FLAG: days_since_start has substantial weight. Under a monotonic time')
    print('index, this feature encodes the training-period trend. It likely')
    print('contributes to the negative test R² because its learned mapping')
    print('does not extrapolate. Document as a structural failure mode.')
else:
    print('No flag: low weight on days_since_start.')

## Similar check on roll_mean_30d (deepest rolling window, most drift-exposed)
for feat in ['roll_mean_30d', 'roll_mean_7d', 'roll_median_30d']:
    if feat in gain_full:
        g = gain_full[feat] * 100
        p = perm_scores.get(feat, {'mean_drop':0})['mean_drop']
        print(f'{feat}: gain= {g:.2f}%, perm Δ= {p:+.4f}')

In [ ]:
## LSTM: Permutation importance on sequence features
## Shuffle one feature across all timesteps, re-predict val, measure R² drop

def permutation_importance_lstm(model, X_seq, mask, y_true, feature_names,
                                  device, n_repeats=5, seed=42):
    rng = np.random.default_rng(seed)
    model.eval()

    with torch.no_grad():
        base_pred = model(torch.from_numpy(X_seq).to(device),
                          torch.from_numpy(mask).to(device)).cpu().numpy()
    base_r2 = r2_score(y_true, base_pred)

    results = {}
    for i, fname in enumerate(feature_names):
        drops = []
        for _ in range(n_repeats):
            X_perm = X_seq.copy()
            ## Shuffle feature i across ALL sequences and timesteps simultaneously
            flat = X_perm[:, :, i].flatten()
            rng.shuffle(flat)
            X_perm[:, :, i] = flat.reshape(X_perm.shape[:2])

            with torch.no_grad():
                perm_pred = model(torch.from_numpy(X_perm).to(device),
                                  torch.from_numpy(mask).to(device)).cpu().numpy()
            drops.append(base_r2 - r2_score(y_true, perm_pred))
        results[fname] = {'mean_drop': float(np.mean(drops)),
                           'std_drop': float(np.std(drops))}
    return base_r2, results

print('Computing LSTM permutation importance on val...')
lstm_base_r2, lstm_perm = permutation_importance_lstm(
    model_lstm, X_seq_n[va_m], mask_seq[va_m], y_seq[va_m],
    LSTM_FEATURES, device, n_repeats=5, seed=CONFIG['seed']
)
print(f'LSTM base val R²(log) = {lstm_base_r2:.4f}')
print('\nLSTM PERMUTATION IMPORTANCE')
for f, d in sorted(lstm_perm.items(), key=lambda x: -x[1]['mean_drop']):
    print(f'{f:<25}: Δ R² = {d["mean_drop"]:+.4f} ± {d["std_drop"]:.4f}')

In [ ]:
## Which timestep does the LSTM weigh most heavily?
## Zero out position k for all sequences, re-predict, measure drop.

def sequence_position_ablation(model, X_seq, mask, y_true, device, seq_len):
    model.eval()
    with torch.no_grad():
        base_pred = model(torch.from_numpy(X_seq).to(device),
                          torch.from_numpy(mask).to(device)).cpu().numpy()
    base_r2 = r2_score(y_true, base_pred)

    drops = []
    for k in range(seq_len):
        X_ab = X_seq.copy()
        X_ab[:, k, :] = 0.0  ## zero out timestep k
        with torch.no_grad():
            ab_pred = model(torch.from_numpy(X_ab).to(device),
                            torch.from_numpy(mask).to(device)).cpu().numpy()
        drops.append(base_r2 - r2_score(y_true, ab_pred))
    return base_r2, drops

print('Computing LSTM sequence-position ablation on val...')
_, pos_drops = sequence_position_ablation(
    model_lstm, X_seq_n[va_m], mask_seq[va_m], y_seq[va_m],
    device, seq_len=K_SEQ
)

## Plot: position 0 is the oldest in the window, position K-1 is most recent
fig, ax = plt.subplots(figsize=(9, 4))
positions = list(range(K_SEQ))
ax.bar(positions, pos_drops, color='coral')
ax.set_xlabel('Sequence position (0 = oldest, K-1 = most recent)')
ax.set_ylabel('Δ val R²(log) when zeroed')
ax.set_title('LSTM sequence-position importance (val)')
ax.grid(alpha=0.3, axis='y')
ax.set_xticks(positions)
plt.tight_layout()
plt.savefig(CONFIG['results_dir']/'figures'/'lstm_position_importance.png',
            dpi=120, bbox_inches='tight')
plt.show()

print('\nLSTM SEQUENCE-POSITION DROPS')
for k, d in enumerate(pos_drops):
    label = 'oldest' if k == 0 else ('most recent' if k == K_SEQ-1 else '')
    print(f'pos {k} ({label}): Δ R² = {d:+.4f}')

## Interpretation
recent_frac = pos_drops[-1] / (sum(pos_drops) + 1e-8)
print(f'\nMost recent timestep accounts for {recent_frac*100:.1f}% of total position importance.')
if recent_frac > 0.5:
    print('LSTM depends primarily on the most recent transaction. Near-AR(1) behaviour.')
elif recent_frac > 0.3:
    print('LSTM weighs recent history but uses multiple timesteps.')
else:
    print('LSTM distributes attention across the sequence. Genuine long-memory use.')

In [ ]:
## Synthesis: economic intuition check
## Economic intuition: rolling prices should outrank calendar features

roll_price_gain = sum(gain_full.get(c, 0) for c in finfo['rolling']) * 100
calendar_gain = sum(gain_full.get(c, 0) for c in finfo['calendar']) * 100
roll_price_perm = sum(perm_scores.get(c, {'mean_drop':0})['mean_drop'] for c in finfo['rolling'])
calendar_perm = sum(perm_scores.get(c, {'mean_drop':0})['mean_drop'] for c in finfo['calendar'])

print('ECONOMIC INTUITION CHECK')
print(f'Rolling price features (gain): {roll_price_gain:.1f}%  vs calendar: {calendar_gain:.1f}%')
print(f'Rolling price features (perm): {roll_price_perm:+.4f}  vs calendar: {calendar_perm:+.4f}')
if roll_price_gain > calendar_gain and roll_price_perm > calendar_perm:
    print('PASS: rolling prices outrank calendar in both metrics.')
elif roll_price_gain > calendar_gain:
    print('MIXED: rolling wins on gain but not permutation (calendar may be')
    print('spuriously important via days_since_start drift exploitation on train).')
else:
    print('FAIL: calendar features outrank rolling prices. Primary has inverted')
    print('economic priority. A red flag consistent with the drift finding.')

## Save all Section 8 artefacts
section8_results = {
    'xgboost': {
        'gain_importance': gain_full,
        'permutation_importance': perm_scores,
        'base_val_r2': float(base_r2),
        'class_summary': class_summary,
        'top20_gain': [{'feature': f, 'gain_pct': v*100} for f, v in gain_sorted[:20]],
        'top20_perm': [{'feature': f, 'mean_drop': d['mean_drop']}
                        for f, d in perm_sorted[:20]],
    },
    'lstm': {
        'feature_permutation_importance': lstm_perm,
        'base_val_r2': float(lstm_base_r2),
        'position_importance': {f'pos_{k}': float(d) for k, d in enumerate(pos_drops)},
        'most_recent_fraction': float(recent_frac),
    },
    'economic_intuition_check': {
        'rolling_vs_calendar_gain': {
            'rolling_pct':  float(roll_price_gain),
            'calendar_pct': float(calendar_gain),
        },
        'rolling_vs_calendar_perm': {
            'rolling_drop':  float(roll_price_perm),
            'calendar_drop': float(calendar_perm),
        },
    },
}

with open(CONFIG['results_dir'] / 'section8_interpretability.json', 'w') as f:
    json.dump(section8_results, f, indent=2, default=str)

print(f'\nSaved: {CONFIG["results_dir"] / "section8_interpretability.json"}')

## Section 9: Feature ablation study

**Critical findings**

Section 8 revealed that several engineered features carry gain-based importance
but near-zero permutation importance on val, consistent with training-period
shortcut learning. This section tests whether removing these features improves
out-of-sample generalisation.

**Ablation variants:**

| Variant | Feature set | Question |
|---|---|---|
| V0 Primary | All 31 features (baseline from Section 6) | Reference |
| V1 No monotonic time | Drop `days_since_start` | Isolate the monotonic-index effect |
| V2 No drift-exposed | Drop days_since_start, rolling means, momentum, volatility | Are these collectively harmful? |
| V3 Drift-robust only | Keep static, rolling medians, volume, count, day_of_week, month (17 features) | Can we build a better primary from permutation-robust features? |

**Decision rule (stated in advance):**
The designated primary going forward is the variant with the highest test
R²(log), provided training behaviour is not pathological. If two variants
are within 0.02 of each other on test R², choose the simpler one.

All variants use identical XGBoost hyperparameters, identical splits,
identical early stopping. The only difference is the feature set.

In [ ]:
## Section 9: Feature ablation variants
## Identical training protocol across all variants. Only features differ

V0_COLS = FEATURE_COLS  ## all 31 (from Section 6)

V1_COLS = [c for c in FEATURE_COLS if c != 'days_since_start']

## V2: drop monotonic time + all rolling means + momentum + volatility
## Keep: static, rolling medians, rolling counts, volume, calendar (except days_since_start)
V2_DROP = (['days_since_start']
           + [c for c in FEATURE_COLS if c.startswith('roll_mean')]
           + [c for c in FEATURE_COLS if c.startswith('momentum')]
           + [c for c in FEATURE_COLS if c.startswith('roll_std')]
           + [c for c in FEATURE_COLS if c.startswith('roll_cv')])
V2_COLS = [c for c in FEATURE_COLS if c not in V2_DROP]

## V3: drift-robust only. Features with positive permutation importance in Section 8
## static (3), rolling medians (3), volume (6), count (3), day_of_week, month = 17
V3_COLS = (['grade_numeric', 'card_name_encoded', 'rarity_encoded']
           + ['roll_median_7d', 'roll_median_14d', 'roll_median_30d']
           + ['card_volume_7d', 'card_volume_14d', 'card_volume_30d',
              'market_volume_7d', 'market_volume_14d', 'market_volume_30d']
           + ['roll_count_7d', 'roll_count_14d', 'roll_count_30d']
           + ['day_of_week', 'month'])

## Sanity check: all variants reference valid features
for name, cols in [('V1', V1_COLS), ('V2', V2_COLS), ('V3', V3_COLS)]:
    missing = [c for c in cols if c not in FEATURE_COLS]
    assert len(missing) == 0, f'{name} references missing features: {missing}'

print('ABLATION VARIANT FEATURE COUNTS')
print(f'V0 Primary           : {len(V0_COLS)} features')
print(f'V1 No monotonic time : {len(V1_COLS)} features (dropped: days_since_start)')
print(f'V2 No drift-exposed  : {len(V2_COLS)} features (dropped {len(V2_DROP)} drift-exposed)')
print(f'V3 Drift-robust only : {len(V3_COLS)} features (permutation-positive only)')

print(f'\nV2 dropped features: {V2_DROP}')
print(f'\nV3 kept features: {V3_COLS}')

DECISION_RULE = {
    'criterion': 'max_test_r2_log',
    'tiebreak': 'simpler_feature_set',
    'tiebreak_threshold': 0.02,
    'invalidation': 'pathological_training_behaviour',
}
print(f'\nDECISION RULE: {DECISION_RULE}')

In [ ]:
## Train all four variants with the same hyperparameters and same early stopping

def train_variant(feature_cols, name):
    X_tr = df.loc[df['split']=='train', feature_cols].values
    X_va = df.loc[df['split']=='val',   feature_cols].values
    X_te = df.loc[df['split']=='test',  feature_cols].values
    y_tr = df.loc[df['split']=='train', 'log_price'].values
    y_va = df.loc[df['split']=='val',   'log_price'].values
    y_te = df.loc[df['split']=='test',  'log_price'].values

    dtr = xgb.DMatrix(X_tr, label=y_tr, feature_names=feature_cols)
    dva = xgb.DMatrix(X_va, label=y_va, feature_names=feature_cols)
    dte = xgb.DMatrix(X_te, label=y_te, feature_names=feature_cols)

    params = {
        'objective':        'reg:squarederror',
        'max_depth':        CONFIG['xgb_params']['max_depth'],
        'learning_rate':    CONFIG['xgb_params']['learning_rate'],
        'subsample':        CONFIG['xgb_params']['subsample'],
        'colsample_bytree': 0.8,
        'min_child_weight': 3,
        'reg_lambda':       1.0,
        'seed':             CONFIG['seed'],
        'verbosity':        0,
    }

    evals_result = {}
    model = xgb.train(
        params=params, dtrain=dtr,
        num_boost_round=CONFIG['xgb_params']['n_estimators'],
        evals=[(dtr, 'train'), (dva, 'val')],
        early_stopping_rounds=30,
        evals_result=evals_result,
        verbose_eval=0,
    )

    iter_range = (0, model.best_iteration + 1)
    pred_tr = model.predict(dtr, iteration_range=iter_range)
    pred_va = model.predict(dva, iteration_range=iter_range)
    pred_te = model.predict(dte, iteration_range=iter_range)

    metrics = {
        'train': evaluate_predictions(y_tr, pred_tr),
        'val':   evaluate_predictions(y_va, pred_va),
        'test':  evaluate_predictions(y_te, pred_te),
    }

    ## Training behaviour diagnostics
    train_rmse_final = evals_result['train']['rmse'][model.best_iteration]
    val_rmse_final   = evals_result['val']['rmse'][model.best_iteration]
    gap = val_rmse_final - train_rmse_final

    return {
        'name':            name,
        'features':        feature_cols,
        'n_features':      len(feature_cols),
        'model':           model,
        'best_iteration':  model.best_iteration,
        'pred_train':      pred_tr,
        'pred_val':        pred_va,
        'pred_test':       pred_te,
        'metrics':         metrics,
        'train_rmse_log':  float(train_rmse_final),
        'val_rmse_log':    float(val_rmse_final),
        'train_val_gap':   float(gap),
    }


print('Training all variants...')
variants = {}
for name, cols in [('V0', V0_COLS), ('V1', V1_COLS), ('V2', V2_COLS), ('V3', V3_COLS)]:
    t0 = time.time()
    variants[name] = train_variant(cols, name)
    v = variants[name]
    print(f'{name}: n_feat={v["n_features"]:>2d}  best_iter={v["best_iteration"]:>3d}  '
          f'train_rmse={v["train_rmse_log"]:.3f}  val_rmse={v["val_rmse_log"]:.3f}  '
          f'gap={v["train_val_gap"]:.3f}  ({time.time()-t0:.1f}s)')

In [ ]:
## Comparison table: all variants on the same test set
## Includes V0 results already in memory and Section 7 baselines for context

print('SECTION 9 ABLATION: TEST SET COMPARISON (n=1050)')
print(f'{"Variant":<6} {"n_feat":>6} {"best_iter":>9} {"gap_log":>8}  '
      f'{"MAE $":>10} {"RMSE $":>10} {"MAPE %":>7} {"R²(log)":>8}')
for name in ['V0', 'V1', 'V2', 'V3']:
    v = variants[name]
    m = v['metrics']['test']
    print(f'{name:<6} {v["n_features"]:>6d} {v["best_iteration"]:>9d} '
          f'{v["train_val_gap"]:>8.3f}  '
          f'${m["mae_usd"]:>9,.0f} ${m["rmse_usd"]:>9,.0f} '
          f'{m["mape_pct"]:>6.1f}% {m["r2_log"]:>+8.3f}')

## Reference lines from Section 7
print(f'{"Context from Section 7:"}')
print(f'Static-only          R²(log) = -0.256    (XGBoost, 3 features)')
print(f'Static + calendar    R²(log) = +0.108    (XGBoost, 6 features)')
print(f'LSTM                 R²(log) = +0.307    (sequence model)')
print(f'Sanity (train mean)  R²(log) = -0.870')


In [ ]:
## Coverage-stratified and temporal-segment analysis for all variants
## This is what reveals where each variant wins or loses

test_df_ablation = df[df['split']=='test'].copy().reset_index(drop=True)
test_df_ablation['V0_pred'] = variants['V0']['pred_test']
test_df_ablation['V1_pred'] = variants['V1']['pred_test']
test_df_ablation['V2_pred'] = variants['V2']['pred_test']
test_df_ablation['V3_pred'] = variants['V3']['pred_test']
test_df_ablation['has_rolling'] = test_df_ablation['roll_mean_7d'].notna()

print('\nCOVERAGE-STRATIFIED ABLATION')
coverage_ablation = {}
for has_roll, label in [(True, 'has_7d_rolling'), (False, 'cold_start')]:
    m = test_df_ablation['has_rolling'] == has_roll
    print(f'\n{label} (n={m.sum()})')
    coverage_ablation[label] = {}
    for variant in ['V0', 'V1', 'V2', 'V3']:
        r = evaluate_predictions(
            test_df_ablation.loc[m, 'log_price'].values,
            test_df_ablation.loc[m, f'{variant}_pred'].values,
            label=f'{variant}/{label}'
        )
        coverage_ablation[label][variant] = r

print('\nTEMPORAL SEGMENT ABLATION')
test_sorted = test_df_ablation.sort_values('date_sold').reset_index(drop=True)
n_seg = len(test_sorted) // 3
segment_ablation = {}
for i, seg_name in enumerate(['early', 'middle', 'late']):
    seg = test_sorted.iloc[i*n_seg : (i+1)*n_seg if i<2 else len(test_sorted)]
    print(f'\n{seg_name.upper()} ({seg["date_sold"].min().date()} to '
          f'{seg["date_sold"].max().date()}, n={len(seg)})')
    segment_ablation[seg_name] = {}
    for variant in ['V0', 'V1', 'V2', 'V3']:
        r = evaluate_predictions(
            seg['log_price'].values, seg[f'{variant}_pred'].values,
            label=f'{variant}/{seg_name}'
        )
        segment_ablation[seg_name][variant] = r

In [ ]:
## Apply the decision rule

print('DECISION: DESIGNATED PRIMARY GOING FORWARD')

## Summary table for the decision
summary = [(name,
            variants[name]['n_features'],
            variants[name]['metrics']['test']['r2_log'],
            variants[name]['metrics']['test']['mae_usd'],
            variants[name]['train_val_gap'])
           for name in ['V0', 'V1', 'V2', 'V3']]

print(f'{"Variant":<6} {"n_feat":>6} {"test_R²":>9} {"test_MAE":>10} {"train_val_gap":>14}')
for name, nf, r2, mae, gap in summary:
    print(f'{name:<6} {nf:>6d} {r2:>+9.3f} ${mae:>9,.0f} {gap:>14.3f}')

## Apply rule
by_r2 = sorted(summary, key=lambda x: -x[2])
best_r2 = by_r2[0][2]
within_threshold = [s for s in by_r2 if best_r2 - s[2] <= DECISION_RULE['tiebreak_threshold']]

if len(within_threshold) > 1:
    ## Tiebreak on simplicity
    chosen = sorted(within_threshold, key=lambda x: x[1])[0]
    print(f'\nTiebreak triggered (variants within {DECISION_RULE["tiebreak_threshold"]} '
          f'of best test R²): {[s[0] for s in within_threshold]}')
    print(f'Selecting simplest: {chosen[0]} with {chosen[1]} features')
else:
    chosen = by_r2[0]
    print(f'\nClear winner on test R²(log): {chosen[0]}')

DESIGNATED_PRIMARY = chosen[0]
print(f'\nDESIGNATED PRIMARY GOING FORWARD: {DESIGNATED_PRIMARY}')
print(f'n_features: {chosen[1]}')
print(f'test R²(log): {chosen[2]:+.3f}')
print(f'test MAE: ${chosen[3]:,.0f}')
print(f'train-val gap: {chosen[4]:.3f}')

## Pathology check
if abs(variants[DESIGNATED_PRIMARY]['train_val_gap']) > 1.0:
    print(f'\nWARNING: train-val gap ({chosen[4]:.3f}) is pathological. Manual review.')
else:
    print(f'No pathology detected.')

In [ ]:
## Save all ablation artefacts

ablation_results = {
    'variants': {
        name: {
            'features':       v['features'],
            'n_features':     v['n_features'],
            'best_iteration': v['best_iteration'],
            'train_rmse_log': v['train_rmse_log'],
            'val_rmse_log':   v['val_rmse_log'],
            'train_val_gap':  v['train_val_gap'],
            'metrics':        v['metrics'],
        }
        for name, v in variants.items()
    },
    'coverage_stratified': coverage_ablation,
    'temporal_segments':   segment_ablation,
    'decision_rule':       DECISION_RULE,
    'designated_primary':  DESIGNATED_PRIMARY,
}

with open(CONFIG['results_dir'] / 'section9_ablation.json', 'w') as f:
    json.dump(ablation_results, f, indent=2, default=str)

## Save all variant models
for name, v in variants.items():
    v['model'].save_model(str(models_dir / f'ablation_{name.lower()}_xgboost.json'))

## Save variant predictions aligned by listing_id for downstream use
ablation_preds = []
for name, v in variants.items():
    for split in ['train', 'val', 'test']:
        lids = df.loc[df['split']==split, 'listing_id'].values
        y_true = df.loc[df['split']==split, 'log_price'].values
        y_pred = v[f'pred_{split}']
        for lid, yt, yp in zip(lids, y_true, y_pred):
            ablation_preds.append({
                'variant':             name,
                'listing_id':          lid,
                'split':               split,
                'log_price_actual':    float(yt),
                'log_price_predicted': float(yp),
            })
pd.DataFrame(ablation_preds).to_parquet(
    CONFIG['results_dir'] / 'ablation_predictions.parquet', index=False
)

## Update the reference: which model is the designated primary for Section 10+
## This is the variable downstream sections will import
model_primary_final = variants[DESIGNATED_PRIMARY]['model']
FEATURE_COLS_FINAL = variants[DESIGNATED_PRIMARY]['features']
PRIMARY_PREDS_FINAL = {
    split: variants[DESIGNATED_PRIMARY][f'pred_{split}']
    for split in ['train', 'val', 'test']
}

print('\nALL SAVED')
print(f'Results JSON: {CONFIG["results_dir"] / "section9_ablation.json"}')
print(f'Models:       ablation_v0/v1/v2/v3_xgboost.json in {models_dir}')
print(f'Predictions:  ablation_predictions.parquet')
print(f'\nDesignated primary: {DESIGNATED_PRIMARY}')
print(f'Carried forward as: model_primary_final, FEATURE_COLS_FINAL, PRIMARY_PREDS_FINAL')

In [ ]:
## Diagnostic: Is days_since_start really the cold-start advantage source?
## Train V0 minus just days_since_start (this is V1) and compare cold-start only
cold_mask = (df['split'] == 'test') & (df['roll_mean_7d'].isna())
print(f'Cold-start test rows: {cold_mask.sum()}')

## Check date distribution of cold-start test rows
cold_dates = df.loc[cold_mask, 'date_sold']
print(f'Cold-start test date range: {cold_dates.min().date()} to {cold_dates.max().date()}')
print(f'Cold-start test year counts:')
print(cold_dates.dt.year.value_counts().sort_index())

## Are cold-start test rows concentrated in a specific period that days_since_start could tag?
## Compare V0 vs V1 MAE on cold-start, segmented by year-quarter
cold_df = df.loc[cold_mask, ['date_sold', 'log_price']].copy()
cold_df['V0_pred'] = variants['V0']['pred_test'][df.loc[df['split']=='test'].reset_index(drop=True)['roll_mean_7d'].isna().values]
cold_df['V1_pred'] = variants['V1']['pred_test'][df.loc[df['split']=='test'].reset_index(drop=True)['roll_mean_7d'].isna().values]
cold_df['V0_abs_err'] = np.abs(np.expm1(cold_df['log_price']) - np.clip(np.expm1(cold_df['V0_pred']), 0, None))
cold_df['V1_abs_err'] = np.abs(np.expm1(cold_df['log_price']) - np.clip(np.expm1(cold_df['V1_pred']), 0, None))
cold_df['yq'] = cold_df['date_sold'].dt.to_period('Q')
print(cold_df.groupby('yq').agg(n=('log_price','size'), v0_mae=('V0_abs_err','mean'), v1_mae=('V1_abs_err','mean')).round(0))

## Section 9 Verdict

**Ablation results summary:**
- V0 Primary: test R²(log) = −0.134
- V1 No monotonic time: test R²(log) = −0.296
- V2 No drift-exposed: test R²(log) = −0.234
- V3 Drift-robust only: test R²(log) = −0.312

No hybrid variant recovers positive test R². Dropping features does not address the drift problem, therefore the finding reinforces rather than weakens the Section 7 conclusion.

**Designated models for downstream sections:**
- V0 as hybrid representative for comparison purposes
- Static + calendar XGBoost and LSTM for embeddings extraction for fusion module

**Implications for market-encoder verdict:**

 After investigating the hybrid approach across four variants and finding it underperformed a simple static baseline under the drift conditions of this dataset, the static+calendar baseline model from Section 7 is therefore the designated XGBoost primary for the market module. It has test R² = +0.108. Actually positive. It is the only XGBoost-family model that beats predicting the test mean.

**Implications for Section 11 (market embedding):**
Static + calendar XGBoost's embedding will be extracted alongside the LSTM
penultimate-layer embedding for dual-embedding fusion in fusion-vs-unimodal.

## Section 10: Overfitting and stability checks

This section re-evaluates test for stability
characterisation, not for model selection.

**Models selected:**
- V0 (hybrid XGBoost, 31 features): market-encoder hybrid representative
- Static+calendar XGBoost (6 features): candidate for fusion extrinsic encoder
- LSTM: candidate for fusion extrinsic encoder

**Checks:**
- Train vs val gap for Static+calendar XGBoost and LSTM (already have for V0)
- Seed robustness across 3 seeds (tests whether reported metrics are stable)
- Temporal segment stability on test for Static+calendar XGBoost and LSTM (already have for V0)
- Spike-specific performance check (confirm v1 temporal clustering is resolved)


In [ ]:
## Train vs val gap for all three models
## V0 already computed (0.506). Recompute for completeness; add others.

print('TRAIN VS VAL GAP (log-space RMSE)')

## V0 (hybrid)
v0 = variants['V0']
print(f'V0 hybrid XGBoost:')
print(f'Train RMSE (log): {v0["train_rmse_log"]:.4f}')
print(f'Val   RMSE (log): {v0["val_rmse_log"]:.4f}')
print(f'Gap:              {v0["train_val_gap"]:.4f}')

## Static+calendar XGBoost (recompute from stored model)
sc = static_plus_cal  ## from Section 7
sc_train_rmse = np.sqrt(((np.expm1(y['train']) - np.expm1(sc['pred_train']))**2).mean())  ## guard
sc_m = sc['metrics']
## Use log-space RMSE directly from residuals:
sc_train_rmse_log = np.sqrt(((y['train'] - sc['pred_train'])**2).mean())
sc_val_rmse_log   = np.sqrt(((y['val']   - sc['pred_val'])**2).mean())
sc_gap = sc_val_rmse_log - sc_train_rmse_log
print(f'\nStatic+calendar XGBoost:')
print(f'Train RMSE (log): {sc_train_rmse_log:.4f}')
print(f'Val   RMSE (log): {sc_val_rmse_log:.4f}')
print(f'Gap:              {sc_gap:.4f}')

## LSTM
lstm_train_rmse_log = np.sqrt(((y_seq[tr_m] - lstm_pred_tr)**2).mean())
lstm_val_rmse_log   = np.sqrt(((y_seq[va_m] - lstm_pred_va)**2).mean())
lstm_gap = lstm_val_rmse_log - lstm_train_rmse_log
print(f'\nLSTM:')
print(f'Train RMSE (log): {lstm_train_rmse_log:.4f}')
print(f'Val   RMSE (log): {lstm_val_rmse_log:.4f}')
print(f'Gap:              {lstm_gap:.4f}')

**INTERPRETATION:**

- V0: large gap (0.5) consistent with diagnosed drift-sensitive features fitting training-period noise
- Static+calendar: smaller gap (fewer features to overfit)
- LSTM: gap reflects how much the sequence memorises train prices

In [ ]:
## Seed robustness across 3 seeds
## XGBoost: seed affects subsample + colsample_bytree row/column selection
## Tests whether reported test R² is stable under different stochastic draws

SEEDS = [42, 123, 7]

def train_xgb_with_seed(feature_cols, seed):
    X_tr = df.loc[df['split']=='train', feature_cols].values
    X_va = df.loc[df['split']=='val',   feature_cols].values
    X_te = df.loc[df['split']=='test',  feature_cols].values
    y_tr = df.loc[df['split']=='train', 'log_price'].values
    y_va = df.loc[df['split']=='val',   'log_price'].values
    y_te = df.loc[df['split']=='test',  'log_price'].values

    dtr = xgb.DMatrix(X_tr, label=y_tr, feature_names=feature_cols)
    dva = xgb.DMatrix(X_va, label=y_va, feature_names=feature_cols)
    dte = xgb.DMatrix(X_te, label=y_te, feature_names=feature_cols)

    params = {
        'objective':        'reg:squarederror',
        'max_depth':        CONFIG['xgb_params']['max_depth'],
        'learning_rate':    CONFIG['xgb_params']['learning_rate'],
        'subsample':        CONFIG['xgb_params']['subsample'],
        'colsample_bytree': 0.8,
        'min_child_weight': 3,
        'reg_lambda':       1.0,
        'seed':             seed,
        'verbosity':        0,
    }
    m = xgb.train(params=params, dtrain=dtr,
                  num_boost_round=CONFIG['xgb_params']['n_estimators'],
                  evals=[(dtr,'train'),(dva,'val')],
                  early_stopping_rounds=30, verbose_eval=0)
    iter_range = (0, m.best_iteration+1)
    pred_te = m.predict(dte, iteration_range=iter_range)
    return evaluate_predictions(y_te, pred_te)

print('SEED ROBUSTNESS (test metrics across 3 seeds)')

seed_results = {'V0_hybrid': [], 'Static+calendar': [], 'LSTM': []}

print('\nV0 hybrid XGBoost (31 features):')
for s in SEEDS:
    r = train_xgb_with_seed(FEATURE_COLS, s)
    seed_results['V0_hybrid'].append(r)
    print(f'seed={s}: MAE=${r["mae_usd"]:,.0f}  RMSE=${r["rmse_usd"]:,.0f}  '
          f'R²(log)={r["r2_log"]:+.3f}')

print('\nStatic+calendar XGBoost (6 features):')
for s in SEEDS:
    r = train_xgb_with_seed(STATIC_PLUS_CAL_COLS, s)
    seed_results['Static+calendar'].append(r)
    print(f'seed={s}: MAE=${r["mae_usd"]:,.0f}  RMSE=${r["rmse_usd"]:,.0f}  '
          f'R²(log)={r["r2_log"]:+.3f}')

In [ ]:
## LSTM seed robustness
## Re-train LSTM with 3 seeds (weight init, dropout, shuffling all affected)

def train_lstm_with_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

    Xt_tr_l = torch.from_numpy(X_seq_n[tr_m]).float()
    Mt_tr_l = torch.from_numpy(mask_seq[tr_m]).float()
    yt_tr_l = torch.from_numpy(y_seq[tr_m]).float()
    Xt_va_l = torch.from_numpy(X_seq_n[va_m]).float()
    Mt_va_l = torch.from_numpy(mask_seq[va_m]).float()
    Xt_te_l = torch.from_numpy(X_seq_n[te_m]).float()
    Mt_te_l = torch.from_numpy(mask_seq[te_m]).float()

    model = LSTMRegressor(N_LSTM_FEATURES, hidden=64, dropout=0.2).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    loss_fn = nn.MSELoss()

    loader = DataLoader(TensorDataset(Xt_tr_l, Mt_tr_l, yt_tr_l),
                        batch_size=64, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))

    best_val, patience_ctr, best_state = float('inf'), 0, None
    for epoch in range(1, 81):
        model.train()
        for xb, mb, yb in loader:
            xb, mb, yb = xb.to(device), mb.to(device), yb.to(device)
            pred = model(xb, mb)
            loss = loss_fn(pred, yb)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vp = model(Xt_va_l.to(device), Mt_va_l.to(device)).cpu().numpy()
        vl = float(np.mean((vp - y_seq[va_m])**2))
        if vl < best_val - 1e-4:
            best_val = vl
            best_state = {k: v.clone().cpu() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
        if patience_ctr >= 10:
            break

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    with torch.no_grad():
        tp = model(Xt_te_l.to(device), Mt_te_l.to(device)).cpu().numpy()
    return evaluate_predictions(y_seq[te_m], tp)


print('\nLSTM (sequence model):')
for s in SEEDS:
    r = train_lstm_with_seed(s)
    seed_results['LSTM'].append(r)
    print(f'seed={s}: MAE=${r["mae_usd"]:,.0f}  RMSE=${r["rmse_usd"]:,.0f}  '
          f'R²(log)={r["r2_log"]:+.3f}')

In [ ]:
## Summary: mean ± std across seeds

print('\n\nSUMMARY: SEED ROBUSTNESS (mean ± std across 3 seeds)')
print(f'{"Model":<20} {"MAE $ (mean±std)":>22} {"RMSE $ (mean±std)":>24} {"R²(log) (mean±std)":>24}')

seed_summary = {}
for name, runs in seed_results.items():
    mae_vals  = [r['mae_usd']  for r in runs]
    rmse_vals = [r['rmse_usd'] for r in runs]
    r2_vals   = [r['r2_log']   for r in runs]
    seed_summary[name] = {
        'mae_mean':  float(np.mean(mae_vals)),  'mae_std':  float(np.std(mae_vals)),
        'rmse_mean': float(np.mean(rmse_vals)), 'rmse_std': float(np.std(rmse_vals)),
        'r2_mean':   float(np.mean(r2_vals)),   'r2_std':   float(np.std(r2_vals)),
        'runs': runs,
    }
    print(f'{name:<20} '
          f'${np.mean(mae_vals):>7,.0f} ± ${np.std(mae_vals):>5,.0f}   '
          f'${np.mean(rmse_vals):>7,.0f} ± ${np.std(rmse_vals):>5,.0f}   '
          f'{np.mean(r2_vals):>+7.3f} ± {np.std(r2_vals):>.3f}')

## Verdict
print('\nVERDICT:')
for name, s in seed_summary.items():
    if s['r2_std'] < 0.02:
        print(f'{name}: STABLE (R² std < 0.02)')
    elif s['r2_std'] < 0.05:
        print(f'{name}: ACCEPTABLE (R² std < 0.05). Rankings robust')
    else:
        print(f'{name}: UNSTABLE (R² std = {s["r2_std"]:.3f}). Rankings may not replicate')

## Are the LSTM > static+calendar > V0 rankings stable?
## Compute pairwise rank stability across seeds
r2_by_seed = {
    s: {name: seed_results[name][i]['r2_log'] for name in seed_results}
    for i, s in enumerate(SEEDS)
}
rankings_per_seed = []
for s in SEEDS:
    ranked = sorted(r2_by_seed[s].items(), key=lambda x: -x[1])
    rankings_per_seed.append([name for name, _ in ranked])
print(f'\nTest R² rankings per seed:')
for s, r in zip(SEEDS, rankings_per_seed):
    print(f'seed {s}: {" > ".join(r)}')

In [ ]:
## Temporal segment stability on test
## Already computed for V0 in Section 9. Extend to static+calendar and LSTM.

## Reuse sorted test frame. Add static+calendar and LSTM predictions
test_sorted_v10 = df[df['split']=='test'].copy().reset_index(drop=True)
test_sorted_v10['V0_pred']  = variants['V0']['pred_test']
test_sorted_v10['SC_pred']  = static_plus_cal['pred_test']
## LSTM predictions aligned by listing_id
lstm_map = dict(zip(lid_seq[te_m], lstm_pred_te))
test_sorted_v10['LSTM_pred'] = test_sorted_v10['listing_id'].map(lstm_map)
test_sorted_v10 = test_sorted_v10.sort_values('date_sold').reset_index(drop=True)

n_seg = len(test_sorted_v10) // 3
print('TEMPORAL SEGMENT STABILITY (test)')
i3_results = {}
for i, seg_name in enumerate(['early', 'middle', 'late']):
    seg = test_sorted_v10.iloc[i*n_seg : (i+1)*n_seg if i<2 else len(test_sorted_v10)]
    print(f'\n{seg_name.upper()} ({seg["date_sold"].min().date()} to '
          f'{seg["date_sold"].max().date()}, n={len(seg)})')
    i3_results[seg_name] = {}
    for col, label in [('V0_pred','V0 hybrid'), ('SC_pred','Static+cal'), ('LSTM_pred','LSTM')]:
        r = evaluate_predictions(
            seg['log_price'].values, seg[col].values,
            label=f'{seg_name}/{label:<12}'
        )
        i3_results[seg_name][label] = r

## Segment-level winner per segment
print('\nSEGMENT WINNERS (by R²(log)):')
for seg_name in ['early', 'middle', 'late']:
    ranked = sorted(i3_results[seg_name].items(), key=lambda x: -x[1]['r2_log'])
    winner, m = ranked[0]
    print(f'{seg_name:<7}: {winner} (R²={m["r2_log"]:+.3f})')

In [ ]:
## Confirm no single date dominates test performance
## v1 had 76% of data in top 10 dates. Verify.

## Test-set date concentration
date_counts = df[df['split']=='test']['date_sold'].dt.date.value_counts()
top10_pct = date_counts.head(10).sum() / len(df[df['split']=='test']) * 100
print('SPIKE-SPECIFIC CHECK')
print(f'Top 10 test dates: {date_counts.head(10).sum()} rows / {len(df[df["split"]=="test"])} '
      f'= {top10_pct:.1f}%')
print(f'Top 10 dates by count:')
for d, n in date_counts.head(10).items():
    print(f'{d}: {n} transactions')

## Per-date test performance. Are some dates disproportionately easy/hard?
test_sorted_v10['date_only'] = test_sorted_v10['date_sold'].dt.date
date_perf = (test_sorted_v10.groupby('date_only')
             .apply(lambda g: pd.Series({
                 'n': len(g),
                 'V0_mae_log':   float(np.abs(g['log_price']-g['V0_pred']).mean()),
                 'SC_mae_log':   float(np.abs(g['log_price']-g['SC_pred']).mean()),
                 'LSTM_mae_log': float(np.abs(g['log_price']-g['LSTM_pred']).mean()),
             }))
             .reset_index())

## Are top-10-date MAEs systematically different from the rest?
top10_dates = set(date_counts.head(10).index)
date_perf['is_top10'] = date_perf['date_only'].apply(lambda d: d in top10_dates)
print(f'\nMean log-MAE: top-10 dates vs rest')
for col in ['V0_mae_log', 'SC_mae_log', 'LSTM_mae_log']:
    top_mae = date_perf.loc[date_perf['is_top10'], col].mean()
    rest_mae = date_perf.loc[~date_perf['is_top10'], col].mean()
    print(f'{col}: top10={top_mae:.3f}, rest={rest_mae:.3f}, '
          f'diff={top_mae-rest_mae:+.3f}')

if top10_pct > 15:
    print(f'\nFLAG: top 10 dates account for {top10_pct:.1f}% of test. High concentration')
else:
    print(f'\nOK: top 10 dates are {top10_pct:.1f}% of test. v1 clustering resolved')

In [ ]:
## Consolidate and save

section10_results = {
    'I1_train_val_gaps': {
        'V0_hybrid': {
            'train_rmse_log': float(v0['train_rmse_log']),
            'val_rmse_log':   float(v0['val_rmse_log']),
            'gap':            float(v0['train_val_gap']),
        },
        'Static+calendar': {
            'train_rmse_log': float(sc_train_rmse_log),
            'val_rmse_log':   float(sc_val_rmse_log),
            'gap':            float(sc_gap),
        },
        'LSTM': {
            'train_rmse_log': float(lstm_train_rmse_log),
            'val_rmse_log':   float(lstm_val_rmse_log),
            'gap':            float(lstm_gap),
        },
    },
    'I2_seed_robustness': {
        'seeds': SEEDS,
        'per_seed_results': {name: [{k: float(v) if isinstance(v,(int,float)) else v
                                       for k,v in r.items()} for r in runs]
                             for name, runs in seed_results.items()},
        'summary': seed_summary,
        'rankings_per_seed': {str(s): r for s, r in zip(SEEDS, rankings_per_seed)},
    },
    'I3_temporal_stability': {
        seg: {model: {k: float(v) if isinstance(v,(int,float)) else v
                      for k,v in r.items()}
              for model, r in models.items()}
        for seg, models in i3_results.items()
    },
    'I4_spike_check': {
        'top10_pct': float(top10_pct),
        'v1_top10_pct_reference': 76.0,
        'v2_resolved': top10_pct < 15,
    },
}

with open(CONFIG['results_dir'] / 'section10_stability.json', 'w') as f:
    json.dump(section10_results, f, indent=2, default=str)

print(f'\nALL SAVED')
print(f'  Results JSON: {CONFIG["results_dir"] / "section10_stability.json"}')

## Section 11: Market-state embedding construction (Dual)

Extract two market-state representations for fusion:
1. LSTM embedding (64-dim hidden state at final real timestep)
2. Static+calendar XGBoost embedding (leaf-index one-hot → PCA 64)

Both aligned by listing_id. V0 hybrid is not extracted (negative test R²
would poison fusion. See Section 9 and Section 10 findings).

**Design note:** The XGBoost leaf-index + PCA representation is an engineering
choice, not a learned representation. This is documented explicitly as a
limitation.

In [ ]:
## (LSTM): Extract 64-dim hidden state at the final REAL timestep per row
## The existing LSTMRegressor returns the prediction to get a modified forward that returns the pre-head hidden state.

## Reload the best LSTM (seed 42, saved in Section 7 as lstm_baseline.pt)
model_lstm.load_state_dict({k: v.to(device) for k, v in best_state.items()})
model_lstm.eval()

def extract_lstm_hidden(model, X, mask, device, batch_size=128):
    """
    Run sequences through the LSTM, return the hidden state at the final
    REAL timestep (using the mask, not position K-1, to handle padding).
    Returns: (N, hidden_dim) numpy array
    """
    model.eval()
    N = X.shape[0]
    hidden_dim = model.lstm.hidden_size
    out = np.zeros((N, hidden_dim), dtype=np.float32)

    with torch.no_grad():
        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)
            xb = torch.from_numpy(X[start:end]).float().to(device)
            mb = torch.from_numpy(mask[start:end]).float().to(device)

            lengths = mb.sum(dim=1).clamp(min=1).long()
            lstm_out, _ = model.lstm(xb)  ## (B, K, H)

            ## Gather hidden at last REAL timestep
            idx = (lengths - 1).unsqueeze(1).unsqueeze(2).expand(-1, 1, lstm_out.size(-1))
            last = lstm_out.gather(1, idx).squeeze(1)  ## (B, H)

            out[start:end] = last.cpu().numpy()
    return out

## Extract for all sequences (train+val+test combined, as produced in Section 7)
print('Extract LSTM hidden states')
t0 = time.time()
lstm_embeds_all = extract_lstm_hidden(model_lstm, X_seq_n, mask_seq, device)
print(f'Done in {time.time()-t0:.1f}s. Shape: {lstm_embeds_all.shape}')

## Sanity: dimensions and no NaN
assert lstm_embeds_all.shape == (len(lid_seq), 64), f'Shape mismatch: {lstm_embeds_all.shape}'
assert not np.isnan(lstm_embeds_all).any(), 'NaN in LSTM embeddings'
n_zero_vecs = (np.linalg.norm(lstm_embeds_all, axis=1) < 1e-8).sum()
print(f'LSTM embeddings. NaN: 0, zero vectors: {n_zero_vecs}')

## Normalise per-dim using TRAIN split statistics only (no leakage)
tr_mask_seq = (split_seq == 'train')
lstm_mu = lstm_embeds_all[tr_mask_seq].mean(axis=0)
lstm_sd = lstm_embeds_all[tr_mask_seq].std(axis=0) + 1e-8
lstm_embeds_n = (lstm_embeds_all - lstm_mu) / lstm_sd
print(f'Normalisation (mu range): [{lstm_mu.min():.3f}, {lstm_mu.max():.3f}], '
      f'sd range: [{lstm_sd.min():.3f}, {lstm_sd.max():.3f}]')

In [ ]:
## (XGBoost): Extract leaf indices from the static+calendar model
## Each row → vector of tree count ints (one leaf index per tree)

sc_model = static_plus_cal['model']  ## from Section 7
sc_features = STATIC_PLUS_CAL_COLS
sc_best_iter = static_plus_cal['best_iteration']

print(f'Static+calendar model: {sc_best_iter} trees, {len(sc_features)} features')

## Build DMatrices for all three splits
X_tr_sc = df.loc[df['split']=='train', sc_features].values
X_va_sc = df.loc[df['split']=='val',   sc_features].values
X_te_sc = df.loc[df['split']=='test',  sc_features].values

dtr_sc = xgb.DMatrix(X_tr_sc, feature_names=sc_features)
dva_sc = xgb.DMatrix(X_va_sc, feature_names=sc_features)
dte_sc = xgb.DMatrix(X_te_sc, feature_names=sc_features)

## Extract leaf indices (one integer per tree per row)
iter_range = (0, sc_best_iter + 1)
leaves_tr = sc_model.predict(dtr_sc, pred_leaf=True, iteration_range=iter_range)
leaves_va = sc_model.predict(dva_sc, pred_leaf=True, iteration_range=iter_range)
leaves_te = sc_model.predict(dte_sc, pred_leaf=True, iteration_range=iter_range)

print(f'Leaf index shapes: train {leaves_tr.shape}, val {leaves_va.shape}, test {leaves_te.shape}')
print(f'Leaf index range: [{leaves_tr.min()}, {leaves_tr.max()}]')

In [ ]:
## One-hot encode leaf indices across all trees, then PCA to 64 dims
## Fit PCA on TRAIN only (no leakage)

## One-hot: each (tree, leaf) combination is a unique column
ohe = OneHotEncoder(sparse_output=True, handle_unknown='ignore', dtype=np.float32)
ohe.fit(leaves_tr)

ohe_tr = ohe.transform(leaves_tr)
ohe_va = ohe.transform(leaves_va)
ohe_te = ohe.transform(leaves_te)
print(f'One-hot shapes: train {ohe_tr.shape}, val {ohe_va.shape}, test {ohe_te.shape}')
print(f'One-hot density (train): {ohe_tr.nnz / (ohe_tr.shape[0]*ohe_tr.shape[1]) * 100:.2f}%')

## PCA to 64: fit on TRAIN only
N_EMBED_DIM = 64
pca = PCA(n_components=N_EMBED_DIM, random_state=CONFIG['seed'])
sc_embeds_tr = pca.fit_transform(ohe_tr.toarray())
sc_embeds_va = pca.transform(ohe_va.toarray())
sc_embeds_te = pca.transform(ohe_te.toarray())
print(f'PCA embedding shapes: train {sc_embeds_tr.shape}, val {sc_embeds_va.shape}, test {sc_embeds_te.shape}')
print(f'PCA variance explained (first 64): {pca.explained_variance_ratio_.sum()*100:.1f}%')
print(f'First 10 component variance: {(pca.explained_variance_ratio_[:10]*100).round(2)}')

## Combine for a single aligned output
sc_embeds_all = np.zeros((len(df), N_EMBED_DIM), dtype=np.float32)
sc_embeds_all[df['split']=='train'] = sc_embeds_tr
sc_embeds_all[df['split']=='val']   = sc_embeds_va
sc_embeds_all[df['split']=='test']  = sc_embeds_te

## Sanity
assert not np.isnan(sc_embeds_all).any(), 'NaN in XGBoost embeddings'
n_zero_sc = (np.linalg.norm(sc_embeds_all, axis=1) < 1e-8).sum()
print(f'XGBoost embeddings. NaN: 0, zero vectors: {n_zero_sc}')

In [ ]:
## Build aligned embedding dataframes with listing_id, grade, price, split
## This is the contract the fusion module will consume

## LSTM embeddings: sequence-ordered, need to re-sort by listing_id
lstm_embed_df = pd.DataFrame(
    lstm_embeds_n,
    columns=[f'mkt_lstm_{i:03d}' for i in range(64)]
)
lstm_embed_df['listing_id'] = lid_seq
lstm_embed_df['split']      = split_seq
lstm_embed_df['log_price']  = y_seq
## Merge in card_name, grade, price from df for traceability
lstm_embed_df = lstm_embed_df.merge(
    df[['listing_id', 'card_name', 'grade', 'price']],
    on='listing_id', how='left'
)
## Reorder columns
meta_cols = ['listing_id', 'split', 'card_name', 'grade', 'price', 'log_price']
embed_cols = [f'mkt_lstm_{i:03d}' for i in range(64)]
lstm_embed_df = lstm_embed_df[meta_cols + embed_cols]

## XGBoost embeddings: df-ordered (same row ordering as df)
sc_embed_df = pd.DataFrame(
    sc_embeds_all,
    columns=[f'mkt_sc_{i:03d}' for i in range(64)]
)
sc_embed_df['listing_id'] = df['listing_id'].values
sc_embed_df['split']      = df['split'].values
sc_embed_df['card_name']  = df['card_name'].values
sc_embed_df['grade']      = df['grade'].values
sc_embed_df['price']      = df['price'].values
sc_embed_df['log_price']  = df['log_price'].values
embed_cols_sc = [f'mkt_sc_{i:03d}' for i in range(64)]
sc_embed_df = sc_embed_df[meta_cols + embed_cols_sc]

print('Embedding dataframe shapes:')
print(f'LSTM:   {lstm_embed_df.shape}')
print(f'XGBoost: {sc_embed_df.shape}')

## Alignment check: same listing_ids in both
lstm_ids = set(lstm_embed_df['listing_id'])
sc_ids = set(sc_embed_df['listing_id'])
full_ids = set(df['listing_id'])
print(f'\nlisting_id alignment:')
print(f'LSTM:   {len(lstm_ids)} unique')
print(f'XGBoost: {len(sc_ids)} unique')
print(f'df:     {len(full_ids)} unique')
assert lstm_ids == sc_ids == full_ids, 'listing_id set mismatch'
print(f'All three sets identical: PASS')

## Alignment with vision embeddings (sanity)
try:
    vision_emb = pd.read_parquet(CONFIG['embeddings_dir'] / 'visual_embeddings_v2.parquet') \
                 if 'embeddings_dir' in CONFIG else \
                 pd.read_parquet(PROJECT_ROOT / 'data/embeddings/visual_embeddings_v2.parquet')
    vision_ids = set(vision_emb['listing_id'])
    overlap = full_ids & vision_ids
    print(f'\nVision embeddings: {len(vision_ids)} unique')
    print(f'Overlap with market: {len(overlap)} ({len(overlap)/len(full_ids)*100:.1f}%)')
except Exception as e:
    print(f'(vision embedding file not loaded: {e})')

In [ ]:
## Save both embedding files with complete metadata

embeddings_dir = Path(PROJECT_ROOT / 'data/embeddings')
embeddings_dir.mkdir(parents=True, exist_ok=True)

## LSTM embeddings
lstm_embed_path = embeddings_dir / 'market_embeddings_lstm.parquet'
lstm_embed_df.to_parquet(lstm_embed_path, index=False)

## XGBoost embeddings
sc_embed_path = embeddings_dir / 'market_embeddings_static_cal.parquet'
sc_embed_df.to_parquet(sc_embed_path, index=False)

## Save PCA fit and LSTM normalisation stats for reproducibility
artefacts = {
    'lstm_norm_mu': lstm_mu,
    'lstm_norm_sd': lstm_sd,
    'pca_components':       pca.components_,
    'pca_mean':             pca.mean_,
    'pca_explained_var':    pca.explained_variance_ratio_,
    'ohe_categories':       ohe.categories_,
    'sc_best_iteration':    sc_best_iter,
    'sc_features':          sc_features,
    'lstm_features':        LSTM_FEATURES,
    'lstm_seq_len':         K_SEQ,
}
with open(CONFIG['results_dir'] / 'market_embedding_artefacts.pkl', 'wb') as f:
    pickle.dump(artefacts, f)

## JSON summary
embed_summary = {
    'lstm_embedding': {
        'path': str(lstm_embed_path),
        'shape': list(lstm_embed_df.shape),
        'n_dims': 64,
        'source_model': 'LSTM (seed 42)',
        'source_test_r2_log_meanstd': {'mean': 0.273, 'std': 0.020},
        'extraction': 'hidden state at final real timestep, mask-aware',
        'normalisation': 'z-score per dim using train-split statistics',
        'n_zero_vectors': int(n_zero_vecs),
    },
    'xgboost_embedding': {
        'path': str(sc_embed_path),
        'shape': list(sc_embed_df.shape),
        'n_dims': 64,
        'source_model': 'Static+calendar XGBoost',
        'source_test_r2_log_meanstd': {'mean': 0.144, 'std': 0.025},
        'extraction': f'leaf-index one-hot ({sc_best_iter} trees) → PCA 64',
        'pca_variance_explained_pct': float(pca.explained_variance_ratio_.sum() * 100),
        'pca_fit_on': 'train split only',
        'n_zero_vectors': int(n_zero_sc),
        'is_learned_representation': False,
        'note': 'Engineering choice, not a learned representation. Tree structure '
                'captures interaction patterns; PCA reduces to fixed dimensionality. '
                'Limitation documented in Section 14.',
    },
    'alignment': {
        'listing_id_sets_identical': True,
        'total_rows': len(df),
    },
}

with open(CONFIG['results_dir'] / 'section11_embeddings.json', 'w') as f:
    json.dump(embed_summary, f, indent=2, default=str)

print('ALL SAVED')
print(f'LSTM embeddings:     {lstm_embed_path}')
print(f'XGBoost embeddings:  {sc_embed_path}')
print(f'Artefacts (pickle):  {CONFIG["results_dir"] / "market_embedding_artefacts.pkl"}')
print(f'Summary JSON:        {CONFIG["results_dir"] / "section11_embeddings.json"}')

## Section 12: Embedding quality assessment

Tests whether the market embeddings encode information useful for valuation.

**Scope:**
- LSTM: full (t-SNE), (internal PCA variance), (price correlation),
   (within-card stability), (vision comparison)
- XGBoost: reduced (t-SNE), (price correlation), (vision comparison)
- Internal PCA variance and within-card stability are algorithmic properties
  of PCA of one-hot and not meaningfully interpretable as learned representation quality

In [ ]:
## Reload embeddings and vision embeddings for Section 12
## (already in memory from Section 11, this is defensive)

lstm_df = lstm_embed_df.copy()  ## from Section 11
xgb_df  = sc_embed_df.copy()    ## from Section 11

lstm_cols = [c for c in lstm_df.columns if c.startswith('mkt_lstm_')]
xgb_cols  = [c for c in xgb_df.columns  if c.startswith('mkt_sc_')]
print(f'LSTM embedding: {len(lstm_cols)} dims, {len(lstm_df)} rows')
print(f'XGB embedding:  {len(xgb_cols)} dims, {len(xgb_df)} rows')

## Vision embedding
vision_emb = pd.read_parquet(
    str(PROJECT_ROOT / 'data/embeddings/visual_embeddings_v2.parquet')
)
vision_cols = [c for c in vision_emb.columns if c.startswith('cond_emb_')]
print(f'Vision embedding: {len(vision_cols)} dims, {len(vision_emb)} rows')

## Align all three on listing_id for comparisons
common_ids = (set(lstm_df['listing_id']) & set(xgb_df['listing_id'])
              & set(vision_emb['listing_id']))
print(f'Common listing_ids across LSTM, XGB, Vision: {len(common_ids)}')
assert len(common_ids) == 3812, f'Expected 3812, got {len(common_ids)}'

In [ ]:
## (LSTM): t-SNE projection of 64-dim LSTM embedding
X_lstm = lstm_df[lstm_cols].values

tsne = TSNE(n_components=2, perplexity=30, random_state=CONFIG['seed'],
             init='pca', learning_rate='auto')
t0 = time.time()
lstm_2d = tsne.fit_transform(X_lstm)
print(f'Done in {time.time()-t0:.1f}s')

lstm_df['tsne_x'] = lstm_2d[:, 0]
lstm_df['tsne_y'] = lstm_2d[:, 1]
lstm_df['price_quintile'] = pd.qcut(lstm_df['price'], 5, labels=False, duplicates='drop')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

## Coloured by grade
for g, color in zip([8, 9, 10], ['tab:blue', 'tab:orange', 'tab:green']):
    m = lstm_df['grade'] == g
    axes[0].scatter(lstm_df.loc[m, 'tsne_x'], lstm_df.loc[m, 'tsne_y'],
                     c=color, label=f'PSA {g}', alpha=0.4, s=12)
axes[0].set_title('LSTM embedding: t-SNE coloured by grade')
axes[0].legend()
axes[0].set_xlabel('t-SNE 1'); axes[0].set_ylabel('t-SNE 2')

## Coloured by price quintile
sc = axes[1].scatter(lstm_df['tsne_x'], lstm_df['tsne_y'],
                      c=lstm_df['price_quintile'], cmap='viridis', alpha=0.5, s=12)
axes[1].set_title('LSTM embedding: t-SNE coloured by price quintile')
axes[1].set_xlabel('t-SNE 1'); axes[1].set_ylabel('t-SNE 2')
plt.colorbar(sc, ax=axes[1], label='Price quintile (0=cheapest, 4=priciest)')

plt.tight_layout()
plt.savefig(CONFIG['results_dir']/'figures'/'lstm_embedding_tsne.png',
             dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
## (LSTM): How much of the LSTM embedding space is actually used?

pca_lstm = PCA(n_components=64, random_state=CONFIG['seed'])
pca_lstm.fit(X_lstm)

## Cumulative variance
cumvar = np.cumsum(pca_lstm.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, 65), cumvar*100, 'o-', markersize=3)
ax.axhline(90, color='red', linestyle='--', alpha=0.5, label='90%')
ax.axhline(95, color='orange', linestyle='--', alpha=0.5, label='95%')
ax.set_xlabel('PCA component')
ax.set_ylabel('Cumulative variance explained (%)')
ax.set_title('LSTM embedding: Internal PCA cumulative variance')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CONFIG['results_dir']/'figures'/'lstm_embedding_pca.png',
             dpi=120, bbox_inches='tight')
plt.show()

## Effective dimensionality
n_90 = int(np.searchsorted(cumvar, 0.90) + 1)
n_95 = int(np.searchsorted(cumvar, 0.95) + 1)
print(f'LSTM embedding effective dimensionality:')
print(f'90% variance in first {n_90} components')
print(f'95% variance in first {n_95} components')
print(f'First 5 components: {(pca_lstm.explained_variance_ratio_[:5]*100).round(2)}')

In [ ]:
## (LSTM): Spearman correlation of each dimension with log_price

lstm_price_corr = []
for col in lstm_cols:
    rho, _ = spearmanr(lstm_df[col], lstm_df['log_price'])
    lstm_price_corr.append(abs(rho))
lstm_price_corr = np.array(lstm_price_corr)

## Top-correlated dimensions
top_idx = np.argsort(-lstm_price_corr)[:10]
print('LSTM: top 10 dimensions by |Spearman(dim, log_price)|')
for rank, i in enumerate(top_idx, 1):
    rho_signed, _ = spearmanr(lstm_df[lstm_cols[i]], lstm_df['log_price'])
    print(f'{rank:2d}. {lstm_cols[i]}: |ρ|={lstm_price_corr[i]:.3f}  (signed: {rho_signed:+.3f})')

print(f'\nLSTM per-dim |Spearman| summary:')
print(f'mean: {lstm_price_corr.mean():.3f}')
print(f'max:  {lstm_price_corr.max():.3f}')
print(f'dims with |ρ| > 0.1: {(lstm_price_corr > 0.1).sum()}/64')
print(f'dims with |ρ| > 0.3: {(lstm_price_corr > 0.3).sum()}/64')

In [ ]:
## (LSTM): Do transactions of the same card+grade cluster in embedding space?
## Measure: within-group mean pairwise cosine distance vs between-group

X_lstm_n = normalize(X_lstm, axis=1)  ## unit-normalise for cosine geometry

## Group identifier
lstm_df['group'] = lstm_df['card_name'].astype(str) + '_PSA' + lstm_df['grade'].astype(str)
groups = lstm_df['group'].values

## Within-group mean cosine distance per group
within_dist = []
group_sizes = []
for g in lstm_df['group'].unique():
    idx = np.where(groups == g)[0]
    if len(idx) < 2:
        continue
    sub = X_lstm_n[idx]
    d = cosine_distances(sub)
    iu = np.triu_indices(len(idx), k=1)
    within_dist.append(d[iu].mean())
    group_sizes.append(len(idx))

## Between-group: sample 1000 pairs from different groups
rng = np.random.default_rng(CONFIG['seed'])
n_samples = 1000
all_idx = np.arange(len(lstm_df))
between_pairs = []
while len(between_pairs) < n_samples:
    i, j = rng.choice(all_idx, 2, replace=False)
    if groups[i] != groups[j]:
        between_pairs.append(cosine_distances(X_lstm_n[i:i+1], X_lstm_n[j:j+1])[0, 0])
between_dist = np.array(between_pairs)

within_dist = np.array(within_dist)
print('(LSTM): Within-card clustering')
print(f'Within-group mean cosine distance:  {within_dist.mean():.4f} (n_groups={len(within_dist)})')
print(f'Between-group mean cosine distance: {between_dist.mean():.4f}')
print(f'Ratio (within/between):             {within_dist.mean()/between_dist.mean():.3f}')
print(f'(Ratio < 1 ⇒ same-card rows cluster together. Smaller = stronger.)')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(within_dist, bins=25, alpha=0.6, label='Within-group (per group)', color='tab:blue')
ax.axvline(between_dist.mean(), color='red', linestyle='--',
            label=f'Between-group mean ({between_dist.mean():.3f})')
ax.axvline(within_dist.mean(), color='blue', linestyle='--',
            label=f'Within-group mean ({within_dist.mean():.3f})')
ax.set_xlabel('Mean cosine distance')
ax.set_ylabel('Number of groups')
ax.set_title('LSTM embedding: Within-card vs between-card cosine distance')
ax.legend()
plt.tight_layout()
plt.savefig(CONFIG['results_dir']/'figures'/'lstm_within_card_stability.png',
             dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
## Compare each embedding's price correlation strength

## Align vision embedding to market by listing_id
vision_aligned = vision_emb.merge(
    lstm_df[['listing_id', 'log_price']], on='listing_id', how='inner'
)
vision_cols_list = [c for c in vision_aligned.columns if c.startswith('cond_emb_')]

vision_price_corr = []
for col in vision_cols_list:
    rho, _ = spearmanr(vision_aligned[col], vision_aligned['log_price'])
    vision_price_corr.append(abs(rho))
vision_price_corr = np.array(vision_price_corr)

print('Per-dim |Spearman(dim, log_price)| across embeddings')
print(f'{"Embedding":<20} {"n_dims":>6} {"mean":>7} {"max":>7} {">0.1":>6} {">0.3":>6} {">0.5":>6}')
for name, arr, n in [
    ('LSTM market',   lstm_price_corr,   64),
    ('Vision',         vision_price_corr, 256),
]:
    print(f'{name:<20} {n:>6d} {arr.mean():>7.3f} {arr.max():>7.3f} '
          f'{int((arr>0.1).sum()):>6d} {int((arr>0.3).sum()):>6d} {int((arr>0.5).sum()):>6d}')

## Histogram comparison
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(lstm_price_corr,   bins=30, alpha=0.6, label='LSTM (64 dims)',   color='tab:orange')
ax.hist(vision_price_corr, bins=30, alpha=0.6, label='Vision (256 dims)', color='tab:green')
ax.set_xlabel('|Spearman(dim, log_price)|')
ax.set_ylabel('Number of dimensions')
ax.set_title('Per-dim price correlation strength: LSTM market vs vision')
ax.legend()
plt.tight_layout()
plt.savefig(CONFIG['results_dir']/'figures'/'embedding_price_correlation_comparison.png',
             dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
## (XGBoost): t-SNE on XGBoost leaf-PCA embedding

X_xgb = xgb_df[xgb_cols].values
xgb_2d = TSNE(n_components=2, perplexity=30, random_state=CONFIG['seed'],
               init='pca', learning_rate='auto').fit_transform(X_xgb)
xgb_df['tsne_x'] = xgb_2d[:, 0]
xgb_df['tsne_y'] = xgb_2d[:, 1]
xgb_df['price_quintile'] = pd.qcut(xgb_df['price'], 5, labels=False, duplicates='drop')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for g, color in zip([8, 9, 10], ['tab:blue', 'tab:orange', 'tab:green']):
    m = xgb_df['grade'] == g
    axes[0].scatter(xgb_df.loc[m, 'tsne_x'], xgb_df.loc[m, 'tsne_y'],
                     c=color, label=f'PSA {g}', alpha=0.4, s=12)
axes[0].set_title('XGBoost embedding: t-SNE coloured by grade')
axes[0].legend(); axes[0].set_xlabel('t-SNE 1'); axes[0].set_ylabel('t-SNE 2')

sc2 = axes[1].scatter(xgb_df['tsne_x'], xgb_df['tsne_y'],
                       c=xgb_df['price_quintile'], cmap='viridis', alpha=0.5, s=12)
axes[1].set_title('XGBoost embedding: t-SNE coloured by price quintile')
axes[1].set_xlabel('t-SNE 1'); axes[1].set_ylabel('t-SNE 2')
plt.colorbar(sc2, ax=axes[1], label='Price quintile')
plt.tight_layout()
plt.savefig(CONFIG['results_dir']/'figures'/'xgb_embedding_tsne.png',
             dpi=120, bbox_inches='tight')
plt.show()

## L3 (XGBoost): Spearman correlation per dim
xgb_price_corr = np.array([abs(spearmanr(xgb_df[c], xgb_df['log_price'])[0]) for c in xgb_cols])
print('\nXGBoost: per-dim |Spearman(dim, log_price)|')
print(f'mean: {xgb_price_corr.mean():.3f}')
print(f'max:  {xgb_price_corr.max():.3f}')
print(f'dims with |ρ| > 0.1: {(xgb_price_corr > 0.1).sum()}/64')
print(f'dims with |ρ| > 0.3: {(xgb_price_corr > 0.3).sum()}/64')

In [ ]:
## Final comparison table: all three embeddings side by side

print('FULL per-dim |Spearman| across all three embeddings')
print(f'{"Embedding":<20} {"n_dims":>6} {"mean":>7} {"max":>7} {">0.1":>6} {">0.3":>6} {">0.5":>6}')
for name, arr, n in [
    ('LSTM market',   lstm_price_corr,   64),
    ('XGBoost market', xgb_price_corr,   64),
    ('Vision',         vision_price_corr, 256),
]:
    print(f'{name:<20} {n:>6d} {arr.mean():>7.3f} {arr.max():>7.3f} '
          f'{int((arr>0.1).sum()):>6d} {int((arr>0.3).sum()):>6d} {int((arr>0.5).sum()):>6d}')

## Three-way histogram
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(lstm_price_corr,   bins=30, alpha=0.5, label='LSTM market (64)',   color='tab:orange', density=True)
ax.hist(xgb_price_corr,    bins=30, alpha=0.5, label='XGBoost market (64)', color='tab:red',    density=True)
ax.hist(vision_price_corr, bins=30, alpha=0.5, label='Vision (256)',        color='tab:green',  density=True)
ax.set_xlabel('|Spearman(dim, log_price)|')
ax.set_ylabel('Density')
ax.set_title('Per-dim price correlation strength: three embeddings')
ax.legend()
plt.tight_layout()
plt.savefig(CONFIG['results_dir']/'figures'/'embedding_price_correlation_three_way.png',
             dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
section12_results = {
    'lstm': {
        'n_dims': 64,
        'internal_pca_90_pct_dims': int(n_90),
        'internal_pca_95_pct_dims': int(n_95),
        'price_corr_mean': float(lstm_price_corr.mean()),
        'price_corr_max':  float(lstm_price_corr.max()),
        'n_dims_corr_gt_0.1': int((lstm_price_corr > 0.1).sum()),
        'n_dims_corr_gt_0.3': int((lstm_price_corr > 0.3).sum()),
        'within_card_mean_cosine': float(within_dist.mean()),
        'between_card_mean_cosine': float(between_dist.mean()),
        'within_between_ratio': float(within_dist.mean() / between_dist.mean()),
    },
    'xgboost': {
        'n_dims': 64,
        'price_corr_mean': float(xgb_price_corr.mean()),
        'price_corr_max':  float(xgb_price_corr.max()),
        'n_dims_corr_gt_0.1': int((xgb_price_corr > 0.1).sum()),
        'n_dims_corr_gt_0.3': int((xgb_price_corr > 0.3).sum()),
    },
    'vision_reference': {
        'n_dims': 256,
        'price_corr_mean': float(vision_price_corr.mean()),
        'price_corr_max':  float(vision_price_corr.max()),
        'n_dims_corr_gt_0.1': int((vision_price_corr > 0.1).sum()),
        'n_dims_corr_gt_0.3': int((vision_price_corr > 0.3).sum()),
    },
}

with open(CONFIG['results_dir'] / 'section12_embedding_quality.json', 'w') as f:
    json.dump(section12_results, f, indent=2)

print(f'Saved: {CONFIG["results_dir"] / "section12_embedding_quality.json"}')

## Section 13: Fusion readiness check

Verify alignment and completeness across all four embeddings:
- Vision identity (256 dims, Stage 1: identity_embeddings_v2.parquet)
- Vision condition (256 dims, Stage 2: visual_embeddings_v2.parquet)
- Market LSTM (64 dims: market_embeddings_lstm.parquet)
- Market XGBoost (64 dims: market_embeddings_static_cal.parquet)

Also: lock the 13-variant fusion ablation matrix and the fusion-head
architecture as a data contract for the fusion notebook.

In [ ]:
## Load and verify listing_id alignment across all four embeddings

embeddings_dir = Path(PROJECT_ROOT / 'data/embeddings')

identity_emb = pd.read_parquet(embeddings_dir / 'identity_embeddings_v2.parquet')
condition_emb = pd.read_parquet(embeddings_dir / 'visual_embeddings_v2.parquet')
lstm_emb = pd.read_parquet(embeddings_dir / 'market_embeddings_lstm.parquet')
xgb_emb  = pd.read_parquet(embeddings_dir / 'market_embeddings_static_cal.parquet')

## Auto-detect embedding column prefixes (identity file may use different naming)
def detect_emb_cols(emb_df, likely_prefixes):
    for p in likely_prefixes:
        cols = [c for c in emb_df.columns if c.startswith(p)]
        if cols:
            return cols
    ## fallback: any column that isn't metadata
    meta = {'listing_id','card_name','grade','price','log_price','split','date_sold'}
    return [c for c in emb_df.columns if c not in meta]

id_cols   = detect_emb_cols(identity_emb, ['ident_emb_', 'id_emb_', 'identity_emb_'])
cond_cols = detect_emb_cols(condition_emb, ['cond_emb_'])
lstm_cols = detect_emb_cols(lstm_emb,  ['mkt_lstm_'])
xgb_cols  = detect_emb_cols(xgb_emb,   ['mkt_sc_'])

print('EMBEDDING INVENTORY')
print(f'{"Embedding":<20} {"rows":>6} {"dims":>6}')
print(f'{"Vision identity":<20} {len(identity_emb):>6d} {len(id_cols):>6d}')
print(f'{"Vision condition":<20} {len(condition_emb):>6d} {len(cond_cols):>6d}')
print(f'{"Market LSTM":<20} {len(lstm_emb):>6d} {len(lstm_cols):>6d}')
print(f'{"Market XGBoost":<20} {len(xgb_emb):>6d} {len(xgb_cols):>6d}')

id_set   = set(identity_emb['listing_id'])
cond_set = set(condition_emb['listing_id'])
lstm_set = set(lstm_emb['listing_id'])
xgb_set  = set(xgb_emb['listing_id'])
df_set   = set(df['listing_id'])

print('\nLISTING_ID SET ALIGNMENT')
print(f'df (master):       {len(df_set)}')
print(f'Vision identity:   {len(id_set)}  (overlap with df: {len(id_set & df_set)})')
print(f'Vision condition:  {len(cond_set)}  (overlap with df: {len(cond_set & df_set)})')
print(f'Market LSTM:       {len(lstm_set)}  (overlap with df: {len(lstm_set & df_set)})')
print(f'Market XGBoost:    {len(xgb_set)}  (overlap with df: {len(xgb_set & df_set)})')

common_all = id_set & cond_set & lstm_set & xgb_set & df_set
print(f'\nFour-way intersection with df: {len(common_all)} / {len(df_set)}')
assert len(common_all) == 3812, f'FAIL: expected 3812 common rows, got {len(common_all)}'
print('PASS: all four embeddings cover identical listing_ids')

In [ ]:
## Zero-vector audit across all four embeddings

print('ZERO VECTOR AUDIT')
zero_vectors_per_embedding = {}
for name, emb_df, cols in [
    ('Vision identity',  identity_emb,  id_cols),
    ('Vision condition', condition_emb, cond_cols),
    ('Market LSTM',      lstm_emb,      lstm_cols),
    ('Market XGBoost',   xgb_emb,       xgb_cols),
]:
    norms = np.linalg.norm(emb_df[cols].values, axis=1)
    n_zero = int((norms < 1e-8).sum())
    n_nan  = int(np.isnan(emb_df[cols].values).sum())
    zero_ids = emb_df.loc[norms < 1e-8, 'listing_id'].tolist() if n_zero > 0 else []
    zero_vectors_per_embedding[name] = zero_ids
    print(f'{name:<20}: zero vectors = {n_zero}, NaN cells = {n_nan}')
    if n_zero > 0:
        print(f'Zero-vector listing_ids: {zero_ids}')

## Capture condition zero vectors specifically known failure mode
cond_zero_ids = zero_vectors_per_embedding['Vision condition']
print(f'\nCondition-embedding zero vectors: {len(cond_zero_ids)}')
print('These rows will feed zero-vector condition inputs into any fusion variant')
print('that includes condition (variants 4, 7, 10, 12, 13). The fusion notebook')
print('will track predictions for these rows separately.')

In [ ]:
## Build the master aligned dataframe that the fusion notebook consumes except for variant 2

## Start from df
master = df[['listing_id', 'card_name', 'grade', 'price', 'log_price',
             'split', 'date_sold']].copy()

## Identity: Normalise column names
rename_map = {c: f'ident_emb_{i:03d}' for i, c in enumerate(id_cols)}
identity_renamed = identity_emb.rename(columns=rename_map)
id_cols_final = [f'ident_emb_{i:03d}' for i in range(len(id_cols))]
master = master.merge(
    identity_renamed[['listing_id'] + id_cols_final],
    on='listing_id', how='left'
)

## Condition
master = master.merge(
    condition_emb[['listing_id'] + cond_cols],
    on='listing_id', how='left'
)

## Market LSTM
master = master.merge(
    lstm_emb[['listing_id'] + lstm_cols],
    on='listing_id', how='left'
)

## Market XGBoost
master = master.merge(
    xgb_emb[['listing_id'] + xgb_cols],
    on='listing_id', how='left'
)

## Flag condition zero vectors
master['condition_zero_flag'] = master['listing_id'].isin(cond_zero_ids)

## Shape verification
expected_cols = (7                               ## metadata
                + 1                              ## flag
                + len(id_cols_final)
                + len(cond_cols)
                + len(lstm_cols)
                + len(xgb_cols))
print(f'Master fusion dataframe: {master.shape}')
print(f'Expected columns: {expected_cols}')
assert master.shape[0] == 3812
assert master.shape[1] == expected_cols

## NaN check per embedding group
embed_col_groups = {
    'identity':  id_cols_final,
    'condition': cond_cols,
    'lstm':      lstm_cols,
    'xgb':       xgb_cols,
}
print('\nNaN check per embedding group:')
for group, cols in embed_col_groups.items():
    n_nan = master[cols].isna().sum().sum()
    print(f'  {group:<10}: {n_nan} NaN cells ({len(cols)} dims)')
    assert n_nan == 0, f'NaN found in {group}'

print('\nSAMPLES PER SPLIT:')
print(master['split'].value_counts().sort_index())

## viability
if master.shape[0] < 300:
    print('\nFLAG: < 300 fusion-ready samples: fusion viability at risk')
else:
    print(f'\nOK: {master.shape[0]} fusion-ready samples (>>300 threshold)')

In [ ]:
master_path = embeddings_dir / 'fusion_master.parquet'
master.to_parquet(master_path, index=False)
print(f'Saved master: {master_path}')
print(f'Shape: {master.shape}, size: {master_path.stat().st_size / 1e6:.1f} MB')

column_contract = {
    'metadata_cols':       ['listing_id', 'card_name', 'grade', 'price',
                             'log_price', 'split', 'date_sold'],
    'flag_cols':           ['condition_zero_flag'],
    'identity_cols':       id_cols_final,
    'condition_cols':      cond_cols,
    'market_lstm_cols':    lstm_cols,
    'market_xgb_cols':     xgb_cols,
    'n_identity_dims':     len(id_cols_final),
    'n_condition_dims':    len(cond_cols),
    'n_market_lstm_dims':  len(lstm_cols),
    'n_market_xgb_dims':   len(xgb_cols),
    'total_rows':          len(master),
    'flagged_listing_ids': cond_zero_ids,
}
with open(CONFIG['results_dir'] / 'fusion_column_contract.json', 'w') as f:
    json.dump(column_contract, f, indent=2, default=str)
print(f'Saved column contract: fusion_column_contract.json')

In [ ]:
## 13-variant ablation matrix. Data contract consumed by fusion notebook

FUSION_VARIANTS = [
    {'id':  1, 'label': 'Sanity (train mean)',          'inputs': [],                                       'type': 'sanity'},
    {'id':  2, 'label': 'Monolithic (XGBoost raw)',     'inputs': ['raw_features'],                          'type': 'monolithic_xgb'},
    {'id':  3, 'label': 'Vision-identity only',          'inputs': ['identity'],                              'type': 'mlp'},
    {'id':  4, 'label': 'Vision-condition only',         'inputs': ['condition'],                             'type': 'mlp'},
    {'id':  5, 'label': 'Market-LSTM only',              'inputs': ['market_lstm'],                           'type': 'mlp'},
    {'id':  6, 'label': 'Market-XGBoost only',           'inputs': ['market_xgb'],                            'type': 'mlp'},
    {'id':  7, 'label': 'Vision full (id + cond)',       'inputs': ['identity', 'condition'],                 'type': 'mlp'},
    {'id':  8, 'label': 'Market full (LSTM + XGB)',      'inputs': ['market_lstm', 'market_xgb'],             'type': 'mlp'},
    {'id':  9, 'label': 'Fusion: id + mkt-LSTM',         'inputs': ['identity', 'market_lstm'],               'type': 'mlp'},
    {'id': 10, 'label': 'Fusion: id + cond + mkt-LSTM',  'inputs': ['identity', 'condition', 'market_lstm'], 'type': 'mlp'},
    {'id': 11, 'label': 'Fusion: id + mkt-XGB',          'inputs': ['identity', 'market_xgb'],                'type': 'mlp'},
    {'id': 12, 'label': 'Fusion: id + cond + mkt-XGB',   'inputs': ['identity', 'condition', 'market_xgb'],  'type': 'mlp'},
    {'id': 13, 'label': 'Fusion: full (all four)',        'inputs': ['identity', 'condition',
                                                                     'market_lstm', 'market_xgb'],           'type': 'mlp'},
]

input_dim_map = {
    'identity':    len(id_cols_final),
    'condition':   len(cond_cols),
    'market_lstm': len(lstm_cols),
    'market_xgb':  len(xgb_cols),
}

print('FUSION ABLATION MATRIX (13 variants)')
print(f'{"#":>2} {"Label":<40} {"Type":<18} {"Input dim":>10}')
for v in FUSION_VARIANTS:
    if v['type'] in ('sanity', 'monolithic_xgb'):
        dim_str = 'n/a'
    else:
        dim_str = str(sum(input_dim_map[i] for i in v['inputs']))
    print(f'{v["id"]:>2} {v["label"]:<40} {v["type"]:<18} {dim_str:>10}')

print('\nKEY ABLATION TESTS')
ablations = [
    ('10 - 9',  'Does condition add lift over identity+market-LSTM?'),
    ('12 - 11', 'Does condition add lift over identity+market-XGBoost?'),
    ('9 vs 11', 'Which market encoder fuses better with identity?'),
    ('13 - 10', 'Does adding market-XGBoost help once market-LSTM is present?'),
    ('13 - 12', 'Does adding market-LSTM help once market-XGBoost is present?'),
    ('13 vs 2', 'decomposition headline: decomposition vs monolithic'),
    ('13 vs 7', 'Does adding market to vision help?'),
    ('13 vs 8', 'Does adding vision to market help?'),
]
for test, question in ablations:
    print(f'{test:<10}: {question}')

In [ ]:
## MLP fusion head. Identical across all MLP variants (3-13)

FUSION_MLP_CONFIG = {
    'hidden_layers':           [256, 64],
    'activation':              'relu',
    'dropout':                 0.2,
    'output_dim':              1,
    'loss':                    'huber',
    'huber_delta':             1.0,
    'optimizer':               'adam',
    'learning_rate':           1e-3,
    'weight_decay':            1e-5,
    'batch_size':              64,
    'max_epochs':              150,
    'early_stopping_patience': 15,
    'early_stopping_metric':   'val_loss',
    'seeds':                   [42, 123, 7],
    'target':                  'log_price',
    'eval_convention':         ('dollar scale via expm1, clip negatives, '
                                 'R² on log scale: uses evaluate_predictions()'),
}

MONOLITHIC_XGB_CONFIG = {
    'inputs': 'raw engineered features (31 features, same as V0 in Section 9)',
    'params': {
        'objective':             'reg:squarederror',
        'max_depth':             6,
        'learning_rate':         0.05,
        'subsample':             0.8,
        'colsample_bytree':      0.8,
        'min_child_weight':      3,
        'reg_lambda':            1.0,
        'n_estimators':          300,
        'early_stopping_rounds': 30,
        'seed':                  42,
    },
    'note': 'Identical protocol to Section 9 V0. Represents the "no decomposition" '
             'baseline for decomposition modular-vs-monolithic comparison. Same seeds as MLP.',
}

print('FUSION MLP ARCHITECTURE (locked)')
print(json.dumps(FUSION_MLP_CONFIG, indent=2))
print('\nMONOLITHIC BASELINE ARCHITECTURE (locked)')
print(json.dumps(MONOLITHIC_XGB_CONFIG, indent=2))

fusion_contract = {
    'column_contract':   column_contract,
    'variants':          FUSION_VARIANTS,
    'mlp_config':        FUSION_MLP_CONFIG,
    'monolithic_config': MONOLITHIC_XGB_CONFIG,
    'evaluation_helper': 'evaluate_predictions (Section 5)',
    'key_ablations':     [
        {'test': '10 - 9',  'question': 'Does condition add lift over identity+LSTM?'},
        {'test': '12 - 11', 'question': 'Does condition add lift over identity+XGB?'},
        {'test': '9 vs 11', 'question': 'LSTM vs XGB as fusion partner'},
        {'test': '13 vs 2', 'question': 'decomposition decomposition vs monolithic'},
        {'test': '13 vs 7', 'question': 'Market adds to vision'},
        {'test': '13 vs 8', 'question': 'Vision adds to market'},
    ],
}

with open(CONFIG['results_dir'] / 'fusion_contract.json', 'w') as f:
    json.dump(fusion_contract, f, indent=2, default=str)

print(f'\nSaved fusion contract: {CONFIG["results_dir"] / "fusion_contract.json"}')
print('\nFusion notebook (03_Fusion_Module.ipynb) will consume:')
print(f'- {master_path}')
print(f'- {CONFIG["results_dir"] / "fusion_contract.json"}')

# Section 14: Failure mode documentation

This section consolidates every negative result, limitation, and methodological weakness identified during Sections 0–13 of the market module. All findings are sourced from existing artefacts.

The output is a single `section14_failure_modes.json` capturing, in structured form:
- Data survival through feature engineering
- Leakage verification in rolling features
- Train-val-test generalisation gaps
- Strength of primary model over static baseline
- Embedding stability and collapse
- All additional negative findings

This section is intentionally read-only over prior artefacts. The project writeup references the JSON produced here.

In [ ]:
## Section 14: Failure Mode Documentation
## Consolidates negative results and methodological limitations from Sections 0-13.

results_dir    = Path(CONFIG['results_dir'])
embeddings_dir = Path(CONFIG['embeddings_dir'])

print('SECTION 14: FAILURE MODE DOCUMENTATION')
print(f'Results directory : {results_dir}')
print(f'Embeddings dir    : {embeddings_dir}')

In [ ]:
## Load every artefact this section audits. No recomputation.
## Filenames match the contents of results/market/ on disk.

artefacts = {}

artefact_files = {
    'market_config'        : results_dir / 'market_config.json',
    'temporal_split'       : results_dir / 'temporal_split_info.json',
    'feature_columns'      : results_dir / 'feature_columns.json',
    'target_handling'      : results_dir / 'target_handling.json',
    'lstm_feasibility'     : results_dir / 'lstm_feasibility.json',
    'primary_results'      : results_dir / 'primary_xgboost_results.json',
    'section7_baselines'   : results_dir / 'section7_baselines.json',
    'section8_interp'      : results_dir / 'section8_interpretability.json',
    'section9_ablation'    : results_dir / 'section9_ablation.json',
    'section10_stability'  : results_dir / 'section10_stability.json',
    'section11_embeddings' : results_dir / 'section11_embeddings.json',
    'section12_emb_quality': results_dir / 'section12_embedding_quality.json',
    'fusion_contract'      : results_dir / 'fusion_contract.json',
    'fusion_col_contract'  : results_dir / 'fusion_column_contract.json',
}

for name, path in artefact_files.items():
    if path.exists():
        with open(path, 'r') as f:
            artefacts[name] = json.load(f)
        print(f'{name:<22} loaded from {path.name}')
    else:
        artefacts[name] = None
        print(f'{name:<22} expected at {path}')

## Reload master for the live integrity check in N2
master_path = embeddings_dir / 'fusion_master.parquet'
master = pd.read_parquet(master_path)
print(f'\nfusion_master loaded: {master.shape}')

In [ ]:
## Did feature engineering or filtering destroy the dataset?

n1 = {}

n1['raw_transactions_input']    = 3812            ## PSA 8-10 filter applied upstream
n1['post_feature_eng_rows']     = int(master.shape[0])

## Counts per split from master
split_counts = master['split'].value_counts().to_dict()
n1['train_rows'] = int(split_counts.get('train', 0))
n1['val_rows']   = int(split_counts.get('val',   0))
n1['test_rows']  = int(split_counts.get('test',  0))

n1['survival_rate']    = round(n1['post_feature_eng_rows'] / n1['raw_transactions_input'], 4)
n1['fusion_viable']    = n1['post_feature_eng_rows'] >= 300   ## threshold
n1['row_loss_from_FE'] = n1['raw_transactions_input'] - n1['post_feature_eng_rows']

## Known sparsity and integrity flags
n1['sparse_year_flag']  = {
    'year': 2022,
    'n_transactions': 148,
    'note': 'Early-period sparsity. More NaN rolling features, under-represented in training.',
}
n1['repeat_certs_kept'] = {
    'count': 614,
    'note' : 'Repeat PSA cert sales flagged and kept as valid resales.',
}

print('DATA SURVIVAL')
for k, v in n1.items():
    print(f'{k:<25}: {v}')

n1['verdict'] = (
    'feature engineering preserved all 3,812 rows (XGBoost handles NaN '
    'natively, no row dropping). Fusion-viable. 2022 sparsity (148 txns) means '
    'early training is under-represented, documented limitation.'
)
print(f"\nVERDICT: {n1['verdict']}")

In [ ]:
## Leakage integrity in rolling features and temporal split.

n2 = {}

## Design-level evidence from Section 3 (strictly past rolling, spot-checked)
n2['design_evidence'] = {
    'strictly_past_windows': True,
    'method': 'Rolling features computed per (card_name, grade) group with '
              'current transaction excluded from its own window.',
    'section3_spot_check': 'roll_mean_7d = 480.53 (manual and automated match).',
}

## (b) Live structural checks on the master dataframe
assert master['listing_id'].is_unique, 'FAIL: duplicate listing_ids in master'
split_by_id = master.groupby('listing_id')['split'].nunique()
n2['listing_id_split_uniqueness'] = bool((split_by_id == 1).all())

## (c) Temporal ordering: train max <= val min, val max <= test min
train_dates = pd.to_datetime(master.loc[master['split'] == 'train', 'date_sold'])
val_dates   = pd.to_datetime(master.loc[master['split'] == 'val',   'date_sold'])
test_dates  = pd.to_datetime(master.loc[master['split'] == 'test',  'date_sold'])

n2['train_max_date'] = str(train_dates.max().date())
n2['val_min_date']   = str(val_dates.min().date())
n2['val_max_date']   = str(val_dates.max().date())
n2['test_min_date']  = str(test_dates.min().date())
n2['test_max_date']  = str(test_dates.max().date())

n2['temporal_ordering_ok'] = bool(
    (train_dates.max() <= val_dates.min()) and
    (val_dates.max()   <= test_dates.min())
)

print('LEAKAGE VERIFICATION')
for k, v in n2.items():
    if isinstance(v, dict):
        print(f'{k}:')
        for k2, v2 in v.items():
            print(f'{k2:<28}: {v2}')
    else:
        print(f'{k:<30}: {v}')

n2['verdict'] = (
    'Rolling windows strictly past; temporal ordering confirmed '
    '(train_max <= val_min, val_max <= test_min); every listing_id assigned '
    'to exactly one split. No leakage detected.'
)
print(f"\nVERDICT: {n2['verdict']}")

In [ ]:
## Train -> val -> test R² degradation.
## Source: primary_xgboost_results.json for V0 hybrid (single-seed, Section 6).
## Seed-averaged test values from section10_stability.json:I2_seed_robustness.summary.

n3 = {}

primary = artefacts.get('primary_results') or {}
metrics = primary.get('metrics', {})

n3['v0_hybrid_single_seed'] = {
    'train_r2_log'    : metrics.get('train', {}).get('r2_log'),
    'val_r2_log'      : metrics.get('val',   {}).get('r2_log'),
    'test_r2_log'     : metrics.get('test',  {}).get('r2_log'),
    'train_mae_usd'   : metrics.get('train', {}).get('mae_usd'),
    'val_mae_usd'     : metrics.get('val',   {}).get('mae_usd'),
    'test_mae_usd'    : metrics.get('test',  {}).get('mae_usd'),
    'best_iteration'  : primary.get('best_iteration'),
    'best_val_rmse_log': primary.get('best_val_rmse_log'),
}

def _gap(a, b):
    if a is None or b is None:
        return None
    return round(a - b, 4)

tr = n3['v0_hybrid_single_seed']['train_r2_log']
va = n3['v0_hybrid_single_seed']['val_r2_log']
te = n3['v0_hybrid_single_seed']['test_r2_log']
n3['v0_hybrid_single_seed']['train_minus_val']  = _gap(tr, va)
n3['v0_hybrid_single_seed']['val_minus_test']   = _gap(va, te)
n3['v0_hybrid_single_seed']['train_minus_test'] = _gap(tr, te)

## Seed-averaged test numbers pulled by exact label keys from I2_seed_robustness.summary
stab = artefacts.get('section10_stability') or {}
i2_summary = (stab.get('I2_seed_robustness') or {}).get('summary', {})

n3['seed_averaged_test'] = {
    'V0_hybrid'      : i2_summary.get('V0_hybrid'),
    'Static+calendar': i2_summary.get('Static+calendar'),
    'LSTM'           : i2_summary.get('LSTM'),
}

## Val residual diagnostics (Section 6) — systematic bias
n3['val_residual_diagnostics'] = primary.get('val_residual_diagnostics')

n3['findings'] = [
    'V0 hybrid single-seed: train R²(log) = 0.86, val R²(log) = 0.38, '
    'test R²(log) = -0.13. Gaps are 0.48 (train->val) and 0.51 (val->test), '
    'totalling 0.99 train->test. The degradation is not classical '
    'overfitting; it is a regime shift across the 4x train->test price '
    'drift (train median $134 -> test median $520).',
    'Section 8 diagnosed the mechanism: days_since_start has 6% gain '
    'importance but 0% permutation importance — a pure training-period '
    'shortcut carrying no information under drift.',
    'Val residuals have mean 0.233 and median 0.109 on the log scale — '
    'the model systematically under-predicts on val, consistent with '
    'drift-driven upward price-level shift.',
    'Seed-averaged stability (n=3) confirms the pattern is not seed noise: '
    'V0_hybrid r2_mean = -0.117 (std 0.043); Static+calendar +0.144 '
    '(std 0.026); LSTM +0.273 (std 0.020). Rankings identical in all 3 seeds.',
]

print('GENERALISATION GAP')
print('V0 hybrid (single-seed, Section 6):')
for k, v in n3['v0_hybrid_single_seed'].items():
    print(f'{k:<22}: {v}')
print('\nSeed-averaged test summary (from I2_seed_robustness.summary):')
for name, block in n3['seed_averaged_test'].items():
    if block is None:
        print(f'{name:<18}: (missing)')
    else:
        print(f'{name:<18}: r2_mean = {block.get("r2_mean"):+.4f} '
              f'(std {block.get("r2_std"):.4f}), '
              f'mae_mean = ${block.get("mae_mean"):.0f}')
print('\nFindings:')
for i, f in enumerate(n3['findings'], 1):
    print(f'[{i}] {f}')

n3['verdict'] = (
    'FLAGGED: the V0 train->test gap is structural (drift-driven), not '
    'classical overfitting. Diagnosed via gain-vs-permutation disagreement '
    'on days_since_start. No regularisation fix applies at current data '
    'scale.'
)
print(f"\nVERDICT: {n3['verdict']}")

In [ ]:
## Does the primary engineered-features model beat the simplest baseline?
## Source: section10_stability.json:I2_seed_robustness.summary (seed-averaged headline)
##         primary_xgboost_results.json:coverage_stratified_metrics (V0 single-seed disaggregation)

n4 = {}

## (a) Seed-averaged headline: label keys taken verbatim from I2_seed_robustness.summary
n4['seed_averaged'] = {
    'V0_hybrid'       : i2_summary.get('V0_hybrid'),
    'Static+calendar' : i2_summary.get('Static+calendar'),
    'LSTM'            : i2_summary.get('LSTM'),
}

## (b) Core gaps (seed-averaged r2_mean)
def _r2_mean(name):
    block = i2_summary.get(name) or {}
    return block.get('r2_mean')

v0   = _r2_mean('V0_hybrid')
stat = _r2_mean('Static+calendar')
lstm = _r2_mean('LSTM')

n4['v0_minus_static_cal']   = _gap(v0,   stat)
n4['lstm_minus_static_cal'] = _gap(lstm, stat)
n4['lstm_minus_v0']         = _gap(lstm, v0)

## (c) Coverage-stratified disaggregation on V0 single-seed test (Section 6)
##     This is the sharper version of the N4 finding: V0 is net-harmful on rolling-coverage rows.
cov = primary.get('coverage_stratified_metrics', {})
n4['v0_coverage_stratified'] = {
    'has_7d_rolling': cov.get('has_7d_rolling'),
    'cold_start'    : cov.get('cold_start'),
}

has_r2 = (cov.get('has_7d_rolling') or {}).get('r2_log')
cold_r2 = (cov.get('cold_start')    or {}).get('r2_log')
n4['v0_coverage_gap_cold_minus_has'] = _gap(cold_r2, has_r2)

## (d) Rankings per seed: stability check
n4['rankings_per_seed'] = (stab.get('I2_seed_robustness') or {}).get('rankings_per_seed', {})

## (e) Ablation confirms no feature subset recovers positive test R²
abl = artefacts.get('section9_ablation') or {}
n4['ablation_summary'] = {
    'variants_tested'    : abl.get('variants_tested', ['V0', 'V1', 'V2', 'V3']),
    'any_positive_test_r2': abl.get('any_positive_test_r2', False),
    'note': 'Four hybrid variants tested in Section 9; none recovered positive '
            'test R². Drift is structural, not feature-selection-fixable.',
}

n4['findings'] = [
    'The primary engineered-features model (V0 hybrid, 31 features) '
    'underperforms the 6-feature static+calendar baseline on seed-averaged '
    'test R²(log) by 0.261 (V0 -0.117 vs Static+calendar +0.144).',
    'LSTM wins seed-averaged at +0.273 but Section 8 position-ablation shows '
    'it relies primarily on the two most recent timesteps (54% of position importance, vs 46% distributed across the remaining eight).'
    'This is regime-local last-price anchoring, with some medium-range context, not long-memory sequence modelling.',
    'Coverage-stratified disaggregation is the sharpest finding: V0 test '
    f'R²(log) on has_7d_rolling = {has_r2} vs cold_start = {cold_r2}. '
    'The model performs worse on rows where rolling features are available '
    'than on rows where they are NaN. Rolling features are net-harmful on '
    'test under drift.',
    'Section 9 ablation (V0-V3) confirmed no feature subset recovers positive '
    'test R². Drift is structural, not a feature-selection problem.',
    'Rankings identical across all 3 seeds: [LSTM, Static+calendar, V0_hybrid]. '
    'The finding is not seed-sensitive.',
]

print('BASELINE COMPARISON (seed-averaged test R²(log))')
for name in ['V0_hybrid', 'Static+calendar', 'LSTM']:
    block = n4['seed_averaged'][name]
    if block is None:
        print(f'{name:<18}: (missing)')
    else:
        print(f'{name:<18}: r2_mean = {block["r2_mean"]:+.4f} '
              f'(std {block["r2_std"]:.4f})')

print('')
print(f'V0 - Static+cal   gap : {n4["v0_minus_static_cal"]:+.4f}')
print(f'LSTM - Static+cal gap : {n4["lstm_minus_static_cal"]:+.4f}')
print(f'LSTM - V0         gap : {n4["lstm_minus_v0"]:+.4f}')

print('\nV0 coverage-stratified (single-seed test):')
print(f'has_7d_rolling r2_log : {has_r2}')
print(f'cold_start     r2_log : {cold_r2}')
print(f'gap (cold - has)      : {n4["v0_coverage_gap_cold_minus_has"]}')

print('\nRankings per seed:')
for seed, ranking in n4['rankings_per_seed'].items():
    print(f'seed {seed}: {ranking}')

print('\nFindings:')
for i, f in enumerate(n4['findings'], 1):
    print(f'[{i}] {f}')

n4['verdict'] = (
    'CORE NEGATIVE FINDING: engineered temporal features degrade valuation '
    'under 4x train-to-test price drift. Coverage-stratified analysis shows '
    'the degradation occurs specifically on rows where rolling features are '
    'present. market-encoder not supported. This is the central empirical finding of '
    'the market module.'
)
print(f"\nVERDICT: {n4['verdict']}")

In [ ]:
## Embedding collapse, weak per-dim signal, drift exposure.
## Sources: section12_embedding_quality.json, master dataframe zero-vector audit.

n5 = {}

emb_quality = artefacts.get('section12_emb_quality') or {}

n5['lstm_embedding']               = emb_quality.get('lstm', {})
n5['xgboost_embedding']            = emb_quality.get('xgboost', {})
n5['vision_reference_embedding'] = emb_quality.get('vision_reference', {})  #

## Zero-vector audit replicated on master (live)
def _zero_count(df, prefix):
    cols = [c for c in df.columns if c.startswith(prefix)]
    if not cols:
        return None, 0
    norms = np.linalg.norm(df[cols].values, axis=1)
    return int((norms < 1e-8).sum()), len(cols)

## zero-vector audit
id_zero,   id_n   = _zero_count(master, 'ident_emb_')
cond_zero, cond_n = _zero_count(master, 'cond_emb_')
lstm_zero, lstm_n = _zero_count(master, 'mkt_lstm_')
xgb_zero,  xgb_n  = _zero_count(master, 'mkt_sc_')

print(f'identity  : n_zero = {id_zero}, n_dims = {id_n}')
print(f'condition : n_zero = {cond_zero}, n_dims = {cond_n}')
print(f'lstm      : n_zero = {lstm_zero}, n_dims = {lstm_n}')
print(f'xgboost   : n_zero = {xgb_zero}, n_dims = {xgb_n}')

n5['zero_vectors'] = {
    'identity' : {'n_zero': id_zero,   'n_dims': id_n},
    'condition': {'n_zero': cond_zero, 'n_dims': cond_n},
    'lstm'     : {'n_zero': lstm_zero, 'n_dims': lstm_n},
    'xgboost'  : {'n_zero': xgb_zero,  'n_dims': xgb_n},
    'flagged_listing_ids_tracked_in_fusion_contract': True,
}

n5['drift_exposure'] = [
    'LSTM embedding extracted at final real timestep, z-scored using TRAIN '
    'statistics. Under drift, test embeddings will shift relative to '
    'train-based z-scores. Monitored in fusion; no correction applied.',
    'XGBoost embedding uses leaf indices from V0 hybrid, which has negative '
    'test R². The geometric grade separation in t-SNE is interpretable but '
    'per-dim predictive signal is weak (mean |rho| = 0.119).',
    'Vision condition embedding global silhouette is near zero (-0.0185). '
    'Grade separation is local within-card, not global. By design,'
    'orthogonality to identity cost -3.1pp accuracy vs v1 (failure mode '
    'in vision module).',
]

print('EMBEDDING STABILITY')
for name in ['lstm_embedding', 'xgboost_embedding',
             'vision_reference_embedding']:
    print(f'\n{name}:')
    block = n5[name]
    if not block:
        print('(not found in section12_embedding_quality.json)')
        continue
    for k, v in block.items():
        print(f'{k:<28}: {v}')

print('\nZero-vector audit (live on master):')
for name, block in n5['zero_vectors'].items():
    if isinstance(block, dict):
        print(f'{name:<10}: n_zero = {block["n_zero"]}, n_dims = {block["n_dims"]}')
    else:
        print(f'{name:<40}: {block}')

## Verdict composed from live audit
cond_flag = (cond_zero or 0) > 0
n5['verdict'] = (
    f'MIXED. LSTM embedding strong (mean |rho| high, 4 PCA dims = 90% '
    f'variance). XGBoost embedding weak per-dim but geometrically coherent. '
    f'Vision condition embedding has ~5 effective dims by design. '
    f'{cond_zero} condition zero vectors carried through and tracked as '
    f'flagged_listing_ids in fusion_contract.'
)
print(f"\nVERDICT: {n5['verdict']}")

In [ ]:
## Sources cross-reference section10_stability, section8_interpretability,
## primary_xgboost_results.

stab_i4 = (stab.get('I4_spike_check') or {})
stab_i3 = (stab.get('I3_temporal_stability') or {})

n6 = {}

## (a) Top-10 test date share
n6['top10_test_dates_share'] = {
    'top10_pct'     : stab_i4.get('top10_pct'),
    'v1_reference'  : stab_i4.get('v1_top10_pct_reference'),
    'reduction_fold': round(76.0 / stab_i4.get('top10_pct', 1), 1),
    'interpretation': ('Co-located with drifted late-test period. Reflects '
                       'market-activity clustering (auction dates, end-of-month '
                       'sales), not a sampling artefact.'),
    'note' : ('Top 10 test dates are 18.5% of test, a 4x reduction from v1 '
              '(76%). Not eliminated. Reported as a characteristic of the '
              'test period, not a clustering bug.'),
}
## (b) Rolling coverage drift
n6['rolling_coverage_drift'] = {
    'train_has_7d_rolling': 0.332,
    'val_has_7d_rolling'  : 0.378,
    'test_has_7d_rolling' : 0.639,
    'note' : 'Test rows have nearly double the rolling-feature coverage of '
             'train. Coverage-stratified evaluation applied throughout.',
}

## (c) PSA 10 proportion drift
n6['psa10_proportion_drift'] = {
    'train': 0.257,
    'test' : 0.369,
    'note' : 'Grade mix shifts between splits. Test is richer in PSA 10, '
             'which carries higher prices, compounding the $134 -> $520 drift.',
}

## (d) Price-level drift (central framing)
n6['price_level_drift'] = {
    'train_median_usd': 134,
    'val_median_usd'  : 250,
    'test_median_usd' : 520,
    'fold_change'     : round(520 / 134, 2),
    'note' : 'Central modelling challenge. Treated as a feature of the '
             'problem, not a limitation.',
}

## (e) LSTM time-gap insensitivity (Section 8 finding)
n6['lstm_time_gap_insensitivity'] = {
    'days_since_prev_perm_delta': 0.0,
    'note' : 'Permutation importance on days_since_prev in the LSTM is ~0. '
             'The LSTM treats sequences as uniformly spaced despite 13.1-day '
             'mean inter-transaction gap. Qualifies any claim that the LSTM '
             'models irregular time structure.',
}

## (f) LSTM position concentration (Section 8)
n6['lstm_position_importance'] = {
    'recent_2_timesteps_share': 0.54,
    'note' : 'The LSTM draws 54% of position importance from the last 2 '
             'timesteps. Effectively a regime-local price anchor, not a '
             'long-memory sequence learner.',
}

## (g) days_since_start shortcut (Section 8)
n6['days_since_start_shortcut'] = {
    'gain_importance'       : 0.06,
    'permutation_importance': 0.00,
    'note' : 'High gain, zero permutation. Classic shortcut signature. '
             'Selected during training as a proxy for the training regime, '
             'carries no transferable information.',
}

## (h) V0 hybrid exclusion rationale
n6['v0_hybrid_excluded_from_fusion'] = {
    'note' : 'V0 hybrid XGBoost has negative test R². Its leaf indices would '
             'encode a model that failed out-of-sample, risking poisoning of '
             'the extrinsic branch. LSTM and static+calendar XGBoost '
             'extracted as market embeddings instead.',
}

## (i) Temporal segment degradation of V0 (single-seed, from primary_results.segment_metrics)
seg = primary.get('segment_metrics', {})
n6['v0_temporal_segments'] = {
    'early' : (seg.get('early')  or {}).get('r2_log'),
    'middle': (seg.get('middle') or {}).get('r2_log'),
    'late'  : (seg.get('late')   or {}).get('r2_log'),
    'note' : 'V0 R²(log) collapses through test: +0.28 early -> -0.32 middle '
             '-> -0.60 late. Deterioration tracks drift intensity within the '
             'test window.',
}

## (j) Temporal segments: LSTM is the only positive-R² model across all three
n6['lstm_segment_robustness'] = {
    'early_r2_log' : (stab_i3.get('early',  {}).get('LSTM')  or {}).get('r2_log'),
    'middle_r2_log': (stab_i3.get('middle', {}).get('LSTM')  or {}).get('r2_log'),
    'late_r2_log'  : (stab_i3.get('late',   {}).get('LSTM')  or {}).get('r2_log'),
    'note' : 'LSTM is the ONLY model with positive R²(log) in all three test '
             'segments. Static+cal wins early but goes negative in middle and late.',
}

## (k) Scope limits
n6['scope_limits'] = {
    'n_pokemon'     : 7,
    'n_transactions': 3812,
    'date_range'    : '2021-01-01 to 2026-04-13',
    'grade_range'   : 'PSA 8-10',
    'grading_house' : 'PSA only',
    'note' : 'Findings conditional on this dataset scale and drift magnitude. '
            }

## (l) Set metadata missingness
n6['set_metadata_missingness'] = {
    'fraction_missing': 0.325,
    'note' : 'Set field missing for 32.5% of rows. Not used as a modelling '
             'feature, reported for completeness.',
}

## (m) Permutation-on-val methodological issue
n6['permutation_on_val_issue'] = {
    'note' : 'Permutation importance measured on val '
             'cannot reliably select features for test under large '
             'val-to-test drift. Documented as methodological contribution '
             'in the discussion chapter.',
}

print('ADDITIONAL NEGATIVE FINDINGS')
for key, block in n6.items():
    print(f'\n{key}:')
    for k, v in block.items():
        print(f'  {k:<32}: {v}')

In [ ]:
## Single structured artefact for the writeup.

section14 = {
    'section'          : 14,
    'title'            : 'Failure Mode Documentation',
    'checklist_items'  : ['N1', 'N2', 'N3', 'N4', 'N5', 'N6'],
    'source_artefacts' : sorted([k for k, v in artefacts.items() if v is not None]),
    'n1_data_survival' : n1,
    'n2_leakage_check' : n2,
    'n3_generalisation': n3,
    'n4_baseline_gap'  : n4,
    'n5_embeddings'    : n5,
    'n6_additional'    : n6,
}

section14['summary'] = {
    'passes'               : ['N1', 'N2'],
    'flagged'              : ['N3', 'N5'],
    'core_negative_finding': ('N4: engineered temporal features degrade valuation '
                              'under 4x price drift. Coverage-stratified analysis '
                              'shows V0 R²(log) is lower on has_7d_rolling rows '
                              'than on cold_start rows. market-encoder not supported.'),
    'headline_for_writeup' : (
        'The market module passes data survival and leakage checks. The '
        'central negative finding is that engineered temporal features '
        '(rolling, momentum, volatility) degrade rather than improve '
        'valuation under 4x train-to-test price-level drift, with the '
        'mechanism traced to training-period shortcut learning '
        '(days_since_start: gain 6%, permutation 0%). Coverage-stratified '
        'disaggregation sharpens this: V0 performs WORSE on rows where '
        'rolling features are present than on rows where they are NaN. '
        'The LSTM wins seed-averaged (+0.273 R²_log) but by regime-local '
        'last-price anchoring (54% of position importance in the last 2 '
        'timesteps), not by long-memory or irregular-time-gap modelling. '
        'All findings hold across seeds [42, 123, 7] with identical rankings.'
    ),
}

out_path = results_dir / 'section14_failure_modes.json'
with open(out_path, 'w') as f:
    json.dump(section14, f, indent=2, default=str)

print(f'Output: {out_path}')
print(f'Size  : {out_path.stat().st_size / 1024:.1f} KB')

## Section 15: Final market-encoder assessment


**market-encoder (as posed):** Does representing market state through engineered temporal
features (rolling price averages, momentum, transaction volume, realised
volatility) improve valuation stability compared to a static metadata baseline
and a pure sequence model in thin, regime-dependent collectible markets?

**Verdict:** NOT SUPPORTED, with a qualified regime-conditional mechanism.

This section synthesises Sections 6-14. No new experiments are run.
Outputs: section15_rq3_verdict.json + a writeup-ready markdown summary.

In [ ]:
## Building the O1-O6 verdict structure from existing artefacts
## Reload the key artefacts (defensive — ensures section runs standalone)
with open(results_dir / 'section10_stability.json', 'r') as f:
    stab = json.load(f)
with open(results_dir / 'section9_ablation.json', 'r') as f:
    abl = json.load(f)
with open(results_dir / 'primary_xgboost_results.json', 'r') as f:
    primary = json.load(f)
with open(results_dir / 'section14_failure_modes.json', 'r') as f:
    failure_modes = json.load(f)

## Seed-averaged headline metrics
i2 = stab['I2_seed_robustness']['summary']

## O1: Does engineered temporal modelling outperform the static baseline?
o1 = {
    'question': 'Does V0 hybrid (31 engineered features) outperform the '
                 'static+calendar baseline (6 features) on test?',
    'answer': 'No.',
    'evidence': {
        'v0_test_r2_log':          round(i2['V0_hybrid']['r2_mean'], 4),
        'v0_test_r2_std':          round(i2['V0_hybrid']['r2_std'], 4),
        'static_cal_test_r2_log':  round(i2['Static+calendar']['r2_mean'], 4),
        'static_cal_test_r2_std':  round(i2['Static+calendar']['r2_std'], 4),
        'gap':                     round(i2['V0_hybrid']['r2_mean']
                                          - i2['Static+calendar']['r2_mean'], 4),
        'v0_test_mae_usd':         round(i2['V0_hybrid']['mae_mean'], 0),
        'static_cal_test_mae_usd': round(i2['Static+calendar']['mae_mean'], 0),
    },
    'interpretation': (
        'V0 hybrid underperforms the static+calendar baseline by 0.26 on '
        'test R²(log), seed-averaged across 3 seeds. The hybrid '
        'architecture, which is the central claim of market-encoder, does not '
        'improve out-of-sample valuation stability relative to a simpler '
        'regime-invariant baseline under the drift conditions of this '
        'dataset.'
    ),
}

## O2: Does it outperform a sequence model?
o2 = {
    'question': 'Does V0 hybrid outperform the LSTM sequence baseline on test?',
    'answer':   'No.',
    'evidence': {
        'v0_test_r2_log':   round(i2['V0_hybrid']['r2_mean'], 4),
        'lstm_test_r2_log': round(i2['LSTM']['r2_mean'], 4),
        'gap':              round(i2['V0_hybrid']['r2_mean']
                                   - i2['LSTM']['r2_mean'], 4),
        'lstm_test_mae_usd':round(i2['LSTM']['mae_mean'], 0),
    },
    'interpretation': (
        'V0 hybrid underperforms the LSTM by 0.39 on test R²(log). '
        'However, the LSTM advantage is not attributable to long-memory '
        'modelling. Section 8 position-ablation shows the LSTM relies '
        'primarily on the two most recent timesteps (54% of position '
        'importance), and permutation importance on days_since_prev is '
        '~0, meaning the model is regime-unaware of transaction timing. '
        'The LSTM advantage is regime-local last-price anchoring at '
        'inference, not structural temporal modelling.'
    ),
}

## O3: Are results stable across time periods?
seg_v0   = primary.get('segment_metrics', {})
seg_i3   = stab['I3_temporal_stability']

o3 = {
    'question': 'Are test-set results stable across temporal segments?',
    'answer':   'No for V0 hybrid, partially yes for LSTM.',
    'evidence': {
        'v0_hybrid': {
            'early_r2_log':  round(seg_v0.get('early',  {}).get('r2_log', 0), 4),
            'middle_r2_log': round(seg_v0.get('middle', {}).get('r2_log', 0), 4),
            'late_r2_log':   round(seg_v0.get('late',   {}).get('r2_log', 0), 4),
        },
        'static_plus_calendar': {
            'early_r2_log':  round(seg_i3.get('early',  {}).get('Static+cal', {}).get('r2_log', 0), 4),
            'middle_r2_log': round(seg_i3.get('middle', {}).get('Static+cal', {}).get('r2_log', 0), 4),
            'late_r2_log':   round(seg_i3.get('late',   {}).get('Static+cal', {}).get('r2_log', 0), 4),
        },
        'lstm': {
            'early_r2_log':  round(seg_i3.get('early',  {}).get('LSTM', {}).get('r2_log', 0), 4),
            'middle_r2_log': round(seg_i3.get('middle', {}).get('LSTM', {}).get('r2_log', 0), 4),
            'late_r2_log':   round(seg_i3.get('late',   {}).get('LSTM', {}).get('r2_log', 0), 4),
        },
    },
    'interpretation': (
        'V0 hybrid R²(log) collapses monotonically through test: '
        '+0.28 early -> -0.32 middle -> -0.60 late. Static+calendar is '
        'strongest in the early segment (closest to train) but goes '
        'negative in middle and late. LSTM is the only model with '
        'positive R²(log) across all three segments (+0.19/+0.28/+0.22), '
        'consistent with inference-time regime anchoring. V0 and '
        'Static+calendar both degrade as distance from training '
        'increases, at different rates.'
    ),
}

## O4: Final verdict statement
o4 = {
    'verdict': 'market-encoder NOT SUPPORTED.',
    'qualified_finding': (
        'market-encoder as posed asks whether engineered temporal features improve '
        'valuation stability compared to static and sequence baselines. '
        'The empirical answer is no. But the finding underneath the '
        'question is richer: the hybrid architecture fails because '
        'engineered rolling/momentum/volatility features encode '
        'training-period regime information that does not transfer '
        'across a 4x price-level drift. The static+calendar baseline '
        'wins near training (regime-invariant identity features); the '
        'LSTM wins under drift (inference-time local price anchoring). '
        'The engineered-features hybrid sits in the worst position: '
        'it learns regime-specific mappings without the capacity to '
        'update them at inference.'
    ),
    'mechanism_diagnosed': [
        'days_since_start: gain 6%, permutation 0%. Training-period '
        'shortcut with zero out-of-sample signal',
        'Rolling means and momentum: high gain, near-zero permutation. '
        'Same shortcut pattern',
        'Coverage-stratified sharpening: V0 R²(log) = -0.286 on rows '
        'WITH rolling features vs +0.077 on cold-start rows (rolling '
        'features are net-harmful when available under drift)',
        'Section 9 ablation: no feature subset within the hybrid family '
        '(V0-V3, 17-31 features) recovers positive test R²',
        'Seed-robust: identical rankings across seeds [42, 123, 7]',
    ],
    'contribution_to_literature': (
        'The finding qualifies rather than rejects the engineered-features '
        'argument (Corsi 2009; Taylor & Letham 2018): engineered temporal '
        'features improve forecasting under approximate regime '
        'stationarity, but encode a frozen regime mapping that fails '
        'under large train-to-test drift. The condition for their success '
        'is regime continuity, not feature quality. This is consistent '
        'with the cautionary findings in Hewamalage et al. (2023) on '
        'forecasting evaluation.'
    ),
}

## O5: Limitations and downstream expectations
o5 = {
    'scope_limits': {
        'n_pokemon':       7,
        'n_transactions':  3812,
        'date_range':      '2021-01-01 to 2026-04-13',
        'grade_range':     'PSA 8-10',
        'grading_house':   'PSA only',
        'drift_magnitude': '4x price-level shift (train median $134 -> test median $520)',
    },
    'limitations': [
        'Single-market, single-grading-house, seven-Pokémon case study. '
        'Generalisation to other collectible domains or grading regimes '
        'is a conjecture, not a finding.',
        'Drift magnitude is unusually large. The finding that engineered '
        'features fail is conditional on this drift. A less drifted '
        'period may have produced a different outcome.',
        'Dataset size (n=3,812) is modest. Some of the negative findings '
        '(e.g., non-recovery of positive R² across ablation variants) '
        'could shift at larger scale. The evidence here is that they '
        'do not shift at this scale.',
        'The XGBoost market-state embedding used for fusion comes from '
        'the static+calendar model (test R² +0.144), not from V0 hybrid '
        '(test R² -0.117). This is a deliberate design choice documented '
        'in Section 11: carrying an out-of-sample-negative model into '
        'fusion would poison the extrinsic branch.',
        'Set metadata missing for 32.5% of rows. Not used as a feature.',
    ],
    'methodological_contribution': (
        'Permutation importance measured on val cannot reliably select '
        'features that generalise to test under large val-to-test drift. '
        'In this study, V3 (constructed using only permutation-positive '
        'features on val) performed worse on test than V0 (the full '
        'hybrid). This is reported as a methodological observation for '
        'the forecasting evaluation literature.'
    ),
    'downstream_expectations_for_fusion': [
        'The market module delivers two embeddings for the fusion '
        'notebook: LSTM (64-dim, test R² +0.273) and static+calendar '
        'XGBoost (64-dim via leaf-index + PCA, test R² +0.144). V0 '
        'hybrid is NOT extracted as an embedding.',
        'The LSTM embedding is price-saturated (all 64 dims correlate '
        '|ρ| > 0.1 with log-price, 59/64 at |ρ| > 0.3) but lives on an '
        'effective 4-dimensional manifold (90% variance in 4 PCA '
        'components). Fusion should expect this embedding to dominate '
        'the extrinsic branch.',
        'The XGBoost embedding is weak per-dim (mean |ρ| = 0.12) but '
        'carries geometric grade/identity structure visible in t-SNE. '
        'It is complementary, not a substitute.',
        'Fusion experiments in decomposition/fusion-vs-unimodal will test 13 variants, including '
        'dual-market-encoder configurations. Expect the LSTM to carry '
        'most fusion lift on the market side; the XGBoost embedding to '
        'contribute in interaction rather than marginal per-dim signal.',
    ],
}

## O6: Save all artefacts
print('market-encoder FINAL VERDICT')
print(f'\nO1 (vs static baseline):   {o1["answer"]}')
print(f'V0 hybrid: {o1["evidence"]["v0_test_r2_log"]:+.4f} '
       f'(±{o1["evidence"]["v0_test_r2_std"]:.4f})')
print(f'Static+calendar: {o1["evidence"]["static_cal_test_r2_log"]:+.4f} '
       f'(±{o1["evidence"]["static_cal_test_r2_std"]:.4f})')
print(f'Gap: {o1["evidence"]["gap"]:+.4f}')

print(f'\nO2 (vs sequence baseline): {o2["answer"]}')
print(f'V0 hybrid: {o2["evidence"]["v0_test_r2_log"]:+.4f}')
print(f'LSTM:      {o2["evidence"]["lstm_test_r2_log"]:+.4f}')
print(f'Gap:       {o2["evidence"]["gap"]:+.4f}')

print(f'\nO3 (temporal stability):   {o3["answer"]}')
print(f'V0 hybrid     segments (early/mid/late): '
       f'{o3["evidence"]["v0_hybrid"]["early_r2_log"]:+.3f} / '
       f'{o3["evidence"]["v0_hybrid"]["middle_r2_log"]:+.3f} / '
       f'{o3["evidence"]["v0_hybrid"]["late_r2_log"]:+.3f}')
print(f'    Static+calendar segments (early/mid/late): '
       f'{o3["evidence"]["static_plus_calendar"]["early_r2_log"]:+.3f} / '
       f'{o3["evidence"]["static_plus_calendar"]["middle_r2_log"]:+.3f} / '
       f'{o3["evidence"]["static_plus_calendar"]["late_r2_log"]:+.3f}')
print(f'LSTM          segments (early/mid/late): '
       f'{o3["evidence"]["lstm"]["early_r2_log"]:+.3f} / '
       f'{o3["evidence"]["lstm"]["middle_r2_log"]:+.3f} / '
       f'{o3["evidence"]["lstm"]["late_r2_log"]:+.3f}')

print(f'\nO4 (verdict):    {o4["verdict"]}')
print(f'Mechanism:')
for m in o4['mechanism_diagnosed']:
    print(f'- {m[:95]}{".." if len(m) > 95 else ""}')

print(f'\nO5 (scope and limits): see JSON artefact')
print(f'Scope: {o5["scope_limits"]["n_pokemon"]} Pokémon, '
       f'{o5["scope_limits"]["n_transactions"]} transactions, '
       f'{o5["scope_limits"]["drift_magnitude"]}')

print(f'\nO6 (artefacts): saving section15_rq3_verdict.json')

In [ ]:
## Save the consolidated market-encoder verdict for the writeup

section15 = {
    'section':       15,
    'title':         'Final market-encoder Assessment',
    'research_question': (
        'Does representing market state through engineered temporal '
        'features (rolling price averages, momentum, transaction volume, '
        'realised volatility) improve valuation stability compared to a '
        'static metadata baseline and a pure sequence model in thin, '
        'regime-dependent collectible markets?'
    ),
    'verdict':       'NOT SUPPORTED (qualified, regime-conditional)',
    'o1_vs_static_baseline':   o1,
    'o2_vs_sequence_baseline': o2,
    'o3_temporal_stability':   o3,
    'o4_final_verdict':        o4,
    'o5_limitations':          o5,
    'evidence_chain': [
        'Section 6:  V0 hybrid single-seed test R² = -0.134',
        'Section 7:  Comparison table: Static+calendar and LSTM both '
                    'beat V0 on test',
        'Section 8:  days_since_start (gain 6%, perm 0%) identifies '
                    'shortcut-learning mechanism. LSTM uses 54% of '
                    'attention on last 2 timesteps',
        'Section 9:  Ablation: No hybrid variant (V0-V3) recovers '
                    'positive test R². Feature subsetting does not fix '
                    'drift',
        'Section 10: Seed robustness (3 seeds): Rankings identical, '
                    'std < 0.05 on all three models',
        'Section 10: Temporal segment analysis: V0 collapses across '
                    'test. LSTM stable across all segments',
        'Section 14: Coverage-stratified: V0 worse on rows WITH '
                    'rolling features than on cold-start rows',
    ],
    'headline_summary': (
        'Engineered temporal features do not improve valuation stability '
        'under 4x train-to-test price-level drift in this dataset. The '
        'hybrid regressor is outperformed by a 6-feature static+calendar '
        'baseline (+0.144 vs -0.117 test R²(log), seed-averaged) and by '
        'an LSTM sequence model (+0.273). The mechanism is diagnosable: '
        'engineered features encode training-period regime information '
        'that does not transfer. A static baseline using regime-invariant '
        'identity features wins near training. An LSTM using '
        'inference-time local price conditioning wins under drift. The '
        'engineered hybrid wins neither.'
    ),
}

out_path = results_dir / 'section15_rq3_verdict.json'
with open(out_path, 'w') as f:
    json.dump(section15, f, indent=2, default=str)

print(f'Saved: {out_path}')
print(f'Size:  {out_path.stat().st_size / 1024:.1f} KB')

## market-encoder Final Verdict
**Research Question:** Does representing market state through engineered
temporal features (rolling price averages, momentum, transaction volume,
realised volatility) improve valuation stability compared to a static
metadata baseline and a pure sequence model in thin, regime-dependent
collectible markets?

**Verdict: NOT SUPPORTED**, with a qualified regime-conditional mechanism.

### Empirical result

Seed-averaged test R²(log) across 3 seeds (42, 123, 7):

| Model | Features | Test R²(log) | Test MAE (USD) |
|---|---|---|---|
| LSTM | price sequence + time-gap, K=10 | +0.273 ± 0.020 | 1,309 ± 12 |
| Static + calendar XGBoost | 6 features | +0.144 ± 0.025 | 1,343 ± 14 |
| V0 hybrid XGBoost | 31 engineered features | −0.117 ± 0.043 | 1,408 ± 7 |

The 31-feature hybrid regressor (the architecture that market-encoder defends)
underperforms both baselines. Rankings are identical across all three
seeds.

### Mechanism

Section 8 identified the mechanism through gain-vs-permutation
disagreement. The feature `days_since_start` carries 6% of gain importance
but 0% of permutation importance on validation, the canonical shortcut-
learning signature. Rolling means and momentum show the same pattern in
weaker form. Section 9 confirmed that no feature subset within the hybrid
family (V0 through V3, 17 to 31 features) recovers positive test R².
Section 14 documented the sharpest form of the finding: V0's test R²(log)
is −0.286 on rows with rolling features populated, but +0.077 on
cold-start rows where the rolling features are NaN. The engineered
temporal features are net-harmful on the rows where they exist.

### Temporal-segment dynamics

The three models degrade differently across the test period:

- Static+calendar: +0.361 (early) → +0.017 (middle) → −0.259 (late)
- V0 hybrid:       +0.282 (early) → −0.322 (middle) → −0.598 (late)
- LSTM:            +0.191 (early) → +0.279 (middle) → +0.223 (late)

The LSTM is the only model with positive R²(log) across all three test
segments. This is consistent with its architecture: conditioning on the
most recent transaction prices at inference lets it adapt to the drifted
regime, whereas XGBoost-based models are locked into the feature-price
mapping learned at training time.

### Contribution to the literature

The finding qualifies rather than rejects the engineered-features argument
in financial forecasting (Corsi 2009; Taylor & Letham 2018). Engineered
temporal features are useful under approximate regime stationarity but
encode a frozen regime mapping that fails under large train-to-test drift.
The condition for their success is regime continuity, not feature quality.
This is consistent with the cautionary findings in Hewamalage et al. (2023)
on forecasting evaluation practice.

A secondary methodological observation: permutation importance measured on
validation cannot reliably select features that generalise to test under
large val-to-test drift. In this study, a variant constructed using only
permutation-positive features on val performed worse on test than the full
hybrid.

### Scope

Findings are conditional on: 7 Pokémon, 3,812 transactions, PSA 8–10,
2021-01-01 to 2026-04-13, a 4× price-level shift from train median 134USD
to test median 520USD. Generalisation to other collectible markets, other
grading regimes, or less drifted periods is a conjecture, not a finding.

### Downstream expectations

For the fusion module (decomposition, fusion-vs-unimodal), the market-state branch supplies two
embeddings: LSTM (64-dim, learned) and static+calendar XGBoost (64-dim,
leaf-index + PCA). V0 hybrid is not extracted, because using a model with
negative test R² as a fusion input would poison the extrinsic branch.